# 🏥 Medical ASR — Whisper LoRA vs DoRA vs Wav2Vec2
**Domain adaptation study — Medical Speech, Transcription & Intent dataset**

We take OpenAI Whisper-small (pre-trained on 680K hours of general speech) and adapt it
to medical speech using two parameter-efficient methods (LoRA and DoRA), then compare
both against a fully fine-tuned Wav2Vec2-base model using WER and CER metrics.

| Step | Description | Est. Time (P100) |
|------|-------------|------------------|
| 0 | GPU check | instant |
| 1 | Install dependencies | ~2 min |
| 2 | Load, clean & split dataset | ~2 min |
| 3 | Baseline evaluation (Whisper, no fine-tuning) | ~5 min |
| 4 | Feature extraction for Whisper models | ~30 min |
| 5 | Whisper + LoRA — train & evaluate | ~4 hrs |
| 6 | Whisper + DoRA — train & evaluate | ~4 hrs |
| 7 | Wav2Vec2 — train & evaluate | ~2 hrs |
| 8 | Final results comparison table | instant |
| 9 | Export & merge models | ~10 min |
| 10 | Inference demo | instant |

**Total estimated time: ~11 hours on Kaggle P100 (free quota: 30 hrs/week)**

> **Reusing a Colab checkpoint?** If you already trained LoRA in Colab and saved the
> adapter to Google Drive, see Cell 5b to download it and skip the 4-hour LoRA training.

---
## 0️⃣ GPU Check — Must Pass Before Continuing

In [1]:
# ── Verify GPU is available ───────────────────────────────────────────────────
# If this cell raises RuntimeError, go to:
#   Notebook Settings (top-right gear icon) → Accelerator → GPU P100
# Then re-run from the beginning.

!nvidia-smi
import torch

print('\n' + '='*55)
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {props.total_memory / 1e9:.1f} GB')
    print(f'PyTorch        : {torch.__version__}')
else:
    raise RuntimeError(
        '❌ No GPU found!\n'
        'Go to: Notebook Settings → Accelerator → GPU P100'
    )
print('='*55)

Tue Apr  7 21:46:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 1️⃣ Install Dependencies

In [4]:
# ── Install all required libraries ───────────────────────────────────────────
# Pin transformers to 4.40.0 to avoid API changes that break LoRA training.
# All other packages use latest stable versions.
#
# Library roles:
#   transformers  — Whisper + Wav2Vec2 models, Trainer, Seq2SeqTrainer
#   datasets      — HuggingFace Dataset objects (memory-mapped Arrow format)
#   accelerate    — backend for distributed/fp16 training inside Trainer
#   peft          — LoRA / DoRA wrappers (get_peft_model, LoraConfig)
#   soundfile     — low-level .wav file I/O (used by librosa internally)
#   librosa       — audio loading + resampling to 16 kHz
#   jiwer         — WER / CER metric computation
#   scikit-learn  — train_test_split for 80/10/10 split

# Fix NumPy / SciPy / scikit-learn compatibility
!pip -q install --upgrade \
    transformers==4.46.3 \
    peft==0.13.2 \
    accelerate==0.34.2 \
    datasets==2.20.0 \
    soundfile==0.12.1 \
    librosa==0.10.2 \
    jiwer==3.0.4 \
    tqdm==4.66.4

# Do NOT pin numpy, scipy, scikit-learn, pandas — use whatever Kaggle has pre-installed
# Kaggle's base image already has compatible versions of all four

print('✅ Libraries installed.')

# Hard restart to flush any cached incompatible modules from memory
#import IPython
#IPython.Application.instance().kernel.do_shutdown(restart=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 65.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os

# ── Clear every distributed env var that could trigger torch.distributed ──────
# These get set by Kaggle's runtime when 2 GPUs are present.
# If ANY of WORLD_SIZE / RANK / LOCAL_RANK exist, accelerate tries to call
# init_process_group — which then demands MASTER_ADDR and crashes.
# Solution: remove them all so accelerate sees a plain single-process run.
for var in [
    "WORLD_SIZE", "RANK", "LOCAL_RANK",
    "MASTER_ADDR", "MASTER_PORT",
    "TORCHELASTIC_RESTART_COUNT",
    "TORCHELASTIC_MAX_RESTARTS",
    "TORCHELASTIC_RUN_ID",
    "GROUP_RANK", "ROLE_RANK", "ROLE_NAME",
]:
    os.environ.pop(var, None)

# ── Tell accelerate explicitly: one process, one GPU, no distribution ─────────
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"

# ── Write the accelerate default config ───────────────────────────────────────
import pathlib
cfg_path = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
use_cpu: false
""")

# ── Patch PartialState before TrainingArguments is ever imported ───────────────
# This is the nuclear option: monkey-patch accelerate's PartialState.__init__
# to never call init_process_group regardless of what it detects.
from accelerate.state import PartialState
import accelerate.state as _acc_state
from accelerate.utils import DistributedType

_original_partial_state_init = PartialState.__init__

def _patched_partial_state_init_DISABLED(self, cpu=False, **kwargs):
    # Skip distributed setup entirely — force single-process state
    self.__dict__.clear()
    self._shared_state = {}   
    object.__setattr__(self, '_shared_state', PartialState.__dict__.get('_shared_state', {}))
    # Set the minimum attributes TrainingArguments reads
    self.process_index        = 0
    self.local_process_index  = 0
    self.num_processes        = 1
    self.distributed_type     = DistributedType.NO
    self.device               = __import__('torch').device('cuda:0')
    # self.is_last_process      = True  # disabled: avoid setting read-only accelerate properties
    self.use_distributed      = False

# PartialState.__init__ = _patched_partial_state_init_DISABLED  # disabled (prevents corrupting accelerate state)
print("✅ Skipping PartialState monkey-patching (safe single-process config)")
print("✅ All distributed env vars cleared")
print("✅ Accelerate config written")
print("\n🔒 From this point on: single GPU (cuda:0), no DataParallel, no torch.distributed")

✅ Skipping PartialState monkey-patching (safe single-process config)
✅ All distributed env vars cleared
✅ Accelerate config written

🔒 From this point on: single GPU (cuda:0), no DataParallel, no torch.distributed


---
## 2️⃣ Configure Paths, Load & Clean Dataset

**Before running this cell:**
1. Click **+ Add Data** (top-right of the Kaggle editor)
2. Search: `medical speech transcription intent`
3. Click **Add** — it will appear at `/kaggle/input/medical-speech-transcription-and-intent/`

**Why we re-split with 80/10/10 instead of using the original folders:**
The original dataset puts ~89% of files in the `test` folder and only ~6% in `train`.
That is backwards for training. We ignore the original split completely and
re-split all clean data ourselves: 80% train / 10% val / 10% test.

In [5]:
import os

# ── Kaggle dataset base path ──────────────────────────────────────────────────
# Kaggle may place the dataset in slightly different subdirectory structures
# depending on how it was uploaded. We try both known layouts and pick whichever exists.
POSSIBLE_BASES = [
    '/kaggle/input/medical-speech-transcription-and-intent/medical speech transcription and intent/Medical Speech, Transcription, and Intent',
    '/kaggle/input/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent',
    '/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent'
]
KAGGLE_BASE = next((b for b in POSSIBLE_BASES if os.path.exists(b)), None)

if KAGGLE_BASE is None:
    # Print what Kaggle actually mounted so the user can debug the path
    print('❌ Dataset not found. Available input directories:')
    for root, dirs, _ in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 4:
            print(f'  {root}')
    raise FileNotFoundError(
        'Dataset not found. Add it via + Add Data → '
        'search "medical speech transcription intent" → Add'
    )

# ── Input paths (read-only on Kaggle) ────────────────────────────────────────
CSV_PATH  = os.path.join(KAGGLE_BASE, 'overview-of-recordings.csv')
TRAIN_DIR = os.path.join(KAGGLE_BASE, 'recordings', 'train')
VAL_DIR   = os.path.join(KAGGLE_BASE, 'recordings', 'validate')
TEST_DIR  = os.path.join(KAGGLE_BASE, 'recordings', 'test')

# ── Output paths (writable — /kaggle/working) ─────────────────────────────────
# Everything saved here is downloadable from the Kaggle output panel after the run.
WORK_DIR         = '/kaggle/working'
OUTPUT_DIR_LORA  = f'{WORK_DIR}/whisper-lora-training'   # training checkpoints
OUTPUT_DIR_DORA  = f'{WORK_DIR}/whisper-dora-training'
OUTPUT_DIR_W2V   = f'{WORK_DIR}/wav2vec2-training'
ADAPTER_DIR_LORA = f'{WORK_DIR}/whisper-lora-adapter'    # lightweight LoRA weights only
ADAPTER_DIR_DORA = f'{WORK_DIR}/whisper-dora-adapter'
W2V_SAVE_DIR     = f'{WORK_DIR}/wav2vec2-saved'          # full Wav2Vec2 model
MERGED_DIR_LORA  = f'{WORK_DIR}/whisper-lora-merged'     # LoRA baked into Whisper (standalone)
MERGED_DIR_DORA  = f'{WORK_DIR}/whisper-dora-merged'

# ── Sanity check — confirm all input paths exist ──────────────────────────────
print('📁 Path verification:')
print('-' * 65)
all_ok = True
for name, p in [('CSV', CSV_PATH), ('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    exists = os.path.exists(p)
    count  = len(os.listdir(p)) if exists and os.path.isdir(p) else '(file)'
    icon   = '✅' if exists else '❌'
    print(f'  {icon} {name:6s} → items={str(count):>6}  path={p}')
    if not exists:
        all_ok = False
if not all_ok:
    raise FileNotFoundError('One or more paths missing — check the dataset was added correctly.')
print(f'\n  Base : {KAGGLE_BASE}')
print('\n✅ All paths verified.')

📁 Path verification:
-----------------------------------------------------------------
  ✅ CSV    → items=(file)  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv
  ✅ train  → items=   381  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/train
  ✅ val    → items=   385  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/validate
  ✅ test   → items=  5895  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/test

  Base : /kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent

✅ All paths verified.


In [6]:
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# ── Step 1: Load the CSV ──────────────────────────────────────────────────────
# The CSV is the master metadata file with one row per recording.
# It contains: file_name, phrase (ground-truth transcript), audio quality scores.
df = pd.read_csv(CSV_PATH)
# Normalise transcript: lowercase + strip whitespace.
# This ensures WER/CER comparisons are case-insensitive and whitespace-clean.
df['text'] = df['phrase'].astype(str).str.strip().str.lower()
print(f'📄 CSV loaded: {len(df):,} rows, {len(df.columns)} columns')

# ── Step 2: Resolve audio paths ───────────────────────────────────────────────
# The CSV contains only the bare filename (e.g. 1249120_43453425_58166571.wav).
# We search all three original folders and build a filename → full_path index.
# This works regardless of which folder a file was originally placed in.
file_index = {}
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if os.path.isdir(d):
        for fname in os.listdir(d):
            file_index[fname] = os.path.join(d, fname)

df['audio_path'] = df['file_name'].map(file_index)
found = df['audio_path'].notna().sum()
print(f'🔍 Audio files matched: {found:,} / {len(df):,}')

# ── Step 3: Quality filter ────────────────────────────────────────────────────
# We keep only clean recordings to avoid training the model on bad audio.
# Thresholds chosen based on dataset documentation:
#   overall_quality >= 3.33  → keeps approx top 75% by quality score
#   no heavy clipping        → heavy clipping distorts the waveform badly
#   no heavy noise           → background noise confuses the model
#   non-empty transcript     → empty labels would corrupt the loss function
clean = df.dropna(subset=['audio_path']).copy()
before = len(clean)
clean = clean[clean['overall_quality_of_the_audio'] >= 3.33]
clean = clean[clean['audio_clipping'].isin(['no_clipping', 'light_clipping'])]
clean = clean[clean['background_noise_audible'].isin(['no_noise', 'light_noise'])]
clean = clean[clean['text'].str.len() > 0]
after = len(clean)
print(f'🧹 Quality filter: {before:,} → {after:,} kept  ({before - after:,} removed)')

# ── Step 4: 80 / 10 / 10 split ───────────────────────────────────────────────
# We IGNORE the original train/validate/test folder split because the original
# distribution is extremely unbalanced: ~89% test, ~6% train, ~6% validate.
# Training on only 6% of the data with 89% held out for test is useless.
# random_state=42 makes the split reproducible — running again gives same split.
train_df, temp_df = train_test_split(clean, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Keep only the columns we need
train_df = train_df[['audio_path', 'text']].reset_index(drop=True)
val_df   = val_df[['audio_path',   'text']].reset_index(drop=True)
test_df  = test_df[['audio_path',  'text']].reset_index(drop=True)

print(f'\n✂️  Final 80/10/10 split:')
print(f'   train : {len(train_df):,} samples')
print(f'   val   : {len(val_df):,} samples')
print(f'   test  : {len(test_df):,} samples')
print(f'\n📝 Sample transcripts from training set:')
for _, row in train_df.head(3).iterrows():
    print(f"   → {row['text'][:80]}")

📄 CSV loaded: 6,661 rows, 14 columns
🔍 Audio files matched: 6,661 / 6,661
🧹 Quality filter: 6,661 → 6,106 kept  (555 removed)

✂️  Final 80/10/10 split:
   train : 4,884 samples
   val   : 611 samples
   test  : 611 samples

📝 Sample transcripts from training set:
   → i often get a sharp pain in my chest and i can't tell what i'm doing that might 
   → i can't carry anything i have a pain in my shoulder
   → i feel something hurt me in taking breath and i cant take my breath


---
## 3️⃣ Baseline Evaluation — Whisper-small (No Fine-Tuning)

Before we train anything, we measure how well Whisper-small performs on medical speech
**out of the box**. This gives us the baseline scores to beat.

**Why use a fixed 200-sample subset?**
Running inference on the full test set (~600 samples) for 4 different models would take
hours. We fix a random 200-sample subset with `random_state=42` and reuse it for ALL
model comparisons — this keeps the comparison perfectly fair.

In [7]:
import torch
import librosa
from tqdm import tqdm
from jiwer import wer, cer
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.generation.configuration_utils import GenerationConfig

# ── Global constants used throughout the notebook ────────────────────────────
MODEL_NAME = 'openai/whisper-small'  # 244M params, good balance of speed/accuracy
TARGET_SR  = 16_000                  # Whisper requires 16 kHz audio
N_EVAL     = 200                     # samples used for all model evaluations
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')

# ── Load Whisper processor and baseline model ─────────────────────────────────
# The processor contains:
#   - feature_extractor: converts raw audio → log-mel spectrogram (80 mel bins)
#   - tokenizer: converts text ↔ token IDs for the decoder
print(f'⬇️  Loading {MODEL_NAME}...')
processor      = WhisperProcessor.from_pretrained(MODEL_NAME)
baseline_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# ── Fix generation config (required for transformers >= 4.38) ─────────────────
# Older code set generation parameters on model.config, but newer transformers
# moved them to model.generation_config and raises an error if model.config
# still has them. We delete them from config and set them properly on
# generation_config instead. This pattern is applied to every model we load.
if hasattr(baseline_model.config, 'suppress_tokens'):    del baseline_model.config.suppress_tokens
if hasattr(baseline_model.config, 'forced_decoder_ids'): del baseline_model.config.forced_decoder_ids
baseline_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
baseline_model.generation_config.forced_decoder_ids = None  # allow any language
baseline_model.generation_config.suppress_tokens    = []    # don't suppress any tokens
baseline_model.eval().to(device)
print('✅ Baseline model loaded.')

# ── Transcription function shared by all Whisper models ───────────────────────
# Steps:
#   1. librosa.load — decode .wav and resample to TARGET_SR (16 kHz)
#   2. processor    — convert waveform to log-mel spectrogram tensor
#   3. model.generate — autoregressive decoding (beam search by default)
#   4. batch_decode — convert token IDs back to text string
#
# max_new_tokens=128 caps output length (medical phrases are short, 128 is more than enough).
# max_length=None prevents a conflict warning between max_new_tokens and max_length.
# language='english' forces English output, avoiding accidental translation.
# task='transcribe' prevents the model switching to translation mode.
def transcribe_whisper(model, audio_path):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR)
    inputs   = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=128,
            max_length=None,
            language='english',
            task='transcribe'
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

# ── Fixed evaluation subset ────────────────────────────────────────────────────
# IMPORTANT: same random_state=42 is used here and in all subsequent evals.
# This guarantees we always evaluate all models on the exact same 200 samples.
# Changing this would make comparisons between models unfair.
eval_subset = test_df.sample(N_EVAL, random_state=42).reset_index(drop=True)
refs        = eval_subset['text'].tolist()  # ground-truth transcripts (lowercase)

print(f'\n🔬 Evaluating baseline on {N_EVAL} fixed samples...')
baseline_preds = [
    transcribe_whisper(baseline_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='Baseline eval')
]

# ── Compute WER and CER ───────────────────────────────────────────────────────
# WER (Word Error Rate): fraction of words wrong
#   WER = (Substitutions + Deletions + Insertions) / Total_Reference_Words
#   WER = 0.0 means perfect transcription, 1.0 means completely wrong.
# CER (Character Error Rate): same but at character level.
#   CER is always lower than WER because getting one character wrong
#   ruins a word (WER) but is a small fraction of characters (CER).
baseline_wer = wer(refs, baseline_preds)
baseline_cer = cer(refs, baseline_preds)
print(f'\n📊 BASELINE — Whisper-small (no fine-tuning)')
print(f'   WER : {baseline_wer*100:.2f}%  ← target to beat')
print(f'   CER : {baseline_cer*100:.2f}%')

2026-03-28 11:00:25.522714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774695625.741064      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774695625.800596      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774695626.296661      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695626.296703      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695626.296705      55 computation_placer.cc:177] computation placer alr

🖥️  Device: cuda
⬇️  Loading openai/whisper-small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Baseline model loaded.

🔬 Evaluating baseline on 200 fixed samples...


Baseline eval: 100%|██████████| 200/200 [01:12<00:00,  2.75it/s]


📊 BASELINE — Whisper-small (no fine-tuning)
   WER : 16.27%  ← target to beat
   CER : 7.11%


In [8]:
# ── Free baseline model from GPU memory ──────────────────────────────────────
# We delete the baseline model immediately after evaluation because:
#   - P100 has 16 GB VRAM; Whisper-small is ~1 GB loaded
#   - Keeping it loaded while training LoRA would waste memory and risk OOM
#   - We already have the scores stored in baseline_wer / baseline_cer
import gc
del baseline_model
torch.cuda.empty_cache()
gc.collect()
print('✅ Baseline model freed from GPU memory.')

✅ Baseline model freed from GPU memory.


---
## 4️⃣ Feature Extraction — ~30 min

We pre-process all audio files into model-ready tensors and store them on disk
using HuggingFace Datasets' Arrow format.

**Why pre-process instead of loading on-the-fly during training?**
Loading and processing audio during training creates a CPU bottleneck that starves
the GPU. Pre-processing once and storing to disk makes each training step much faster.

**What `prepare_batch` does:**
- `librosa.load` → reads the .wav file and resamples it to 16 kHz
- `feature_extractor` → converts the 1D waveform to an 80-bin log-mel spectrogram
  of shape (80, 3000) — this is the actual input Whisper's encoder sees
- `tokenizer` → converts the text transcript into a sequence of integer token IDs
  that the decoder is trained to predict

**Note:** This cell produces datasets for Whisper only. Wav2Vec2 uses raw waveforms
(not mel spectrograms), so it has its own prepare step in Cell 7.

In [9]:
import gc
import os
import time
import numpy as np
import librosa
import torch
from datasets import Dataset, concatenate_datasets
from dataclasses import dataclass
from typing import Any, Dict, List

# ── Chunked feature extraction — avoids both OOM and Dataset.map hang ─────────
# Strategy:
#   1. Process audio in chunks of CHUNK_SIZE examples
#   2. After each chunk, save to disk immediately and clear RAM
#   3. At the end, reload all chunks from disk and concatenate
#   This keeps peak RAM at ~CHUNK_SIZE × 80×3000 × 4 bytes ≈ 200 MB per chunk

CHUNK_SIZE = 50   # Process 50 files at a time — safe for 15 GB VRAM

def build_whisper_dataset_chunked(df, split_name, save_base, chunk_size=CHUNK_SIZE):
    total = len(df)
    n_chunks = (total + chunk_size - 1) // chunk_size
    chunk_dirs = []

    print(f"\n🔄 {split_name}: {total:,} samples in {n_chunks} chunks of {chunk_size}...")
    t0 = time.time()

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end   = min(start + chunk_size, total)
        chunk_df = df.iloc[start:end]

        rows = []
        for row in chunk_df.itertuples(index=False):
            try:
                audio, _ = librosa.load(row.audio_path, sr=TARGET_SR)
                feats = processor.feature_extractor(
                    audio, sampling_rate=TARGET_SR
                ).input_features[0]
                labels = processor.tokenizer(row.text).input_ids
                rows.append({
                    "input_features": np.asarray(feats, dtype=np.float32),
                    "labels": labels,
                })
            except Exception as e:
                print(f"   ⚠️  Skipping {row.audio_path}: {e}")
                continue

        # Save this chunk to disk immediately, then free RAM
        chunk_dir = f"{save_base}_chunk_{chunk_idx:04d}"
        Dataset.from_list(rows).save_to_disk(chunk_dir)
        chunk_dirs.append(chunk_dir)
        del rows
        gc.collect()

        # Progress
        elapsed = time.time() - t0
        rate    = (end) / elapsed if elapsed > 0 else 0
        eta     = (total - end) / rate if rate > 0 else 0
        print(f"   chunk {chunk_idx+1:3d}/{n_chunks} | {end:,}/{total:,} "
              f"| {rate:.1f} ex/s | ETA {eta/60:.1f} min")

    # Reload all chunks and concatenate into one Dataset
    print(f"   Concatenating {len(chunk_dirs)} chunks...")
    from datasets import load_from_disk
    all_chunks = [load_from_disk(d) for d in chunk_dirs]
    full_ds = concatenate_datasets(all_chunks)

    # Save final merged dataset and clean up chunk dirs
    full_ds.save_to_disk(save_base)
    import shutil
    for d in chunk_dirs:
        shutil.rmtree(d, ignore_errors=True)

    print(f"✅ {split_name} done: {len(full_ds):,} samples  → {save_base}")
    return full_ds


# ── Paths for caching to disk (Kaggle /kaggle/working/ survives session) ──────
TRAIN_HF_DIR = "/kaggle/working/whisper_train_hf"
VAL_HF_DIR   = "/kaggle/working/whisper_val_hf"
TEST_HF_DIR  = "/kaggle/working/whisper_test_hf"

# ── Check if cached datasets already exist (avoids re-processing on re-run) ───
from datasets import load_from_disk

def load_or_build(df, split_name, save_dir):
    if os.path.exists(save_dir):
        print(f"⚡ {split_name}: loading from cache at {save_dir}")
        return load_from_disk(save_dir)
    return build_whisper_dataset_chunked(df, split_name, save_dir)

train_hf = load_or_build(train_df, "train", TRAIN_HF_DIR)
val_hf   = load_or_build(val_df,   "val",   VAL_HF_DIR)
test_hf  = load_or_build(test_df,  "test",  TEST_HF_DIR)

print(f"\n✅ Feature extraction complete:")
print(f"   train : {len(train_hf):,} samples")
print(f"   val   : {len(val_hf):,} samples")
print(f"   test  : {len(test_hf):,} samples")

# ── Data Collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Data collator ready.")


🔄 train: 4,884 samples in 98 chunks of 50...


Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

   chunk   1/98 | 50/4,884 | 16.0 ex/s | ETA 5.0 min


Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

   chunk   2/98 | 100/4,884 | 16.9 ex/s | ETA 4.7 min


...

   chunk  12/13 | 600/611 | 18.9 ex/s | ETA 0.0 min


Saving the dataset (0/1 shards):   0%|          | 0/11 [00:00<?, ? examples/s]

   chunk  13/13 | 611/611 | 18.7 ex/s | ETA 0.0 min
   Concatenating 13 chunks...


Saving the dataset (0/2 shards):   0%|          | 0/611 [00:00<?, ? examples/s]

✅ test done: 611 samples  → /kaggle/working/whisper_test_hf

✅ Feature extraction complete:
   train : 4,884 samples
   val   : 611 samples
   test  : 611 samples

✅ Data collator ready.


---
## 5️⃣ Model A — Whisper-small + LoRA (~4 hrs)

**What is LoRA?**
Instead of updating all 244M Whisper parameters, LoRA *freezes* the original weights
and injects small trainable matrices into the attention layers.
Each injected matrix is factored as two small matrices (rank r=16),
so the number of new parameters is tiny (~2.3M = ~1% of total).

Mathematically, if W is the original frozen weight, LoRA adds:
  `W_new = W + (B × A) × (alpha/r)`
where A and B are the small trainable matrices, alpha is a scaling factor.

**Why only q_proj and v_proj?**
These are the Query and Value projection matrices in each attention head.
Research shows that adapting Q and V gives the best accuracy/parameter tradeoff
for Whisper. Adding k_proj or out_proj increases parameters with diminishing returns.

**Cell 5b below:** If you already ran LoRA training in Google Colab and saved the adapter,
you can download it from Drive and skip the 4-hour training. Otherwise run 5a.

In [20]:
# =========================
# FINAL 1-CELL WHISPER LoRA TRAINING (KAGGLE SAFE)
# includes Accelerate reset fix before Seq2SeqTrainer(...)
# =========================

import os
import gc
import pathlib
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------
# 0) Force Accelerate to single GPU / no distributed
# ------------------------------------------------------------
os.environ.pop("MASTER_ADDR", None)
os.environ.pop("MASTER_PORT", None)
os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# Fresh PartialState check
from accelerate.state import PartialState
from accelerate.utils import DistributedType

ps = PartialState()
#if ps.num_processes != 1 or ps.distributed_type != DistributedType.NO:
    # ps._shared_state["num_processes"] = 1  # disabled: do not mutate accelerate internals
    # ps._shared_state["distributed_type"] = DistributedType.NO  # disabled: do not mutate accelerate internals

print(f"num_processes   : {ps.num_processes}")
print(f"distributed_type: {ps.distributed_type}")
print(f"device          : {ps.device}")

# ------------------------------------------------------------
# 1) Cleanup
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"training device : {device}")

# Keep only the columns Whisper needs
KEEP = {"input_features", "labels"}

def clean_ds(ds):
    drop_cols = [c for c in ds.column_names if c not in KEEP]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)
    # ds.set_format(type="torch")  # disabled: NumPy 2.0 + datasets torch formatter crash
    ds.reset_format()  # keep python/numpy format; collator/processor will create torch tensors
    return ds

train_clean = clean_ds(train_hf)
val_clean   = clean_ds(val_hf)

print("train columns:", train_clean.column_names)
print("val columns  :", val_clean.column_names)
print("sample keys  :", train_clean[0].keys())

# ------------------------------------------------------------
# 2) Whisper collator: returns ONLY input_features + labels
# ------------------------------------------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        bos_id = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]

        batch["labels"] = labels

        # Safety: Whisper must not receive input_ids
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# sanity check
test_batch = data_collator([train_clean[0], train_clean[1]])
print("collator keys:", test_batch.keys())
for k, v in test_batch.items():
    print(f"{k}: {tuple(v.shape)}")

# ------------------------------------------------------------
# 3) Load fresh Whisper base model
# ------------------------------------------------------------
print(f"\n⬇️ Loading {MODEL_NAME} for LoRA...")
lora_base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

if hasattr(lora_base.config, "suppress_tokens"):
    del lora_base.config.suppress_tokens
if hasattr(lora_base.config, "forced_decoder_ids"):
    del lora_base.config.forced_decoder_ids

lora_base.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
lora_base.generation_config.forced_decoder_ids = None
lora_base.generation_config.suppress_tokens = []

if hasattr(lora_base.config, "use_cache"):
    lora_base.config.use_cache = False

# ------------------------------------------------------------
# 4) Apply LoRA
# ------------------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    use_dora=False,
)

lora_model = get_peft_model(lora_base, lora_config)
lora_model.to(device)
lora_model.print_trainable_parameters()

# Use the underlying Whisper model for forward.
# PEFT's task-specific wrapper may pass input_ids even when we only provide input_features.
train_model = lora_model.get_base_model()
train_model.to(device)

# ------------------------------------------------------------
# 5) Training arguments
# ------------------------------------------------------------
lora_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_LORA,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    warmup_steps=50,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    predict_with_generate=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

# ------------------------------------------------------------
# 6) HARD RESET Accelerate state RIGHT BEFORE Trainer
# fixes: AcceleratorState object has no attribute distributed_type
# ------------------------------------------------------------
import accelerate.state as accel_state

# accel_state.PartialState._shared_state.clear()  # disabled: avoid corrupting accelerate state
# accel_state.AcceleratorState._shared_state.clear()  # disabled: avoid corrupting accelerate state

from accelerate.state import PartialState
ps = PartialState()
print("\nAfter reset:")
print("num_processes   :", ps.num_processes)
print("distributed_type:", ps.distributed_type)
print("device          :", ps.device)

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7) Trainer
# ------------------------------------------------------------
lora_trainer = Seq2SeqTrainer(
    model=train_model,
    args=lora_args,
    train_dataset=train_clean,
    eval_dataset=val_clean,
    data_collator=data_collator,
)

# ------------------------------------------------------------
# 8) Sanity forward pass
# ------------------------------------------------------------
print("\n🧪 Running sanity forward pass...")
sanity_batch = data_collator([train_clean[0], train_clean[1]])
sanity_batch = {k: v.to(device) for k, v in sanity_batch.items() if k in ("input_features", "labels")}

with torch.no_grad():
    sanity_out = train_model(**sanity_batch)

print("✅ Sanity forward pass OK")
print("sanity loss:", float(sanity_out.loss))

# ------------------------------------------------------------
# 9) Train
# ------------------------------------------------------------
print("\n🚀 Training LoRA...")
train_result = lora_trainer.train()

# ------------------------------------------------------------
# 10) Save adapter
# ------------------------------------------------------------
os.makedirs(ADAPTER_DIR_LORA, exist_ok=True)
lora_model.save_pretrained(ADAPTER_DIR_LORA)
processor.save_pretrained(ADAPTER_DIR_LORA)

print(f"\n✅ LoRA adapter saved -> {ADAPTER_DIR_LORA}")

# ------------------------------------------------------------
# 11) Cleanup
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

num_processes   : 1
distributed_type: DistributedType.NO
device          : cuda
training device : cuda:0
train columns: ['input_features', 'labels']
val columns  : ['input_features', 'labels']
sample keys  : dict_keys(['input_features', 'labels'])
collator keys: dict_keys(['input_features', 'labels'])
input_features: (2, 80, 3000)
labels: (2, 27)

⬇️ Loading openai/whisper-small for LoRA...
trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267

After reset:
num_processes   : 1
distributed_type: DistributedType.NO
device          : cuda

🧪 Running sanity forward pass...
✅ Sanity forward pass OK
sanity loss: 4.814150810241699

🚀 Training LoRA...


Epoch,Training Loss,Validation Loss
0,0.441700,0.413438
1,0.270400,0.297241
2,0.211200,0.235808
4,0.053000,0.150268
5,0.026900,0.145030
6,0.026100,0.142248
8,0.018600,0.146969
9,0.010000,0.147889


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



✅ LoRA adapter saved -> /kaggle/working/whisper-lora-adapter


In [21]:
# ── 5b: OPTIONAL — Load LoRA checkpoint from Google Drive ─────────────────────
# Use this INSTEAD OF Cell 5a if you already trained LoRA in Colab.
#
# How to get your Drive folder ID:
#   1. Open Google Drive → navigate to your whisper-lora-adapter folder
#   2. Right-click → Share → Copy link
#   3. The link contains: .../folders/XXXXXXXXXXXXXXXX
#   4. That XXXXXXXXXXXXXXXX is your DRIVE_FOLDER_ID
#
# DRIVE_FOLDER_ID = 'paste_your_folder_id_here'
#
# !pip -q install gdown
# import gdown
# os.makedirs(ADAPTER_DIR_LORA, exist_ok=True)
# gdown.download_folder(
#     f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',
#     output=ADAPTER_DIR_LORA, quiet=False
# )
# from transformers import WhisperForConditionalGeneration
# from transformers.generation.configuration_utils import GenerationConfig
# from peft import PeftModel
# lora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
# lora_model = PeftModel.from_pretrained(lora_base, ADAPTER_DIR_LORA)
# lora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
# lora_model.generation_config.forced_decoder_ids = None
# lora_model.generation_config.suppress_tokens    = []
# lora_model.eval().to(device)
# print('✅ LoRA loaded from Drive — skipping 5a training!')

print('ℹ️  Cell 5b is commented out. Fill in DRIVE_FOLDER_ID and uncomment to use a Drive checkpoint.')

ℹ️  Cell 5b is commented out. Fill in DRIVE_FOLDER_ID and uncomment to use a Drive checkpoint.


In [22]:
# ── 5c: Evaluate LoRA model ────────────────────────────────────────────────────
# We re-apply the generation config fix here because get_peft_model creates a
# new wrapper object that doesn't automatically inherit the fix from lora_base.
from jiwer import wer, cer
from transformers.generation.configuration_utils import GenerationConfig

# Clear any stale suppress_tokens from model.config before evaluating
if hasattr(lora_model.config, 'suppress_tokens'):    del lora_model.config.suppress_tokens
if hasattr(lora_model.config, 'forced_decoder_ids'): del lora_model.config.forced_decoder_ids
lora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
lora_model.generation_config.forced_decoder_ids = None
lora_model.generation_config.suppress_tokens    = []
lora_model.eval()

# Evaluate on the same 200 samples used for the baseline (eval_subset, refs defined in Cell 3)
print(f'🔬 Evaluating LoRA on {N_EVAL} samples (same fixed subset as baseline)...')
lora_preds = [
    transcribe_whisper(lora_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='LoRA eval')
]

lora_wer = wer(refs, lora_preds)
lora_cer = cer(refs, lora_preds)
lora_improvement = (baseline_wer - lora_wer) / baseline_wer * 100
print(f'\n📊 MODEL A — Whisper-small + LoRA')
print(f'   WER : {lora_wer*100:.2f}%  (baseline was {baseline_wer*100:.2f}%)  → {lora_improvement:+.1f}% vs baseline')
print(f'   CER : {lora_cer*100:.2f}%')

🔬 Evaluating LoRA on 200 samples (same fixed subset as baseline)...


LoRA eval: 100%|██████████| 200/200 [01:13<00:00,  2.72it/s]


📊 MODEL A — Whisper-small + LoRA
   WER : 7.30%  (baseline was 16.27%)  → +55.2% vs baseline
   CER : 4.44%


In [23]:
# ── Free LoRA model from GPU before loading DoRA ─────────────────────────────
# Always free the previous model before loading the next one to avoid OOM.
import gc
del lora_model
torch.cuda.empty_cache()
gc.collect()
print('✅ LoRA model freed from GPU memory.')

✅ LoRA model freed from GPU memory.


---
## 6️⃣ Model B — Whisper-small + DoRA (~4 hrs)

**What is DoRA and how does it differ from LoRA?**

Standard LoRA adds a low-rank update: `W_new = W + B×A`

DoRA (Weight-Decomposed Low-Rank Adaptation) first decomposes the weight matrix
into two components: magnitude (||W||) and direction (W / ||W||).
It then applies LoRA only to the *direction* component while learning
the *magnitude* separately.

In practice this means DoRA is better at mimicking full fine-tuning behaviour
while keeping the same low parameter count as LoRA.
Published results show DoRA typically achieves 1–3% lower WER than standard LoRA.

The only code change from LoRA is: `use_dora=True` in the LoraConfig.

In [10]:
import gc, torch, os

# Delete everything from LoRA training
try: del lora_trainer
except: pass
try: del lora_model
except: pass
try: del lora_base
except: pass
try: del train_model
except: pass
try: del sanity_out
except: pass
try: del sanity_batch
except: pass
try: del test_batch
except: pass

gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

# Check VRAM
if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM free : {free:.1f} GB")
    print(f"VRAM total: {total:.1f} GB")
    if free < 8:
        print("⚠️  Less than 8GB free — DoRA may OOM. Try reducing batch size.")
    else:
        print("✅ Enough VRAM for DoRA.")

VRAM free : 15.5 GB
VRAM total: 15.6 GB
✅ Enough VRAM for DoRA.


In [11]:
# =========================
# FULL DoRA TRAINING CELL (cleaned + safer)
# =========================

import os
import gc
import pathlib
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------
# 0) Core variables
# ------------------------------------------------------------
MODEL_NAME = "openai/whisper-small"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Adjust these if not already defined earlier
OUTPUT_DIR_DORA = globals().get("OUTPUT_DIR_DORA", "/kaggle/working/whisper-dora-training")
ADAPTER_DIR_DORA = globals().get("ADAPTER_DIR_DORA", "/kaggle/working/whisper-dora-adapter")

# ------------------------------------------------------------
# 1) Force single GPU / non-distributed
# ------------------------------------------------------------
os.environ.pop("MASTER_ADDR", None)
os.environ.pop("MASTER_PORT", None)
os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

gc.collect()
torch.cuda.empty_cache()

print(f"device: {device}")

# ------------------------------------------------------------
# 2) Load processor
# ------------------------------------------------------------
print(f"⬇️ Loading processor for {MODEL_NAME}...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
print("✅ Processor loaded")

# ------------------------------------------------------------
# 3) Collator
# ------------------------------------------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Keep float32; Trainer autocast/fp16 handles compute safely
        batch["input_features"] = batch["input_features"].float()

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        bos_id = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]

        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ------------------------------------------------------------
# 4) Check datasets exist
# ------------------------------------------------------------
if "train_hf" not in globals() or "val_hf" not in globals():
    raise NameError(
        "train_hf / val_hf are not defined. Re-run the feature extraction cell "
        "or reload them from disk before running this DoRA cell."
    )

# ------------------------------------------------------------
# 5) Clean datasets
# ------------------------------------------------------------
KEEP = {"input_features", "labels"}

def clean_ds(ds):
    drop_cols = [c for c in ds.column_names if c not in KEEP]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)
    ds.reset_format()
    return ds

train_clean = clean_ds(train_hf)
val_clean = clean_ds(val_hf)

print("train columns:", train_clean.column_names)
print("val columns  :", val_clean.column_names)

# ------------------------------------------------------------
# 6) Load base Whisper model
# ------------------------------------------------------------
print(f"⬇️ Loading {MODEL_NAME} for DoRA...")
dora_base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

if hasattr(dora_base.config, "suppress_tokens"):
    del dora_base.config.suppress_tokens
if hasattr(dora_base.config, "forced_decoder_ids"):
    del dora_base.config.forced_decoder_ids

dora_base.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
dora_base.generation_config.forced_decoder_ids = None
dora_base.generation_config.suppress_tokens = []

if hasattr(dora_base.config, "use_cache"):
    dora_base.config.use_cache = False

# Optional memory saver
dora_base.gradient_checkpointing_enable()

# ------------------------------------------------------------
# 7) Apply DoRA
# ------------------------------------------------------------
dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    use_dora=True,
)

dora_model = get_peft_model(dora_base, dora_config)
dora_model.print_trainable_parameters()
dora_model.to(device)

# ------------------------------------------------------------
# 8) Sanity check
# ------------------------------------------------------------
print("\n🧪 Sanity forward pass...")
sb = {k: v.to(device) for k, v in data_collator([train_clean[0], train_clean[1]]).items()}

train_model_dora = dora_model.get_base_model()
train_model_dora.to(device)

with torch.no_grad():
    out = train_model_dora(**sb)

print(f"✅ Sanity OK — loss: {float(out.loss):.4f}")

del sb, out
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 9) Training args (low-memory safer config)
# ------------------------------------------------------------
dora_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_DORA,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=50,
    num_train_epochs=3,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    predict_with_generate=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

# ------------------------------------------------------------
# 10) Trainer
# ------------------------------------------------------------
dora_trainer = Seq2SeqTrainer(
    model=train_model_dora,
    args=dora_args,
    train_dataset=train_clean,
    eval_dataset=val_clean,
    data_collator=data_collator,
)
# ------------------------------------------------------------
# 11) Train
# ------------------------------------------------------------
print("\n🚀 Training DoRA...")
train_result = dora_trainer.train()

# ------------------------------------------------------------
# 12) Save adapter
# ------------------------------------------------------------
os.makedirs(ADAPTER_DIR_DORA, exist_ok=True)
dora_model.save_pretrained(ADAPTER_DIR_DORA)
processor.save_pretrained(ADAPTER_DIR_DORA)

print(f"\n✅ DoRA adapter saved -> {ADAPTER_DIR_DORA}")

gc.collect()
torch.cuda.empty_cache()

device: cuda:0
⬇️ Loading processor for openai/whisper-small...
✅ Processor loaded
train columns: ['input_features', 'labels']
val columns  : ['input_features', 'labels']
⬇️ Loading openai/whisper-small for DoRA...
trainable params: 1,824,768 || all params: 243,559,680 || trainable%: 0.7492

🧪 Sanity forward pass...
✅ Sanity OK — loss: 4.8142

🚀 Training DoRA...


Step,Training Loss
20,5.731000
40,2.810500
60,1.613400
80,1.420300
100,1.126400
120,0.508300
140,0.290200
160,0.266100
180,0.258500
200,0.229900



✅ DoRA adapter saved -> /kaggle/working/whisper-dora-adapter


In [12]:
# ── Evaluate DoRA model ────────────────────────────────────────────────────────
from jiwer import wer, cer
from transformers.generation.configuration_utils import GenerationConfig

if hasattr(dora_model.config, 'suppress_tokens'):    del dora_model.config.suppress_tokens
if hasattr(dora_model.config, 'forced_decoder_ids'): del dora_model.config.forced_decoder_ids
dora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
dora_model.generation_config.forced_decoder_ids = None
dora_model.generation_config.suppress_tokens    = []
dora_model.eval()

print(f'🔬 Evaluating DoRA on {N_EVAL} samples (same fixed subset as baseline)...')
dora_preds = [
    transcribe_whisper(dora_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='DoRA eval')
]

dora_wer = wer(refs, dora_preds)
dora_cer = cer(refs, dora_preds)
dora_improvement = (baseline_wer - dora_wer) / baseline_wer * 100
print(f'\n📊 MODEL B — Whisper-small + DoRA')
print(f'   WER : {dora_wer*100:.2f}%  (baseline was {baseline_wer*100:.2f}%)  → {dora_improvement:+.1f}% vs baseline')
print(f'   CER : {dora_cer*100:.2f}%')

🔬 Evaluating DoRA on 200 samples (same fixed subset as baseline)...


DoRA eval: 100%|██████████| 200/200 [01:49<00:00,  1.83it/s]


📊 MODEL B — Whisper-small + DoRA
   WER : 9.60%  (baseline was 16.27%)  → +41.0% vs baseline
   CER : 4.80%


In [ ]:
# ── Free DoRA model before loading Wav2Vec2 ───────────────────────────────────
import gc
del dora_model
torch.cuda.empty_cache()
gc.collect()
print('✅ DoRA model freed from GPU memory.')

In [ ]:
# Run this AFTER DoRA completes
import gc, torch
try: del dora_trainer
except: pass
try: del dora_model
except: pass
try: del dora_base
except: pass
try: del train_model_dora
except: pass
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
print(f"✅ DoRA cleared. VRAM free: {free:.1f} GB — ready for Wav2Vec2")

---
## 7️⃣ Model C — Wav2Vec2-base Full Fine-Tuning (~2 hrs)

**Architecture difference from Whisper:**

| | Whisper | Wav2Vec2 |
|---|---|---|
| Architecture | Encoder-Decoder | Encoder-only + CTC head |
| Pre-training data | 680K hours supervised | 960 hours LibriSpeech |
| Decoding | Autoregressive (token by token) | CTC (parallel) |
| Input | Log-mel spectrogram | Raw waveform |
| Fine-tuning style | LoRA on attention layers | Full fine-tune (CNN frozen) |

**What is CTC decoding?**
CTC (Connectionist Temporal Classification) predicts one character/token per audio
frame in parallel, then collapses repeated predictions and removes blank tokens.
It is faster than Whisper's autoregressive decoding but typically less accurate
on complex vocabulary.

**Why uppercase the text for Wav2Vec2?**
The `wav2vec2-base-960h` model was pre-trained to predict uppercase characters only.
Feeding it lowercase transcripts would cause training to diverge. We uppercase
the labels for training, then lowercase predictions for fair WER comparison.

In [13]:
import librosa
import torch
from datasets import Dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Dict, List

W2V_MODEL_NAME = 'facebook/wav2vec2-base-960h'
print(f'⬇️  Loading {W2V_MODEL_NAME}...')
w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_MODEL_NAME)
w2v_model     = Wav2Vec2ForCTC.from_pretrained(W2V_MODEL_NAME)

w2v_model.freeze_feature_extractor()
w2v_model.to(device)

total     = sum(p.numel() for p in w2v_model.parameters())
trainable = sum(p.numel() for p in w2v_model.parameters() if p.requires_grad)
print(f'✅ Wav2Vec2 loaded: {total/1e6:.0f}M total, {trainable/1e6:.0f}M trainable ({trainable/total*100:.0f}%)')

def prepare_w2v(batch):
    audio, _ = librosa.load(batch['audio_path'], sr=16000)
    batch['input_values'] = w2v_processor(audio, sampling_rate=16000).input_values[0]
    batch['labels'] = w2v_processor.tokenizer(batch['text'].upper()).input_ids
    return batch

W2V_MAP = dict(
    remove_columns=['audio_path', 'text'],
    keep_in_memory=False,
    writer_batch_size=100,
    num_proc=1
)

print('🔄 Preparing Wav2Vec2 datasets (raw waveforms)...')
train_w2v = Dataset.from_pandas(train_df).map(prepare_w2v, **W2V_MAP)
val_w2v   = Dataset.from_pandas(val_df).map(prepare_w2v, **W2V_MAP)
print('✅ Wav2Vec2 datasets ready.')

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]) -> Dict:
        inputs = self.processor.pad(
            [{'input_values': f['input_values']} for f in features],
            return_tensors='pt',
            padding=True
        )

        labels = self.processor.tokenizer.pad(
            [{'input_ids': f['labels']} for f in features],
            return_tensors='pt',
            padding=True
        )

        inputs['labels'] = labels['input_ids'].masked_fill(
            labels.attention_mask.ne(1), -100
        )
        return inputs

w2v_collator = DataCollatorCTCWithPadding(processor=w2v_processor)

⬇️  Loading facebook/wav2vec2-base-960h...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Wav2Vec2 loaded: 94M total, 90M trainable (96%)
🔄 Preparing Wav2Vec2 datasets (raw waveforms)...


Map:   0%|          | 0/4884 [00:00<?, ? examples/s]

Map:   0%|          | 0/611 [00:00<?, ? examples/s]

✅ Wav2Vec2 datasets ready.


In [14]:
# ── Train Wav2Vec2 ────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer

# group_by_length=True: group samples of similar length into the same batch.
# This reduces padding waste (long and short sequences mixed = lots of PAD tokens).
# It's especially important for Wav2Vec2 whose input lengths vary widely.
w2v_args = TrainingArguments(
    output_dir=OUTPUT_DIR_W2V,
    group_by_length=True,           # reduces padding waste in variable-length audio
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # effective batch = 16
    learning_rate=1e-4,
    warmup_steps=100,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True,
    logging_steps=20,
    report_to='none',
)

w2v_trainer = Trainer(
    model=w2v_model,
    args=w2v_args,
    train_dataset=train_w2v,
    eval_dataset=val_w2v,
    data_collator=w2v_collator,
    tokenizer=w2v_processor.feature_extractor,
)

print('🚀 Training Wav2Vec2 — ~2 hours on P100...')
w2v_trainer.train()

os.makedirs(W2V_SAVE_DIR, exist_ok=True)
w2v_model.save_pretrained(W2V_SAVE_DIR)
w2v_processor.save_pretrained(W2V_SAVE_DIR)
print(f'\n✅ Wav2Vec2 saved → {W2V_SAVE_DIR}')

🚀 Training Wav2Vec2 — ~2 hours on P100...


Epoch,Training Loss,Validation Loss
1,230.543800,220.137375
2,172.761500,160.706085
3,211.528800,156.439774
4,128.844900,135.273682
5,117.368100,135.554428
6,134.834800,119.116150
7,108.283900,112.829071
8,94.459200,112.724007
9,109.431000,118.117470
10,84.674100,117.524605


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


✅ Wav2Vec2 saved → /kaggle/working/wav2vec2-saved


In [15]:
# ── Evaluate Wav2Vec2 ─────────────────────────────────────────────────────────
# CTC transcription is different from Whisper's generate():
#   1. Forward pass outputs logits: shape (1, time_steps, vocab_size)
#   2. argmax picks the highest-probability token at each time step
#   3. batch_decode applies CTC collapse: remove blanks, remove repeated tokens
#      e.g. [H,H,E,_,L,L,L,_,_,O] → 'HELLO'
# We lowercase the output to match our ground-truth refs (which are lowercase).
w2v_model.eval()

def transcribe_w2v(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = w2v_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = w2v_model(**inputs).logits  # shape: (1, T, vocab_size)
    # CTC greedy decode: argmax over vocab at each frame, then collapse
    ids = torch.argmax(logits, dim=-1)
    return w2v_processor.batch_decode(ids)[0].strip().lower()

print(f'🔬 Evaluating Wav2Vec2 on {N_EVAL} samples (same fixed subset as all other models)...')
w2v_preds = [
    transcribe_w2v(p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='Wav2Vec2 eval')
]

w2v_wer = wer(refs, w2v_preds)
w2v_cer = cer(refs, w2v_preds)
print(f'\n📊 MODEL C — Wav2Vec2-base (full fine-tune)')
print(f'   WER : {w2v_wer*100:.2f}%')
print(f'   CER : {w2v_cer*100:.2f}%')

🔬 Evaluating Wav2Vec2 on 200 samples (same fixed subset as all other models)...


Wav2Vec2 eval: 100%|██████████| 200/200 [00:07<00:00, 27.64it/s]


📊 MODEL C — Wav2Vec2-base (full fine-tune)
   WER : 16.85%
   CER : 10.25%


---
## 8️⃣ Final Results Table

In [37]:
# ── Cell 2: Paths ──────────────────────────────────────────────────────────────
LORA_ADAPTER_DIR = "/kaggle/input/datasets/aymendhieb1/whisper-adapter-final-v1/whisper_adapter_final"
DORA_ADAPTER_DIR = "/kaggle/input/datasets/aymendhieb1/whisper-dora-adapter-v1/whisper_dora_adapter"
W2V_DIR          = "/kaggle/input/datasets/aymendhieb1/wav2vec2-saved-v1/wav2vec2-saved"
MODEL_NAME       = "openai/whisper-small"
device           = "cuda"


In [24]:
# ── Debug 4: show exact file structure inside recordings ──────────────────────
import os

DATA_DIR = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings"

for split in os.listdir(DATA_DIR):
    split_dir = os.path.join(DATA_DIR, split)
    if not os.path.isdir(split_dir): continue
    files = os.listdir(split_dir)
    print(f"\n📁 {split}/ — {len(files)} files")
    print(f"   First 5: {files[:5]}")
    
    # check for subfolders
    subfolders = [f for f in files if os.path.isdir(os.path.join(split_dir, f))]
    if subfolders:
        print(f"   Subfolders: {subfolders}")
        for sf in subfolders[:2]:
            sf_path = os.path.join(split_dir, sf)
            sf_files = os.listdir(sf_path)
            print(f"   📁 {sf}/ — {sf_files[:5]}")


📁 validate/ — 385 files
   First 5: ['1249120_44294866_15891095.wav', '1249120_44263136_25998544.wav', '1249120_44263136_58938609.wav', '1249120_44294866_77416341.wav', '1249120_44323331_53313006.wav']

📁 test/ — 5895 files
   First 5: ['1249120_44101988_103474667.wav', '1249120_42210938_34058782.wav', '1249120_44093303_57046343.wav', '1249120_39740177_38724108.wav', '1249120_39740177_95271830.wav']

📁 train/ — 381 files
   First 5: ['1249120_44160489_107692984.wav', '1249120_44176037_39613511.wav', '1249120_44176037_85065458.wav', '1249120_44194084_23344116.wav', '1249120_44220382_48462181.wav']


In [25]:
# ── Debug 6: find the CSV ─────────────────────────────────────────────────────
import os

BASE = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"

for root, dirs, files in os.walk(BASE):
    for f in files:
        if not f.endswith(".wav"):
            print(os.path.join(root, f))

/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv


In [26]:
# ── Cell 3: Load dataset + build eval subset ──────────────────────────────────
import pandas as pd, os, librosa
from sklearn.model_selection import train_test_split

BASE_DIR  = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"
CSV_PATH  = os.path.join(BASE_DIR, "overview-of-recordings.csv")
AUDIO_DIR = os.path.join(BASE_DIR, "recordings")

# Load the CSV that maps filename → transcription
csv_df = pd.read_csv(CSV_PATH)
print("CSV columns:", csv_df.columns.tolist())
print(csv_df.head(3))

CSV columns: ['audio_clipping', 'audio_clipping:confidence', 'background_noise_audible', 'background_noise_audible:confidence', 'overall_quality_of_the_audio', 'quiet_speaker', 'quiet_speaker:confidence', 'speaker_id', 'file_download', 'file_name', 'phrase', 'prompt', 'writer_id']
   audio_clipping  audio_clipping:confidence background_noise_audible  \
0     no_clipping                     1.0000              light_noise   
1  light_clipping                     0.6803                 no_noise   
2     no_clipping                     1.0000                 no_noise   

   background_noise_audible:confidence  overall_quality_of_the_audio  \
0                               1.0000                          3.33   
1                               0.6803                          3.33   
2                               0.6655                          3.33   

     quiet_speaker  quiet_speaker:confidence  speaker_id  \
0  audible_speaker                       1.0    43453425   
1  audible_speak

In [27]:
# ── Cell 3: Load dataset + build eval subset ──────────────────────────────────
import pandas as pd, os, librosa
from sklearn.model_selection import train_test_split

BASE_DIR  = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"
CSV_PATH  = os.path.join(BASE_DIR, "overview-of-recordings.csv")
AUDIO_DIR = os.path.join(BASE_DIR, "recordings")

# Load CSV
csv_df = pd.read_csv(CSV_PATH)

# Find the actual audio file by searching all split subfolders
def find_audio(filename):
    for split in ["train", "validate", "test"]:
        path = os.path.join(AUDIO_DIR, split, filename)
        if os.path.exists(path):
            return path
    return None

rows = []
for _, row in csv_df.iterrows():
    audio_path = find_audio(row["file_name"])
    if audio_path:
        rows.append({
            "audio_path": audio_path,
            "text": str(row["phrase"]).strip().lower()
        })

df = pd.DataFrame(rows)
print(f"Total rows found: {len(df)}")
assert len(df) > 0, "Still empty!"

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

eval_subset = test_df.sample(n=min(200, len(test_df)), random_state=42).reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Eval subset: {len(eval_subset)} samples")
print(eval_subset[["audio_path","text"]].head(3))

Total rows found: 6661
Train: 5328 | Val: 666 | Test: 667
Eval subset: 200 samples
                                          audio_path  \
0  /kaggle/input/datasets/paultimothymooney/medic...   
1  /kaggle/input/datasets/paultimothymooney/medic...   
2  /kaggle/input/datasets/paultimothymooney/medic...   

                                                text  
0           i have a sharp pain in my lower stomach.  
1                             severe pain in the ear  
2  i feel something hurt me in taking breath and ...  


In [32]:
# ── Cell 4: WER/CER helper ────────────────────────────────────────────────────
from jiwer import wer, cer

def compute_metrics(refs, hyps):
    w = wer(refs, hyps)
    c = cer(refs, hyps)
    return w, c

In [33]:
# ── Cell 5: Baseline — Whisper-small, no fine-tuning ─────────────────────────
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor    = WhisperProcessor.from_pretrained(MODEL_NAME)
base_model   = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device).eval()

def transcribe_whisper(model, audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = processor(audio, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=128,
                             language="english", task="transcribe")
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(base_model, row["audio_path"]))

baseline_wer, baseline_cer = compute_metrics(refs, hyps)
print(f"Baseline WER: {baseline_wer*100:.2f}%  CER: {baseline_cer*100:.2f}%")

del base_model; torch.cuda.empty_cache()

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


Baseline WER: 17.93%  CER: 7.78%


In [38]:
# ── Cell 6: Whisper + LoRA — load adapter and evaluate ────────────────────────
from peft import PeftModel

lora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
lora_model = PeftModel.from_pretrained(lora_base, LORA_ADAPTER_DIR).to(device).eval()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(lora_model, row["audio_path"]))

lora_wer, lora_cer = compute_metrics(refs, hyps)
print(f"LoRA WER: {lora_wer*100:.2f}%  CER: {lora_cer*100:.2f}%")

del lora_base, lora_model; torch.cuda.empty_cache()

LoRA WER: 2.81%  CER: 1.64%


In [39]:
# ── Cell 7: Whisper + DoRA — same thing ───────────────────────────────────────
dora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
dora_model = PeftModel.from_pretrained(dora_base, DORA_ADAPTER_DIR).to(device).eval()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(dora_model, row["audio_path"]))

dora_wer, dora_cer = compute_metrics(refs, hyps)
print(f"DoRA WER: {dora_wer*100:.2f}%  CER: {dora_cer*100:.2f}%")

del dora_base, dora_model; torch.cuda.empty_cache()

DoRA WER: 9.50%  CER: 4.52%


In [40]:
# ── Cell 8: Wav2Vec2 — load saved model and evaluate ─────────────────────────
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import numpy as np

w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_DIR)
w2v_model     = Wav2Vec2ForCTC.from_pretrained(W2V_DIR).to(device).eval()

def transcribe_wav2vec(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = w2v_processor(audio, sampling_rate=16000, return_tensors="pt",
                              padding=True).to(device)
    with torch.no_grad():
        logits = w2v_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return w2v_processor.batch_decode(pred_ids)[0].strip().lower()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_wav2vec(row["audio_path"]))

w2v_wer, w2v_cer = compute_metrics(refs, hyps)
print(f"Wav2Vec2 WER: {w2v_wer*100:.2f}%  CER: {w2v_cer*100:.2f}%")

del w2v_model; torch.cuda.empty_cache()

Wav2Vec2 WER: 14.61%  CER: 8.42%


In [41]:
# ── Cell 9: Final comparison table (your original code — now it will work!) ───
results = pd.DataFrame([
    {"Model": "Whisper-small", "Method": "No fine-tuning (baseline)",
     "Trainable params": "244M (100%)", "WER": f"{baseline_wer*100:.2f}%",
     "CER": f"{baseline_cer*100:.2f}%", "vs Baseline": "—"},
    {"Model": "Whisper-small", "Method": "LoRA (r=16, use_dora=False)",
     "Trainable params": "2.3M (~1%)", "WER": f"{lora_wer*100:.2f}%",
     "CER": f"{lora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer-lora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Whisper-small", "Method": "DoRA (r=16, use_dora=True)",
     "Trainable params": "2.3M (~1%)", "WER": f"{dora_wer*100:.2f}%",
     "CER": f"{dora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer-dora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Wav2Vec2-base", "Method": "Full fine-tuning (CNN frozen)",
     "Trainable params": "~90M", "WER": f"{w2v_wer*100:.2f}%",
     "CER": f"{w2v_cer*100:.2f}%", "vs Baseline": "N/A (different arch)"},
])

print("\n" + "="*75)
print("  🏆 FINAL RESULTS — Medical ASR Domain Adaptation Study")
print("="*75)
print(results.to_string(index=False))


  🏆 FINAL RESULTS — Medical ASR Domain Adaptation Study
        Model                        Method Trainable params    WER   CER          vs Baseline
Whisper-small     No fine-tuning (baseline)      244M (100%) 17.93% 7.78%                    —
Whisper-small   LoRA (r=16, use_dora=False)       2.3M (~1%)  2.81% 1.64%               +84.3%
Whisper-small    DoRA (r=16, use_dora=True)       2.3M (~1%)  9.50% 4.52%               +47.0%
Wav2Vec2-base Full fine-tuning (CNN frozen)             ~90M 14.61% 8.42% N/A (different arch)


In [42]:
# ── Save final results to CSV ─────────────────────────────────────────────────
import pandas as pd

results = pd.DataFrame([
    {"Model": "Whisper-small", "Method": "No fine-tuning (baseline)",
     "Trainable params": "244M (100%)", "WER": f"{baseline_wer*100:.2f}%",
     "CER": f"{baseline_cer*100:.2f}%", "vs Baseline": "—"},
    {"Model": "Whisper-small", "Method": "LoRA (r=16, use_dora=False)",
     "Trainable params": "2.3M (~1%)", "WER": f"{lora_wer*100:.2f}%",
     "CER": f"{lora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer - lora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Whisper-small", "Method": "DoRA (r=16, use_dora=True)",
     "Trainable params": "2.3M (~1%)", "WER": f"{dora_wer*100:.2f}%",
     "CER": f"{dora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer - dora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Wav2Vec2-base", "Method": "Full fine-tuning (CNN frozen)",
     "Trainable params": "~90M", "WER": f"{w2v_wer*100:.2f}%",
     "CER": f"{w2v_cer*100:.2f}%", "vs Baseline": "N/A (different arch)"},
])

results.to_csv("/kaggle/working/final_results.csv", index=False)
print("✅ Saved! Download from the Output panel on the right.")
print(results.to_string(index=False))

✅ Saved! Download from the Output panel on the right.
        Model                        Method Trainable params    WER   CER          vs Baseline
Whisper-small     No fine-tuning (baseline)      244M (100%) 17.93% 7.78%                    —
Whisper-small   LoRA (r=16, use_dora=False)       2.3M (~1%)  2.81% 1.64%               +84.3%
Whisper-small    DoRA (r=16, use_dora=True)       2.3M (~1%)  9.50% 4.52%               +47.0%
Wav2Vec2-base Full fine-tuning (CNN frozen)             ~90M 14.61% 8.42% N/A (different arch)


---
## 🎤 Inference Demo — Transcribe Any Audio File

In [44]:
# ── Inference: Load best model (LoRA wins with 2.81% WER) ────────────────────
import torch, librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.generation.configuration_utils import GenerationConfig
from peft import PeftModel

scores = {'LoRA': lora_wer, 'DoRA': dora_wer}
best   = min(scores, key=scores.get)
BEST_ADAPTER_DIR = LORA_ADAPTER_DIR if best == 'LoRA' else DORA_ADAPTER_DIR
print(f'🏆 Best model: {best}  (WER = {scores[best]*100:.2f}%)')

# Load base + best adapter
inf_base      = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
inf_model     = PeftModel.from_pretrained(inf_base, BEST_ADAPTER_DIR).to(device).eval()
inf_processor = WhisperProcessor.from_pretrained(BEST_ADAPTER_DIR)
inf_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
inf_model.generation_config.forced_decoder_ids = None
inf_model.generation_config.suppress_tokens    = []
print('✅ Ready for inference.')

def transcribe_file(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = inf_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = inf_model.generate(
            **inputs,
            max_new_tokens=128,
            max_length=None,
            language='english',
            task='transcribe'
        )
    return inf_processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

# ── Demo on 5 test samples ────────────────────────────────────────────────────
print('\n🎤 Demo transcriptions from test set:')
print('-' * 65)
for _, row in test_df.head(5).iterrows():
    pred  = transcribe_file(row['audio_path'])
    match = '✅' if pred.strip() == row['text'].strip() else '❌'
    print(f'  {match} Truth : {row["text"]}')
    print(f'     Pred  : {pred}')
    print()

🏆 Best model: LoRA  (WER = 2.81%)
✅ Ready for inference.

🎤 Demo transcriptions from test set:
-----------------------------------------------------------------
  ✅ Truth : i feel like my heart is on fire.
     Pred  : i feel like my heart is on fire.

  ✅ Truth : i feel abdominal pain
     Pred  : i feel abdominal pain

  ✅ Truth : i think my wound is infected
     Pred  : i think my wound is infected

  ✅ Truth : i feel a tightness in my chest
     Pred  : i feel a tightness in my chest

  ✅ Truth : i do not feel better in my muscles
     Pred  : i do not feel better in my muscles



In [ ]:
# ── Transcribe your own audio file ────────────────────────────────────────────
# To use with your own recording:
#   1. Click the folder icon in the left panel of Kaggle
#   2. Upload your .wav file
#   3. Set MY_AUDIO to the uploaded path
#   4. Uncomment the two lines below and run

# MY_AUDIO = '/kaggle/working/my_recording.wav'   # ← change to your file path
# print(f'Transcription: {transcribe_file(MY_AUDIO)}')

print('ℹ️  Upload a .wav file and uncomment the lines above to transcribe it.')

---
## Check dataset structure first

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL A — Install extra libs needed for TEDxTN
# ═══════════════════════════════════════════════════════════════════════════════
# jiwer and evaluate are already installed. We just need datasets upgraded
# to handle the TEDxTN CSV properly.

!pip -q install --upgrade datasets==2.20.0 evaluate jiwer
print("✅ Libraries ready.")



In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL B — Download TEDxTN (Tunisian Arabic) — train.csv ONLY
# ═══════════════════════════════════════════════════════════════════════════════
# The dataset has TWO csv files: train.csv (Arabic transcription) and
# train.en.csv (English translation). Loading both at once causes a
# CastError because the column names differ ('transcription' vs 'translation').
# Fix: specify data_files explicitly to load only train.csv.
#
# TEDxTN contains ~15,700 samples of Tunisian Arabic TED talk speech.
# Each row: audio_filename, start_time, end_time, transcription.
# NOTE: This dataset provides metadata only (no raw audio files embedded).
# The audio must be downloaded separately — see the dataset README.
# For fine-tuning Whisper we need the actual .wav files.
# Strategy: use the HuggingFace streaming approach + download audio on-the-fly.

from datasets import load_dataset
import pandas as pd

print("⬇️  Loading TEDxTN metadata (train.csv only)...")

ds_tn = load_dataset(
    "fbougares/TEDxTN",
    data_files={"train": "train.csv"},   # ← KEY FIX: avoids column mismatch
    split="train"
)

print(f"✅ TEDxTN loaded: {len(ds_tn):,} rows")
print(f"   Columns: {ds_tn.column_names}")
print(f"   Sample: {ds_tn[0]}")

⬇️  Loading TEDxTN metadata (train.csv only)...


✅ TEDxTN loaded: 15,703 rows
   Columns: ['audio_filename', 'start_time', 'end_time', 'transcription']
   Sample: {'audio_filename': 'Crois_en_toi_et_tout_le_reste_suivra_Jaafer_Guesmi_TEDxNabeul', 'start_time': 5.179, 'end_time': 8.574, 'transcription': 'آنا مواطن تونسي من مدنين من قرية إسمها أم التمر'}


In [4]:




# ═══════════════════════════════════════════════════════════════════════════════
# CELL C — Inspect & prepare TEDxTN dataframe
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd
from sklearn.model_selection import train_test_split

# Convert to pandas for easy manipulation
tn_df = ds_tn.to_pandas()

# Normalise transcription: strip whitespace, keep Arabic script as-is
# (do NOT lowercase Arabic — case doesn't exist in Arabic script)
tn_df['text'] = tn_df['transcription'].astype(str).str.strip()

# Drop rows with empty transcription
tn_df = tn_df[tn_df['text'].str.len() > 0].reset_index(drop=True)

print(f"📊 After cleaning: {len(tn_df):,} rows")
print(f"   Sample transcriptions:")
for t in tn_df['text'].head(5).tolist():
    print(f"   → {t[:80]}")

# 80/10/10 split (same strategy as your medical notebook)
tn_train, tn_temp = train_test_split(tn_df, test_size=0.20, random_state=42)
tn_val,   tn_test = train_test_split(tn_temp, test_size=0.50, random_state=42)

tn_train = tn_train.reset_index(drop=True)
tn_val   = tn_val.reset_index(drop=True)
tn_test  = tn_test.reset_index(drop=True)

print(f"\n✂️  Split: train={len(tn_train):,}  val={len(tn_val):,}  test={len(tn_test):,}")


📊 After cleaning: 15,702 rows
   Sample transcriptions:
   → آنا مواطن تونسي من مدنين من قرية إسمها أم التمر
   → من عايلة فقيرة جدا
   → توا الأكيد كي بش نبدا نحكي برشا بش يقولوا وكأنه يتاجر يعني بماضيه برشا برشا قالو
   → ولكن النضال أشكال
   → آنا كي حليت عينيا أول فشل الساعة آنا نمن بالفشل

✂️  Split: train=12,561  val=1,570  test=1,571


In [9]:
import os, gc, time, shutil, numpy as np
import librosa
from datasets import Dataset, concatenate_datasets, load_from_disk

def build_tn_dataset_chunked(hf_ds, split_name, save_base, chunk_size=50, audio_col="audio", text_col="transcription"):
    total = len(hf_ds)
    n_chunks = (total + chunk_size - 1) // chunk_size
    chunk_dirs = []

    print(f"\n🔄 {split_name}: {total:,} samples in {n_chunks} chunks...")
    print("Columns:", hf_ds.column_names)

    if audio_col not in hf_ds.column_names:
        raise ValueError(
            f"Column '{audio_col}' not found. Available columns: {hf_ds.column_names}"
        )
    if text_col not in hf_ds.column_names:
        raise ValueError(
            f"Column '{text_col}' not found. Available columns: {hf_ds.column_names}"
        )

    t0 = time.time()

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, total)
        chunk_slice = hf_ds.select(range(start, end))

        rows = []
        for i, ex in enumerate(chunk_slice):
            try:
                audio_obj = ex[audio_col]

                if audio_obj is None:
                    raise ValueError("audio object is None")

                audio_array = np.asarray(audio_obj["array"], dtype=np.float32)
                sr = audio_obj["sampling_rate"]

                if sr != TARGET_SR:
                    audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)

                text = ex[text_col]
                if text is None or not str(text).strip():
                    raise ValueError("empty transcription")

                feats = processor.feature_extractor(
                    audio_array,
                    sampling_rate=TARGET_SR
                ).input_features[0]

                labels = processor.tokenizer(str(text).strip()).input_ids

                rows.append({
                    "input_features": np.asarray(feats, dtype=np.float32),
                    "labels": labels,
                })

            except Exception as e:
                print(f"   ⚠️ Skipping sample {start + i}: {repr(e)}")
                continue

        if len(rows) == 0:
            print(f"   ⚠️ chunk {chunk_idx+1}/{n_chunks} produced 0 valid rows — skipped")
            continue

        chunk_dir = f"{save_base}_chunk_{chunk_idx:04d}"
        Dataset.from_list(rows).save_to_disk(chunk_dir)
        chunk_dirs.append(chunk_dir)

        del rows
        gc.collect()

        elapsed = time.time() - t0
        rate = end / elapsed if elapsed > 0 else 0
        eta = (total - end) / rate if rate > 0 else 0
        print(f"   chunk {chunk_idx+1:3d}/{n_chunks} | {end:,}/{total:,} | {rate:.1f} ex/s | ETA {eta/60:.1f} min")

    if len(chunk_dirs) == 0:
        raise RuntimeError(
            f"No valid chunks were created for {split_name}. "
            f"Check '{audio_col}' and '{text_col}' columns."
        )

    print(f"   Concatenating {len(chunk_dirs)} chunks...")
    all_chunks = [load_from_disk(d) for d in chunk_dirs]
    full_ds = concatenate_datasets(all_chunks)
    full_ds.save_to_disk(save_base)

    for d in chunk_dirs:
        shutil.rmtree(d, ignore_errors=True)

    print(f"✅ {split_name} done: {len(full_ds):,} samples → {save_base}")
    return full_ds
   

In [ ]:



# ═══════════════════════════════════════════════════════════════════════════════
# CELL E — Load your existing LoRA adapter + continue fine-tuning on Tunisian
# ═══════════════════════════════════════════════════════════════════════════════
# Strategy:
#   1. Load whisper-small base
#   2. Load your trained medical LoRA adapter on top
#   3. Continue training on TEDxTN with a LOWER learning rate (1e-5 not 3e-4)
#      to avoid catastrophic forgetting of the audio-text alignment
#   4. Only 3 epochs — Tunisian dialect adapts fast on top of a trained adapter

import os, gc, pathlib, torch
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

# ── Paths ─────────────────────────────────────────────────────────────────────
# Your saved LoRA adapter from the medical training run
MEDICAL_ADAPTER = "/kaggle/input/datasets/aymendhieb1/whisper-adapter-final-v1/whisper_adapter_final"

# Where to save the new Tunisian adapter
TN_ADAPTER_DIR  = "/kaggle/working/whisper-tunisian-adapter"
TN_TRAIN_DIR    = "/kaggle/working/whisper-tunisian-training"

# ── Single GPU setup (same as your existing cells) ────────────────────────────
os.environ.pop("MASTER_ADDR", None)
os.environ.pop("MASTER_PORT", None)
os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

gc.collect()
torch.cuda.empty_cache()
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Training device: {device}")

# ── Data collator (identical to your existing one) ────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        bos_id = processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]
        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

tn_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ── Clean datasets ────────────────────────────────────────────────────────────
KEEP = {"input_features", "labels"}

def clean_ds(ds):
    drop = [c for c in ds.column_names if c not in KEEP]
    if drop:
        ds = ds.remove_columns(drop)
    ds.reset_format()
    return ds

tn_train_clean = clean_ds(tn_train_hf)
tn_val_clean   = clean_ds(tn_val_hf)

print("train columns:", tn_train_clean.column_names)
print("val columns  :", tn_val_clean.column_names)

# ── Load base Whisper ─────────────────────────────────────────────────────────
print(f"\n⬇️  Loading {MODEL_NAME}...")
tn_base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

for attr in ["suppress_tokens", "forced_decoder_ids"]:
    if hasattr(tn_base.config, attr):
        delattr(tn_base.config, attr)

tn_base.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
tn_base.generation_config.forced_decoder_ids = None
tn_base.generation_config.suppress_tokens    = []
tn_base.config.use_cache = False

# ── Load your existing medical LoRA adapter on top ───────────────────────────
print(f"📦 Loading medical LoRA adapter from {MEDICAL_ADAPTER}...")
tn_model = PeftModel.from_pretrained(
    tn_base,
    MEDICAL_ADAPTER,
    is_trainable=True,       # keep adapter weights trainable
)
tn_model.to(device)
tn_model.print_trainable_parameters()

train_model_tn = tn_model.get_base_model()
train_model_tn.to(device)

# ── Sanity forward pass ───────────────────────────────────────────────────────
print("\n🧪 Sanity forward pass...")
sb = tn_collator([tn_train_clean[0], tn_train_clean[1]])
sb = {k: v.to(device) for k, v in sb.items()}
with torch.no_grad():
    out = train_model_tn(**sb)
print(f"✅ Sanity OK — loss: {float(out.loss):.4f}")
del sb, out
gc.collect()
torch.cuda.empty_cache()

# ── Training arguments ────────────────────────────────────────────────────────
# KEY DIFFERENCES from your medical training:
#   learning_rate = 1e-5   (was 3e-4) — lower LR avoids overwriting medical knowledge
#   num_train_epochs = 3   (was 10)   — Tunisian adapts faster on existing adapter
#   warmup_steps = 100                — gentle warmup for continued fine-tuning

tn_args = Seq2SeqTrainingArguments(
    output_dir=TN_TRAIN_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=1e-5,              # ← LOWER than medical (3e-4)
    warmup_steps=100,
    num_train_epochs=3,              # ← fewer epochs, adapter already warm
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    predict_with_generate=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
tn_trainer = Seq2SeqTrainer(
    model=train_model_tn,
    args=tn_args,
    train_dataset=tn_train_clean,
    eval_dataset=tn_val_clean,
    data_collator=tn_collator,
)

# ── Train! ────────────────────────────────────────────────────────────────────
print("\n🚀 Fine-tuning on Tunisian Arabic...")
print("   Estimated time on T4 x2 (single GPU): ~2.5–3 hours")
tn_trainer.train()

# ── Save adapter ──────────────────────────────────────────────────────────────
os.makedirs(TN_ADAPTER_DIR, exist_ok=True)
tn_model.save_pretrained(TN_ADAPTER_DIR)
processor.save_pretrained(TN_ADAPTER_DIR)
print(f"\n✅ Tunisian LoRA adapter saved → {TN_ADAPTER_DIR}")

gc.collect()
torch.cuda.empty_cache()


In [4]:
import os, gc, torch, librosa
import numpy as np
from datasets import load_dataset, Audio
from datasets import Dataset, concatenate_datasets, load_from_disk
from dataclasses import dataclass
from typing import Any, Dict, List

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_ETAG_TIMEOUT"]     = "120"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

TARGET_SR  = 16_000
CHUNK_SIZE = 200
SAVE_BASE  = "/kaggle/working/tedxtn_processed"
os.makedirs(SAVE_BASE, exist_ok=True)

# ── Step 1: Load dataset ──────────────────────────────────────────────────────
print("⬇️  Loading Sabrinek8/tedxtn-train...")
ds = load_dataset(
    "Sabrinek8/tedxtn-train",
    split="train",
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

print(f"✅ Loaded: {len(ds):,} samples")
print(f"   Columns: {ds.column_names}")
print(f"   Sample text: {ds[0]['text']}")
print(f"   Sample audio SR: {ds[0]['audio']['sampling_rate']}")

# ── Step 2: Split 80/10/10 ────────────────────────────────────────────────────
ds = ds.shuffle(seed=42)
n  = len(ds)
n_test = int(n * 0.10)
n_val  = int(n * 0.10)

test_ds  = ds.select(range(0, n_test))
val_ds   = ds.select(range(n_test, n_test + n_val))
train_ds = ds.select(range(n_test + n_val, n))

print(f"\n✂️  Split: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

# ── Step 3: Feature extraction function ───────────────────────────────────────
def process_one(sample):
    try:
        audio = sample["audio"]["array"].astype(np.float32)
        text  = str(sample["text"]).strip()

        duration = len(audio) / TARGET_SR
        if duration < 0.3 or duration > 30.0:
            return None
        if len(text) < 2:
            return None

        input_features = processor.feature_extractor(
            audio, sampling_rate=TARGET_SR
        ).input_features[0]

        labels = processor.tokenizer(text).input_ids

        return {
            "input_features": input_features,
            "labels": labels,
        }
    except Exception:
        return None

# ── Step 4: Process and save in chunks ────────────────────────────────────────
def process_and_save(split_ds, split_name):
    save_dir    = os.path.join(SAVE_BASE, split_name)
    final_path  = save_dir + "_final"

    # Skip if already done
    if os.path.exists(final_path):
        print(f"⚡ {split_name}: loading from cache...")
        return load_from_disk(final_path)

    os.makedirs(save_dir, exist_ok=True)
    chunk_dirs = []
    chunk_idx  = 0
    rows       = []
    total_ok   = 0
    total_skip = 0
    n          = len(split_ds)

    print(f"\n🔄 Processing {split_name}: {n:,} samples...")

    for i in range(n):
        sample = split_ds[i]
        result = process_one(sample)

        if result is not None:
            rows.append(result)
            total_ok += 1
        else:
            total_skip += 1

        # Save chunk
        if len(rows) >= CHUNK_SIZE:
            chunk_path = os.path.join(save_dir, f"chunk_{chunk_idx:04d}")
            Dataset.from_list(rows).save_to_disk(chunk_path)
            chunk_dirs.append(chunk_path)
            rows      = []
            chunk_idx += 1
            gc.collect()

        if i % 500 == 0:
            pct = i / n * 100
            print(f"   [{i:,}/{n:,}] {pct:.1f}%  ok={total_ok:,}  skip={total_skip:,}")

    # Save last chunk
    if rows:
        chunk_path = os.path.join(save_dir, f"chunk_{chunk_idx:04d}")
        Dataset.from_list(rows).save_to_disk(chunk_path)
        chunk_dirs.append(chunk_path)

    if not chunk_dirs:
        raise RuntimeError(f"❌ 0 valid samples for {split_name} — check audio/text columns")

    # Merge all chunks
    print(f"   📦 Merging {len(chunk_dirs)} chunks...")
    merged = concatenate_datasets([load_from_disk(d) for d in chunk_dirs])
    merged.save_to_disk(final_path)

    print(f"✅ {split_name}: {total_ok:,} valid  {total_skip:,} skipped")
    print(f"   Saved → {final_path}")
    return merged

# ── Step 5: Run extraction ────────────────────────────────────────────────────
# Estimated time on Kaggle P100: ~1 hour for all 15k samples
train_hf_tn = process_and_save(train_ds, "train")
val_hf_tn   = process_and_save(val_ds,   "val")
test_hf_tn  = process_and_save(test_ds,  "test")

print(f"\n✅ All done!")
print(f"   train : {len(train_hf_tn):,}")
print(f"   val   : {len(val_hf_tn):,}")
print(f"   test  : {len(test_hf_tn):,}")

⬇️  Loading Sabrinek8/tedxtn-train...


README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

data/train-00000-of-00030.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]

data/train-00001-of-00030.parquet:   0%|          | 0.00/533M [00:00<?, ?B/s]

data/train-00002-of-00030.parquet:   0%|          | 0.00/425M [00:00<?, ?B/s]

...

data/train-00006-of-00030.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00007-of-00030.parquet:   0%|          | 0.00/434M [00:00<?, ?B/s]

data/train-00008-of-00030.parquet:   0%|          | 0.00/371M [00:00<?, ?B/s]

data/train-00009-of-00030.parquet:   0%|          | 0.00/610M [00:00<?, ?B/s]

KeyboardInterrupt: 

💾 Disk space — /kaggle/working
   Total : 21.0 GB
   Used  : 0.0 GB
   Free  : 20.9 GB

✅ After cleanup — Free: 20.9 GB
⬇️  Loading Sabrinek8/tedxtn-train...


README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



✅ Tunisian adapter saved → /kaggle/working/whisper_tunisian_adapter
🗑️  Removed checkpoint dir to free disk space
💾 Disk used: 0.0 GB  free: 20.9 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.6 MB/s eta 0:00:0000:0100:01
🔬 Evaluating on 200 Tunisian test samples...


TN eval:   0%|          | 0/200 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transf


  📊 Whisper-small + LoRA — Tunisian Arabic
  WER : 79.39%
  CER : 46.06%

🎤 Sample predictions (first 5):
  ❌ REF : كيفاش تحبوا بلادنا تشد بعضها وقت اللي عندك زوز ملاين عباد عايشين هكا
     PRED: كيفش تحبوا بلادنا تشربت بعضها وخطيلي عندك زوز ملين عباد عشين هكا

  ❌ REF : علاش إنحبوا مثلا المساواة في الميراث
     PRED: هو يشهد ميس كميس تنجل جاب

  ❌ REF : بعد أخذ ورد وعطاء
     PRED: بعد أخذا ورد وعطا

  ❌ REF : باهي هاذي juste بش إنفدلك وأكاهو
     PRED: البيعي هذه يجيس بشرفات الكوكا

  ❌ REF : إذا كان ثمة حاجات في الحياة تمشي كيما إنحبوا أحنا
     PRED: إذا كان ثم حاجاتها الحياة دمشي كما نحبوا أحنا

✅ Results saved → /kaggle/working/tunisian_results.csv
        Model                           Adapter                         Dataset  Eval samples    WER    CER  Epochs Learning rate
Whisper-small LoRA medical → Tunisian continued TEDxTN (Sabrinek8/tedxtn-train)           200 79.39% 46.06%      15          1e-6

📦 Adapter zipped: 7.3 MB → /kaggle/working/whisper_tunisian_adapter.zip
📥 D

In [17]:
# =========================================================
# CELL 1: Install packages
# =========================================================
!pip install -q transformers accelerate bitsandbytes datasets librosa sentencepiece peft

# =========================================================
# CELL 2: Imports
# =========================================================
import os
import torch
from datasets import load_dataset, Audio
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    WhisperProcessor,
    WhisperForConditionalGeneration
)
from peft import PeftModel, PeftConfig

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Device:", DEVICE)
print("Dtype :", DTYPE)

# =========================================================
# CELL 3: Load a small Tunisian dataset sample for testing
# =========================================================
TARGET_SR = 16000

print("Loading dataset...")
ds = load_dataset("Sabrinek8/tedxtn-train", split="train[:5]")
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

print("Loaded dataset.")
print("Columns:", ds.column_names)
print("Length :", len(ds))
print("Sample text:", ds[0]["text"])
print("Sample SR  :", ds[0]["audio"]["sampling_rate"])

# =========================================================
# CELL 4: Load Qwen LLM for Darija correction
# =========================================================
llm_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading Qwen tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(llm_name)

print("Loading Qwen model...")
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Qwen loaded.")

# =========================================================
# CELL 5: Darija correction function
# =========================================================
def llm_darija_fix(raw_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You correct ASR text written in Tunisian Darija. "
                "Keep the Tunisian dialect. "
                "Do not translate. "
                "Do not explain. "
                "Return only the corrected text."
            ),
        },
        {
            "role": "user",
            "content": f"Correct this Tunisian Darija ASR text:\n{raw_text}",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    fixed = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return fixed


# =========================================================
# CELL 6: Load Whisper ASR model
# Tries merged model first, then PEFT adapter if needed
# =========================================================
asr_model_path = "/kaggle/input/datasets/aymendhieb1/whisper-tunisian-adapter/whisper_tunisian_adapter"

processor = None
asr_model = None

try:
    print("Trying merged Whisper model...")
    processor = AutoProcessor.from_pretrained(asr_model_path)

    asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        asr_model_path,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        device_map="auto"
    )

    print("Loaded merged Whisper model successfully.")

except Exception as e:
    print("Merged model load failed.")
    print("Reason:", str(e))
    print("\nTrying PEFT adapter version...")

    peft_config = PeftConfig.from_pretrained(asr_model_path)
    base_model_name = peft_config.base_model_name_or_path

    print("Base model:", base_model_name)

    processor = WhisperProcessor.from_pretrained(base_model_name)

    base_model = WhisperForConditionalGeneration.from_pretrained(
        base_model_name,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        device_map="auto"
    )

    asr_model = PeftModel.from_pretrained(base_model, asr_model_path)
    asr_model.eval()

    print("Loaded PEFT Whisper adapter successfully.")

# =========================================================
# CELL 7: ASR transcription function
# =========================================================
def transcribe_audio(audio_array, sampling_rate=16000):
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_features = inputs["input_features"].to(asr_model.device, dtype=DTYPE)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features)

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()


# =========================================================
# CELL 8: Full offline pipeline
# =========================================================
def full_offline_pipeline(audio_array, sampling_rate=16000):
    raw = transcribe_audio(audio_array, sampling_rate=sampling_rate)
    clean = llm_darija_fix(raw)
    return {
        "raw": raw,
        "clean": clean
    }


# =========================================================
# CELL 9: Test on one sample
# =========================================================
audio_sample = ds[0]["audio"]["array"]
sr = ds[0]["audio"]["sampling_rate"]

result = full_offline_pipeline(audio_sample, sampling_rate=sr)

print("REFERENCE:")
print(ds[0]["text"])

print("\nRAW:")
print(result["raw"])

print("\nCLEAN:")
print(result["clean"])


# =========================================================
# CELL 10: Test on multiple samples
# =========================================================
for i in range(min(3, len(ds))):
    audio_sample = ds[i]["audio"]["array"]
    sr = ds[i]["audio"]["sampling_rate"]

    result = full_offline_pipeline(audio_sample, sampling_rate=sr)

    print(f"\n================ SAMPLE {i} ================")
    print("REFERENCE:", ds[i]["text"])
    print("RAW      :", result["raw"])
    print("CLEAN    :", result["clean"])

Device: cuda
Dtype : torch.float16
Loading dataset...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loaded dataset.
Columns: ['audio', 'text']
Length : 5
Sample text: آنا مواطن تونسي من مدنين من قرية إسمها أم التمر
Sample SR  : 16000
Loading Qwen tokenizer...
Loading Qwen model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Loaded merged Whisper model successfully.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

REFERENCE:
آنا مواطن تونسي من مدنين من قرية إسمها أم التمر

RAW:
أنا مواطن تونسي من مدنين من قرية اسمعوما التمر

CLEAN:
أنا مواطن تونسي من مدنين من قرية اسماعم التمر

================ SAMPLE 0 ================
REFERENCE: آنا مواطن تونسي من مدنين من قرية إسمها أم التمر
RAW      : أنا مواطن تونسي من مدنين من قرية اسمعوما التمر
CLEAN    : أنا مواطن تونسي من مدنين من قرية اسماعم التمر

================ SAMPLE 1 ================
REFERENCE: من عايلة فقيرة جدا
RAW      : من عالف أقير جدل
CLEAN    : من عاين أقري جدل

================ SAMPLE 2 ================
REFERENCE: توا الأكيد كي بش نبدا نحكي برشا بش يقولوا وكأنه يتاجر يعني بماضيه برشا برشا قالوها لي زعما أوه مناضل لست مناضل ومانيش من عايلة مناضلة
RAW      : طوال أكيد كي بشدناحكي برشة بش يقولوا وكأنه يوتاجر يعني بماضيه برشة برش قلوهل زعمه آه مناذل لسته مناذل ومنش من عايلة مناذلة
CLEAN    : طوال أكيد كي بشدناحكي برشة بش يقولوا وكأنه يشتري من الماضي برشة برش قلوهل زعمه آه مناذل لسته مناذل ومنش من عايلة مناذلة


In [2]:
# =========================================================
# FULL SINGLE CELL: DATASET + WHISPER ADAPTER + SAFE TEST
# No LLM correction, only tiny safe cleanup
# =========================================================

!pip install -q transformers datasets librosa peft sentencepiece accelerate

import os
import re
import torch
from datasets import load_dataset, Audio
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    WhisperProcessor,
    WhisperForConditionalGeneration
)
from peft import PeftModel, PeftConfig

# -------------------------
# Basic setup
# -------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
TARGET_SR = 16000

print("Device:", DEVICE)
print("Dtype :", DTYPE)

# -------------------------
# Load dataset
# -------------------------
print("\nLoading dataset...")
ds = load_dataset("Sabrinek8/tedxtn-train", split="train[:5]")
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

print("Loaded dataset.")
print("Columns:", ds.column_names)
print("Length :", len(ds))
print("Sample text:", ds[0]["text"])

# -------------------------
# Load Whisper model / adapter
# -------------------------
asr_model_path = "/kaggle/input/datasets/aymendhieb1/whisper-tunisian-adapter/whisper_tunisian_adapter"

processor = None
asr_model = None

print("\nLoading Whisper ASR...")
try:
    print("Trying merged Whisper model...")
    processor = AutoProcessor.from_pretrained(asr_model_path)

    asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        asr_model_path,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        device_map="auto"
    )
    print("Loaded merged Whisper model successfully.")

except Exception as e:
    print("Merged model load failed.")
    print("Reason:", str(e))
    print("Trying PEFT adapter version...")

    peft_config = PeftConfig.from_pretrained(asr_model_path)
    base_model_name = peft_config.base_model_name_or_path
    print("Base model:", base_model_name)

    processor = WhisperProcessor.from_pretrained(base_model_name)

    base_model = WhisperForConditionalGeneration.from_pretrained(
        base_model_name,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        device_map="auto"
    )

    asr_model = PeftModel.from_pretrained(base_model, asr_model_path)
    asr_model.eval()
    print("Loaded PEFT Whisper adapter successfully.")

# -------------------------
# Tiny safe text cleanup only
# -------------------------
def safe_cleanup(text):
    text = text.strip()

    # remove super-long repeated chars like آهههههههه
    text = re.sub(r'(.)\1{5,}', r'\1', text)

    # normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# -------------------------
# ASR transcription
# -------------------------
def transcribe_audio(audio_array, sampling_rate=16000):
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_features = inputs["input_features"].to(asr_model.device, dtype=DTYPE)

    forced_decoder_ids = None
    if hasattr(processor, "get_decoder_prompt_ids"):
        try:
            forced_decoder_ids = processor.get_decoder_prompt_ids(
                language="ar",
                task="transcribe"
            )
        except Exception:
            forced_decoder_ids = None

    with torch.no_grad():
        if forced_decoder_ids is not None:
            predicted_ids = asr_model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids
            )
        else:
            predicted_ids = asr_model.generate(input_features)

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return safe_cleanup(text)

# -------------------------
# Test samples
# -------------------------
print("\nRunning test...")
for i in range(min(5, len(ds))):
    audio_sample = ds[i]["audio"]["array"]
    sr = ds[i]["audio"]["sampling_rate"]

    raw_text = transcribe_audio(audio_sample, sampling_rate=sr)

    print(f"\n================ SAMPLE {i} ================")
    print("REFERENCE:", ds[i]["text"])
    print("PREDICTED:", raw_text)

Device: cuda
Dtype : torch.float16

Loading dataset...


README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


Loaded merged Whisper model successfully.

Running test...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


================ SAMPLE 0 ================
REFERENCE: آنا مواطن تونسي من مدنين من قرية إسمها أم التمر
PREDICTED: أنا مواطن تونسي من مدنين من قرية اسمعوما التمر

================ SAMPLE 1 ================
REFERENCE: من عايلة فقيرة جدا
PREDICTED: من عيلة فقيرة جدل

================ SAMPLE 2 ================
REFERENCE: توا الأكيد كي بش نبدا نحكي برشا بش يقولوا وكأنه يتاجر يعني بماضيه برشا برشا قالوها لي زعما أوه مناضل لست مناضل ومانيش من عايلة مناضلة
PREDICTED: طوه الأكيد كي بشدناحكي برشا بش يقوله وكأنه يتاجر يعني بماضيه برشا برش قلوهل زعمه آه

================ SAMPLE 3 ================
REFERENCE: ولكن النضال أشكال
PREDICTED: ولكن النظال أشكيل

================ SAMPLE 4 ================
REFERENCE: آنا كي حليت عينيا أول فشل الساعة آنا نمن بالفشل
PREDICTED: هناك حليت عينية أول فشل الساعة أن أنمن بالفشل


In [3]:
results = []

for i in range(min(5, len(ds))):
    audio_sample = ds[i]["audio"]["array"]
    sr = ds[i]["audio"]["sampling_rate"]

    pred = transcribe_audio(audio_sample, sampling_rate=sr)

    results.append({
        "sample_id": i,
        "reference": ds[i]["text"],
        "prediction": pred
    })

for row in results:
    print(f"\n===== SAMPLE {row['sample_id']} =====")
    print("REFERENCE :", row["reference"])
    print("PREDICTION:", row["prediction"])


===== SAMPLE 0 =====
REFERENCE : آنا مواطن تونسي من مدنين من قرية إسمها أم التمر
PREDICTION: أنا مواطن تونسي من مدنين من قرية اسمعوما التمر

===== SAMPLE 1 =====
REFERENCE : من عايلة فقيرة جدا
PREDICTION: من عيلة فقيرة جدل

===== SAMPLE 2 =====
REFERENCE : توا الأكيد كي بش نبدا نحكي برشا بش يقولوا وكأنه يتاجر يعني بماضيه برشا برشا قالوها لي زعما أوه مناضل لست مناضل ومانيش من عايلة مناضلة
PREDICTION: طوه الأكيد كي بشدناحكي برشا بش يقوله وكأنه يتاجر يعني بماضيه برشا برش قلوهل زعمه آه

===== SAMPLE 3 =====
REFERENCE : ولكن النضال أشكال
PREDICTION: ولكن النظال أشكيل

===== SAMPLE 4 =====
REFERENCE : آنا كي حليت عينيا أول فشل الساعة آنا نمن بالفشل
PREDICTION: هناك حليت عينية أول فشل الساعة أن أنمن بالفشل


In [4]:
!pip install -q jiwer
from jiwer import wer, cer

refs = [row["reference"] for row in results]
preds = [row["prediction"] for row in results]

overall_wer = wer(refs, preds)
overall_cer = cer(refs, preds)

print(f"WER: {overall_wer:.4f}")
print(f"CER: {overall_cer:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:0000:0100:01
WER: 0.5472
CER: 0.2824


In [5]:
from jiwer import wer

for row in results:
    sample_wer = wer([row["reference"]], [row["prediction"]])
    print(f"Sample {row['sample_id']} WER: {sample_wer:.4f}")

Sample 0 WER: 0.3000
Sample 1 WER: 0.5000
Sample 2 WER: 0.6538
Sample 3 WER: 0.6667
Sample 4 WER: 0.5000


In [6]:
# =========================================================
# BOOTSTRAPPING STEP 1: Generate pseudo-labels
# =========================================================

pseudo_data = []

def is_good_pseudo_label(text):
    text = text.strip()

    # basic filters
    if len(text) < 5:
        return False
    if len(text.split()) < 2:
        return False

    # reject too many repeated chars
    import re
    if re.search(r'(.)\1{4,}', text):
        return False

    # reject suspiciously long garbage tokens
    long_tokens = [w for w in text.split() if len(w) > 20]
    if len(long_tokens) > 0:
        return False

    return True

print("Generating pseudo-labels...")

for i in range(len(ds)):
    audio_sample = ds[i]["audio"]["array"]
    sr = ds[i]["audio"]["sampling_rate"]

    pred = transcribe_audio(audio_sample, sampling_rate=sr)

    if is_good_pseudo_label(pred):
        pseudo_data.append({
            "audio": ds[i]["audio"],
            "text": pred,
            "source": "pseudo"
        })

print("Pseudo-labeled samples kept:", len(pseudo_data))
if len(pseudo_data) > 0:
    print("Example pseudo-label:", pseudo_data[0]["text"])

Generating pseudo-labels...
Pseudo-labeled samples kept: 5
Example pseudo-label: أنا مواطن تونسي من مدنين من قرية اسمعوما التمر


In [8]:
# Optional: save pseudo-labeled dataset
pseudo_save_path = "/kaggle/working/pseudo_labeled_tn"
pseudo_ds.save_to_disk(pseudo_save_path)

print("Saved pseudo-labeled dataset to:", pseudo_save_path)

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saved pseudo-labeled dataset to: /kaggle/working/pseudo_labeled_tn


In [9]:
print("\n===== PSEUDO-LABELED DATA PREVIEW =====")

for i in range(min(3, len(pseudo_ds))):
    print(f"\n--- SAMPLE {i} ---")
    print("TEXT :", pseudo_ds[i]["text"])
    print("SOURCE:", pseudo_ds[i]["source"])


===== PSEUDO-LABELED DATA PREVIEW =====

--- SAMPLE 0 ---
TEXT : أنا مواطن تونسي من مدنين من قرية اسمعوما التمر
SOURCE: pseudo

--- SAMPLE 1 ---
TEXT : من عيلة فقيرة جدل
SOURCE: pseudo

--- SAMPLE 2 ---
TEXT : طوه الأكيد كي بشدناحكي برشا بش يقوله وكأنه يتاجر يعني بماضيه برشا برش قلوهل زعمه أهوه مناذل لسته مناذل ومن يشمن عائله مناذلة
SOURCE: pseudo


In [10]:
print("\n===== BOOTSTRAPPING STATS =====")
print("Original samples :", len(ds))
print("Pseudo samples   :", len(pseudo_ds))

ratio = len(pseudo_ds) / len(ds)
print(f"Kept ratio       : {ratio:.2f}")


===== BOOTSTRAPPING STATS =====
Original samples : 5
Pseudo samples   : 5
Kept ratio       : 1.00


In [4]:
!pip install -q -U transformers datasets accelerate evaluate jiwer librosa soundfile sentencepiece peft
# ── Silence all logs ──────────────────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

from datasets.utils.logging import disable_progress_bar
disable_progress_bar()
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# ── Imports ───────────────────────────────────────────────────────────────────
import re, gc, pathlib, shutil
import torch, numpy as np, evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from IPython.display import clear_output

from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import PeftModel

# ── Force single GPU ──────────────────────────────────────────────────────────
for var in ["MASTER_ADDR","MASTER_PORT","WORLD_SIZE","RANK","LOCAL_RANK"]:
    os.environ.pop(var, None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# ── Config ────────────────────────────────────────────────────────────────────
MEDICAL_ADAPTER   = "/kaggle/input/datasets/aymendhieb1/whisper-adapter-final-v1/whisper_adapter_final"
BASE_MODEL_NAME   = "openai/whisper-small"
OUTPUT_DIR        = "/kaggle/working/whisper_linto_tunisian"
DATASET_NAME      = "linagora/linto-dataset-audio-ar-tn"
TARGET_SR         = 16_000
MAX_TRAIN_SAMPLES = 10_000
MAX_EVAL_SAMPLES  = 500
SEED              = 42
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Device  : {DEVICE}")
print(f"✅ Adapter : {MEDICAL_ADAPTER}")
print(f"✅ Dataset : {DATASET_NAME}")
_, _, free = shutil.disk_usage("/kaggle/working")
print(f"✅ Disk free: {free/1e9:.1f} GB")

# ── Arabic text helpers ───────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    text = text.replace("ى","ي")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def transcript_ok(text: str) -> bool:
    text = normalize_ar(text)
    if len(text) < 5:                 return False
    if len(text.split()) < 2:         return False
    if re.search(r"(.)\1{4,}", text): return False
    return True

# ── Load dataset ──────────────────────────────────────────────────────────────
print("\n⬇️  Loading LinTO dataset...")
ds = load_dataset(DATASET_NAME, split="train")
print(f"   Raw size : {len(ds):,} | Columns: {ds.column_names}")

if "transcript" not in ds.column_names:
    raise ValueError(f"Expected 'transcript' column. Found: {ds.column_names}")

ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

def add_duration(ex):
    return {"duration": len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"]}

ds = ds.map(add_duration, num_proc=1, keep_in_memory=False)

def keep_example(ex):
    return transcript_ok(ex["transcript"]) and (0.8 <= ex["duration"] <= 15.0)

ds = ds.filter(keep_example, num_proc=1, keep_in_memory=False)
print(f"   Filtered : {len(ds):,}")

ds         = ds.shuffle(seed=SEED)
eval_size  = min(MAX_EVAL_SAMPLES, max(200, int(0.05 * len(ds))))
train_size = min(MAX_TRAIN_SAMPLES, len(ds) - eval_size)
train_ds   = ds.select(range(train_size))
eval_ds    = ds.select(range(train_size, train_size + eval_size))
print(f"   Train    : {len(train_ds):,} | Eval: {len(eval_ds):,}")

# ── Processor ─────────────────────────────────────────────────────────────────
print("\n⬇️  Loading processor...")
processor = WhisperProcessor.from_pretrained(MEDICAL_ADAPTER)
print("✅ Processor ready")

# ── Feature extraction ────────────────────────────────────────────────────────
def prepare_batch(batch):
    audio = batch["audio"]
    text  = normalize_ar(batch["transcript"])
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

print("🔄 Preprocessing...")
train_ds = train_ds.map(
    prepare_batch,
    remove_columns=train_ds.column_names,
    num_proc=1,
    keep_in_memory=False,
)
eval_ds = eval_ds.map(
    prepare_batch,
    remove_columns=eval_ds.column_names,
    num_proc=1,
    keep_in_memory=False,
)
print(f"✅ Done — columns: {train_ds.column_names}")

# ── Load base model + medical LoRA adapter ────────────────────────────────────
print(f"\n⬇️  Loading base model...")
base = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME, low_cpu_mem_usage=True
)
for attr in ["suppress_tokens", "forced_decoder_ids"]:
    if hasattr(base.config, attr):
        delattr(base.config, attr)
base.config.use_cache = False

print("📦 Loading medical LoRA adapter...")
model = PeftModel.from_pretrained(base, MEDICAL_ADAPTER, is_trainable=True)
model.print_trainable_parameters()

model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)
model.generation_config.suppress_tokens = []
model.to(DEVICE)
print(f"✅ Model on {DEVICE}")

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        bos_id = processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]
        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Sanity check
sb = data_collator([train_ds[0], train_ds[1]])
print(f"\n✅ Collator — input_features: {tuple(sb['input_features'].shape)}  labels: {tuple(sb['labels'].shape)}")
del sb; gc.collect(); torch.cuda.empty_cache()

# ── WER metric ────────────────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str  = [normalize_ar(x) for x in pred_str]
    label_str = [normalize_ar(x) for x in label_str]
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

# ── Training arguments ────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=200,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=2,
)
# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ── Train ─────────────────────────────────────────────────────────────────────
clear_output(wait=True)
print("🚀 Starting fine-tuning on LinTO Tunisian Arabic...")
print(f"   Train : {len(train_ds):,} samples")
print(f"   Eval  : {len(eval_ds):,} samples")
print(f"   Steps : 1000 — WER logged at step 500 and 1000")
print(f"   Est.  : ~1.5 hours on T4\n")
trainer.train()

# ── Save ──────────────────────────────────────────────────────────────────────
print("\n💾 Saving LoRA adapter...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Adapter saved → {OUTPUT_DIR}")

ZIP_PATH = "/kaggle/working/whisper_linto_tunisian.zip"
shutil.make_archive(
    ZIP_PATH.replace(".zip",""), "zip",
    root_dir=os.path.dirname(OUTPUT_DIR),
    base_dir=os.path.basename(OUTPUT_DIR),
)
print(f"📦 Zipped: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB → {ZIP_PATH}")
print("📥 Download from Kaggle Output panel → whisper_linto_tunisian.zip")

_, used, free = shutil.disk_usage("/kaggle/working")
print(f"💾 Final disk: used={used/1e9:.1f}GB  free={free/1e9:.1f}GB")

gc.collect()
torch.cuda.empty_cache()

🚀 Starting fine-tuning on LinTO Tunisian Arabic...
   Train : 10,000 samples
   Eval  : 500 samples
   Steps : 1000 — WER logged at step 500 and 1000
   Est.  : ~1.5 hours on T4

{'loss': '5.82', 'grad_norm': '7.101', 'learning_rate': '8.9e-06', 'epoch': '0.32'}
{'loss': '3.37', 'grad_norm': '4.468', 'learning_rate': '6.678e-06', 'epoch': '0.64'}
{'eval_loss': '1.623', 'eval_wer': '88.81', 'eval_runtime': '280.5', 'eval_samples_per_second': '1.782', 'eval_steps_per_second': '0.225', 'epoch': '0.8'}
{'loss': '3.11', 'grad_norm': '3.053', 'learning_rate': '4.456e-06', 'epoch': '0.96'}
{'loss': '2.988', 'grad_norm': '3.341', 'learning_rate': '2.233e-06', 'epoch': '1.28'}
{'loss': '2.978', 'grad_norm': '3.751', 'learning_rate': '1.111e-08', 'epoch': '1.6'}
{'eval_loss': '1.553', 'eval_wer': '86.46', 'eval_runtime': '278.4', 'eval_samples_per_second': '1.796', 'eval_steps_per_second': '0.226', 'epoch': '1.6'}
{'train_runtime': '2741', 'train_samples_per_second': '5.837', 'train_steps_per_se

In [ ]:
from huggingface_hub import login
login("hf_xxxxxxx")

In [5]:
import os
from huggingface_hub import login

login(os.getenv("HF_TOKEN"))

In [ ]:
import os

os.system("pip uninstall -y peft >/dev/null 2>&1")

os.system("""pip install -q \
  "numpy==2.0.2" \
  "transformers==4.40.2" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.23.4" \
  "datasets==2.19.2" \
  "accelerate==0.30.1" \
  "evaluate==0.4.1" \
  "jiwer==3.0.4" \
  "fsspec==2024.2.0" \
  "safetensors==0.4.3" \
  --upgrade --ignore-installed \
  2>&1 | tail -5
""")

os.kill(os.getpid(), 9)

In [1]:
# ── Env vars FIRST ────────────────────────────────────────────────────────────
import os, re, gc, time, pathlib, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from transformers.generation.configuration_utils import GenerationConfig

# ── Single GPU config ─────────────────────────────────────────────────────────
cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME   = "openai/whisper-small"
OUTPUT_DIR        = "/kaggle/working/whisper_small_linto_fast"
DATASET_NAME      = "linagora/linto-dataset-audio-ar-tn"
TARGET_SR         = 16_000
MAX_TRAIN_SAMPLES = 1800
MAX_EVAL_SAMPLES  = 200
SEED              = 42
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")

# ── Text helpers ──────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا").replace("ى","ي")
    return re.sub(r"\s+", " ", text).strip()

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return len(t) >= 5 and len(t.split()) >= 2 and not re.search(r"(.)\1{4,}", t)

# ── Stream + collect samples (avoids full download) ───────────────────────────
print("\n⬇️  Streaming dataset (no full download)...")

# Retry wrapper for flaky HF connections
def load_with_retry(name, retries=3, wait=15):
    for attempt in range(retries):
        try:
            return load_dataset(name, split="train", streaming=True)
        except Exception as e:
            if attempt < retries - 1:
                print(f"   ⚠️  Attempt {attempt+1} failed ({e}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    
stream_ds = load_with_retry(DATASET_NAME)

# Check columns on first example
first = next(iter(stream_ds))
print(f"   Columns: {list(first.keys())}")
if "transcript" not in first:
    raise ValueError(f"Expected 'transcript' column. Got: {list(first.keys())}")

# Collect only what we need — text filter in streaming, no audio decoded yet
collected = []
needed = MAX_TRAIN_SAMPLES + MAX_EVAL_SAMPLES + 300  # buffer for duration filter

print(f"   Collecting ~{needed} text-filtered samples (streaming)...")
for ex in stream_ds:
    if transcript_ok(ex.get("transcript", "")):
        collected.append(ex)
    if len(collected) >= needed:
        break

print(f"   Collected: {len(collected):,} samples")

# Shuffle deterministically
import random
random.seed(SEED)
random.shuffle(collected)

# Convert to HF Dataset and resample audio
ds = Dataset.from_list(collected)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Duration filter
def add_duration(ex):
    return {"duration": len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"]}

print("📏 Computing durations...")
ds = ds.map(add_duration, num_proc=1)
ds = ds.filter(lambda ex: 0.8 <= ex["duration"] <= 15.0)
print(f"   After duration filter: {len(ds):,}")

# Split
eval_size  = min(MAX_EVAL_SAMPLES, max(100, int(0.1 * len(ds))))
train_size = min(MAX_TRAIN_SAMPLES, len(ds) - eval_size)
train_ds   = ds.select(range(train_size))
eval_ds    = ds.select(range(train_size, train_size + eval_size))
print(f"   Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

# ── Processor ─────────────────────────────────────────────────────────────────
print("\n⬇️  Loading processor...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)
print("✅ Processor ready")

# ── Feature extraction ────────────────────────────────────────────────────────
def prepare_batch(batch):
    batch["input_features"] = processor.feature_extractor(
        batch["audio"]["array"],
        sampling_rate=batch["audio"]["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(
        normalize_ar(batch["transcript"])
    ).input_ids
    return batch

print("🔄 Preprocessing train...")
train_ds = train_ds.map(
    prepare_batch,
    remove_columns=train_ds.column_names,
    num_proc=1,
    keep_in_memory=False,
)
print("🔄 Preprocessing eval...")
eval_ds = eval_ds.map(
    prepare_batch,
    remove_columns=eval_ds.column_names,
    num_proc=1,
    keep_in_memory=False,
)
print("✅ Preprocessing done")

# ── Model ─────────────────────────────────────────────────────────────────────
print("\n⬇️  Loading model...")
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True
)
model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)
model.generation_config.suppress_tokens = []
model.config.use_cache = False
model.to(DEVICE)
print(f"✅ Model on {DEVICE}")

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ── Metric ────────────────────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

# ── Training ──────────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,       # bigger batch = faster on T4
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=3,                   # 3 epochs on 1800 samples ≈ 50 min
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🚀 Training Whisper-small on LinTO Tunisian Arabic")
print(f"   Train: {len(train_ds):,} | Eval: {len(eval_ds):,} | Epochs: 3\n")

train_result = trainer.train()
print(f"\n✅ Done — steps: {train_result.global_step}")

metrics = trainer.evaluate()
print(f"📊 Final WER: {metrics.get('eval_wer', float('nan')):.2f}%")

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"💾 Saved → {OUTPUT_DIR}")

gc.collect()
torch.cuda.empty_cache()

2026-04-08 18:51:40.607421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775674300.836681     187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775674300.898803     187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775674301.424277     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775674301.424311     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775674301.424313     187 computation_placer.cc:177] computation placer alr

✅ Device: cuda

⬇️  Streaming dataset (no full download)...
   Columns: ['audio_id', 'audio', 'segments', 'transcript']
   Collected: 2,300 samples
📏 Computing durations...
   After duration filter: 2,292
   Train: 1,800 | Eval: 200

⬇️  Loading processor...


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Model on cuda



🚀 Training Whisper-small on LinTO Tunisian Arabic
   Train: 1,800 | Eval: 200 | Epochs: 3

{'loss': 4.9132, 'grad_norm': 27.756179809570312, 'learning_rate': 4.2000000000000004e-06, 'epoch': 0.22123893805309736}
{'loss': 2.3887, 'grad_norm': 18.754125595092773, 'learning_rate': 9.200000000000002e-06, 'epoch': 0.4424778761061947}
{'loss': 1.6934, 'grad_norm': 18.339353561401367, 'learning_rate': 9.273356401384083e-06, 'epoch': 0.6637168141592921}
{'loss': 1.5891, 'grad_norm': 17.503047943115234, 'learning_rate': 8.408304498269897e-06, 'epoch': 0.8849557522123894}
{'eval_loss': 1.4980111122131348, 'eval_wer': 80.37333333333333, 'eval_runtime': 111.3405, 'eval_samples_per_second': 1.796, 'eval_steps_per_second': 0.117, 'epoch': 1.0}
{'loss': 1.3103, 'grad_norm': 14.443004608154297, 'learning_rate': 7.5432525951557104e-06, 'epoch': 1.1061946902654867}
{'loss': 1.1113, 'grad_norm': 12.39205551147461, 'learning_rate': 6.6782006920415235e-06, 'epoch': 1.3274336283185841}
{'loss': 1.0476, 'gr

In [ ]:
import os
os.kill(os.getpid(), 9)  # restart kernel cleanly first

In [ ]:
# Install PyTorch built for CUDA 11.8 — supports P100 (CC 6.0)
import os
os.system("""
pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2 \
  --index-url https://download.pytorch.org/whl/cu118 \
  --quiet
""")
os.kill(os.getpid(), 9)  # restart kernel after install

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 79.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 104.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


In [ ]:
# PyTorch+cu118 — includes Pascal (P100, sm_60). Default Kaggle torch is often cu12.8 without P100 kernels,
# which causes: "no kernel image is available for execution on the device".
# Run this ONCE, then Runtime → Restart session, then run training cells (do not pip-upgrade torch after).
import os
print("Installing torch 2.2.2+cu118 (P100-safe)...")
os.system(
    "pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2 "
    "--index-url https://download.pytorch.org/whl/cu118 --quiet"
)
print("Done. Kernel will restart — re-run from the top after restart.")
os.kill(os.getpid(), 9)

Installing torch 2.2.2+cu118 (P100-safe)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 100.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 71.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 97.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 8.8 MB/s

In [1]:
!pip install -q --no-deps peft==0.13.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 6.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
import os
os.system("pip install -q numpy==1.26.4 --force-reinstall")
os.kill(os.getpid(), 9)  # restart kernel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.1 MB/s eta 0:00:00


In [2]:
import os
os.system("pip install -q evaluate jiwer")
os.system("pip install -q --no-deps peft==0.13.2")
print("✅ deps ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 20.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 5.1 MB/s eta 0:00:00
✅ deps ready



## Whisper + LoRA — Tunisian Arabic 
 setup & data → model & collator → metrics (defines `compute_metrics`) → train & save.


In [3]:
import os, re, gc, time, pathlib, warnings, random, json, shutil
os.environ["CUDA_VISIBLE_DEVICES"]              = "0"
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]           = "180"
for v in ["MASTER_ADDR","MASTER_PORT","WORLD_SIZE","RANK","LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from transformers.generation.configuration_utils import GenerationConfig
from peft import PeftModel

# ── Single GPU accelerate config ──────────────────────────────────────────────
cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME   = "openai/whisper-small"
ADAPTER_PATH      = "/kaggle/input/datasets/aymendhieb/whisper-adapter-final-v1/whisper_adapter_final"
OUTPUT_DIR        = "/kaggle/working/whisper_tunisian_final"
DATASET_NAME      = "linagora/linto-dataset-audio-ar-tn"
TARGET_SR         = 16_000
MAX_TRAIN_SAMPLES = 1500   # keep small so preprocessing finishes in ~15 min
MAX_EVAL_SAMPLES  = 150
SEED              = 42
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"✅ Device       : {DEVICE} ({_p.name}, sm_{_p.major}{_p.minor})")
    print(f"✅ torch        : {torch.__version__} (CUDA {torch.version.cuda})")
    try:
        _t = torch.randn(256, 256, device="cuda", dtype=torch.float32)
        float((_t @ _t).sum().item())
        torch.cuda.synchronize()
    except Exception as _e:
        raise RuntimeError(
            "CUDA failed a simple matmul: PyTorch build likely has no kernels for this GPU "
            "(cudaErrorNoKernelImageForDevice). Common on Kaggle P100 + default torch 2.10+cu128.\n\n"
            "Fix A — stay on P100: run the notebook cell that installs torch+cu118 (2.2.2), then "
            "Runtime → Restart session, then run training again.\n"
            "Fix B — switch Kaggle Accelerator to GPU T4 (works with default cu12 torch).\n"
        ) from _e
else:
    print(f"✅ Device       : {DEVICE}")

print(f"✅ GPU count    : {torch.cuda.device_count()}")
print(f"✅ Base model   : {BASE_MODEL_NAME}")
print(f"✅ Adapter      : {ADAPTER_PATH}")
print(f"✅ Train samples: {MAX_TRAIN_SAMPLES}")


def whisper_safe_adapter_dir(adapter_src: str, work_parent: str) -> str:
    """Whisper needs `input_features`, not `input_ids`. If the adapter was saved with
    `task_type` (e.g. SEQ_2_SEQ_LM), PEFT wraps it in a class that forwards `input_ids`
    and training crashes. Copy the adapter, drop `task_type`, load from the copy."""
    src, parent = pathlib.Path(adapter_src), pathlib.Path(work_parent)
    parent.mkdir(parents=True, exist_ok=True)
    out = parent / "_peft_adapter_whisper_safe"
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    cfg_path = out / "adapter_config.json"
    if cfg_path.is_file():
        cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        removed = cfg.pop("task_type", None)
        if removed is not None:
            cfg_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
            print(f"   ⚙️  adapter_config: removed task_type={removed!r} (Whisper + PEFT fix)")
    return str(out)


ADAPTER_LOAD_PATH = whisper_safe_adapter_dir(
    ADAPTER_PATH, str(pathlib.Path(OUTPUT_DIR).parent)
)
print(f"✅ Adapter load : {ADAPTER_LOAD_PATH}")

# ── Text helpers ──────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (text
        .replace("أ","ا").replace("إ","ا").replace("آ","ا")
        .replace("ى","ي").replace("ة","ه")
        .replace("ـ","")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )

# ── Dataset + Processor + True streaming preprocessing ───────────────────────
print("\n⬇️  Loading processor first...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)
print("✅ Processor ready")

print("\n⬇️  Streaming dataset...")

def load_with_retry(name, retries=4, wait=20):
    for attempt in range(retries):
        try:
            ds = load_dataset(name, split="train", streaming=True)
            return ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
        except Exception:
            if attempt < retries - 1:
                print(f"   ⚠️  Attempt {attempt+1} failed — retrying in {wait}s")
                time.sleep(wait)
            else:
                raise

stream_ds = load_with_retry(DATASET_NAME)
first = next(iter(stream_ds))
print(f"   Columns: {list(first.keys())}")

# Single-sample preprocessing directly from stream (no Dataset.from_list / no .map)
def process_sample(ex):
    try:
        arr = np.array(ex["audio"]["array"], dtype=np.float32)
        sr  = int(ex["audio"]["sampling_rate"])

        if arr.ndim != 1 or arr.size == 0:
            return None
        if arr.shape[0] < sr * 0.1:       # < 0.5s
            return None
        if np.max(np.abs(arr)) < 1e-6:    # silent
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:                    # too long
            return None

        text = normalize_ar(ex.get("transcript", ""))
        if not transcript_ok(text):
            return None

        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]
        ids = processor.tokenizer(
            text,
            max_length=448,
            truncation=True,
        ).input_ids
        if len(ids) < 2:
            return None

        return {"input_features": feats, "labels": ids}
    except Exception:
        return None

# Build eval then train in one streaming pass with watchdog-style progress
needed_eval = MAX_EVAL_SAMPLES
needed_train = MAX_TRAIN_SAMPLES
max_seen = (needed_eval + needed_train) * 12  # hard stop to avoid infinite waiting

eval_features, eval_labels = [], []
train_features, train_labels = [], []

seen = 0
accepted = 0
skipped = 0
t0 = time.time()

print(f"   Target good samples: eval={needed_eval}, train={needed_train}")
for ex in stream_ds:
    seen += 1
    item = process_sample(ex)

    if item is None:
        skipped += 1
    else:
        accepted += 1
        if len(eval_features) < needed_eval:
            eval_features.append(item["input_features"])
            eval_labels.append(item["labels"])
        elif len(train_features) < needed_train:
            train_features.append(item["input_features"])
            train_labels.append(item["labels"])

    if seen % 100 == 0:
        elapsed = time.time() - t0
        rate_good = accepted / max(elapsed, 1e-6)
        rem_good = (needed_eval - len(eval_features)) + (needed_train - len(train_features))
        eta = rem_good / max(rate_good, 1e-6)
        print(
            f"   seen={seen:,} | good={accepted:,} | skipped={skipped:,} "
            f"| eval={len(eval_features):,}/{needed_eval:,} "
            f"| train={len(train_features):,}/{needed_train:,} "
            f"| ETA~{eta/60:.1f} min"
        )

    if len(eval_features) >= needed_eval and len(train_features) >= needed_train:
        break
    if seen >= max_seen:
        print("   ⚠️ Reached max_seen guard; proceeding with what was collected.")
        break

elapsed = time.time() - t0

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels,
})
train_ds = Dataset.from_dict({
    "input_features": train_features,
    "labels": train_labels,
})

print(
    f"\n✅ Preprocessing done in {elapsed/60:.1f} min — "
    f"Train: {len(train_ds):,} | Eval: {len(eval_ds):,}"
)

if len(train_ds) < max(200, int(0.3 * MAX_TRAIN_SAMPLES)):
    raise RuntimeError(
        "Too few train samples were collected. Increase max_seen or relax filters."
    )



✅ Device       : cuda (Tesla P100-PCIE-16GB, sm_60)
✅ torch        : 2.2.2+cu118 (CUDA 11.8)
✅ GPU count    : 1
✅ Base model   : openai/whisper-small
✅ Adapter      : /kaggle/input/datasets/aymendhieb/whisper-adapter-final-v1/whisper_adapter_final
✅ Train samples: 1500
   ⚙️  adapter_config: removed task_type='SEQ_2_SEQ_LM' (Whisper + PEFT fix)
✅ Adapter load : /kaggle/working/_peft_adapter_whisper_safe

⬇️  Loading processor first...
✅ Processor ready

⬇️  Streaming dataset...
   Columns: ['audio_id', 'audio', 'segments', 'transcript']
   Target good samples: eval=150, train=1500
   seen=100 | good=100 | skipped=0 | eval=100/150 | train=0/1,500 | ETA~0.4 min
   seen=200 | good=200 | skipped=0 | eval=150/150 | train=50/1,500 | ETA~0.4 min
   seen=300 | good=300 | skipped=0 | eval=150/150 | train=150/1,500 | ETA~0.4 min
   seen=400 | good=400 | skipped=0 | eval=150/150 | train=250/1,500 | ETA~0.4 min
   seen=500 | good=498 | skipped=2 | eval=150/150 | train=348/1,500 | ETA~0.3 min
   se

In [4]:
# ── Load model ────────────────────────────────────────────────────────────────
print("\n⬇️  Loading base Whisper...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

print("📦 Loading LoRA adapter (trainable — no merge)...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_LOAD_PATH,
    is_trainable=True,
)

for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad_(True)
for name, param in model.named_parameters():
    if "model.encoder" in name and "lora_" not in name:
        param.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)
model.generation_config.suppress_tokens = []

# Batched eval: Whisper must not auto-detect mixed languages per sample (ValueError).
# Force Arabic transcription for all eval/generate calls.
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

model.to(DEVICE)
print(f"✅ Model on {DEVICE}")

# PEFT + gradient checkpointing: without this, backward often sees no graph
# ("element 0 of tensors does not require grad and does not have a grad_fn").
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels":         labels,
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)




⬇️  Loading base Whisper...
📦 Loading LoRA adapter (trainable — no merge)...
   Trainable: 1,769,472 / 243,504,384 (0.73%)
✅ Model on cuda


In [5]:
# ── Metrics ───────────────────────────────────────────────────────────────────
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }



In [6]:
# ── Training args ─────────────────────────────────────────────────────────────
# Trainer that always passes Whisper generate kwargs (fixes multi-language batch error on eval)
class WhisperSeq2SeqTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer that forces language/task for Whisper batched generation during evaluate()."""

    def __init__(self, *args, whisper_generate_kwargs=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.whisper_generate_kwargs = whisper_generate_kwargs or {}

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**self.whisper_generate_kwargs, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

    def predict(self, test_dataset, ignore_keys=None, metric_key_prefix="test", **gen_kwargs):
        gen_kwargs = {**self.whisper_generate_kwargs, **gen_kwargs}
        return super().predict(test_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)


training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperSeq2SeqTrainer(
    args=training_args,
    model=model,
    whisper_generate_kwargs={"language": "arabic", "task": "transcribe"},
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Force single GPU
trainer.args._n_gpu = 1

total_steps = (len(train_ds) // (16 * 2)) * 4
print(f"\n🚀 Training Whisper (medical → Tunisian Arabic)")
print(f"   Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")
print(f"   Epochs: 4 | Effective batch: 32 | Steps: ~{total_steps:,}")
print(f"   Est. time on P100: ~{total_steps * 2.2 / 3600:.1f}h\n")

train_result = trainer.train()
print(f"\n✅ Done — steps: {train_result.global_step}")

metrics = trainer.evaluate()
print(f"📊 Final WER : {metrics.get('eval_wer', float('nan')):.2f}%")
print(f"📊 Final CER : {metrics.get('eval_cer', float('nan')):.2f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n💾 Adapter saved → {OUTPUT_DIR}")

gc.collect()
torch.cuda.empty_cache()


🚀 Training Whisper (medical → Tunisian Arabic)
   Train: 1,500 | Eval: 150
   Epochs: 4 | Effective batch: 32 | Steps: ~184
   Est. time on P100: ~0.1h

{'loss': 5.1128, 'grad_norm': 25.7852725982666, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.2127659574468085}
{'loss': 5.0235, 'grad_norm': 25.004976272583008, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.425531914893617}
{'loss': 4.7652, 'grad_norm': 21.653789520263672, 'learning_rate': 3e-06, 'epoch': 0.6382978723404256}
{'loss': 4.3956, 'grad_norm': 20.87601661682129, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.851063829787234}
{'eval_loss': 3.3559300899505615, 'eval_wer': 83.55731225296442, 'eval_cer': 68.51677641894011, 'eval_runtime': 82.8095, 'eval_samples_per_second': 1.811, 'eval_steps_per_second': 0.229, 'epoch': 1.0}
{'loss': 4.0925, 'grad_norm': 16.837753295898438, 'learning_rate': 5e-06, 'epoch': 1.0638297872340425}
{'loss': 3.5997, 'grad_norm': 12.118194580078125, 'learning_rate': 4.935497706683698e


## Whisper + LoRA — Tunisian Arabic - fine tune 2nd time
 setup & data → model & collator → metrics (defines `compute_metrics`) → train & save.


In [7]:
# Tunisian LoRA adapter v1 (PEFT on openai/whisper-small) — set path once per session
import pathlib

ADAPTER_PATH = "/kaggle/input/datasets/aymendhieb/whisper-tunisian-adapter/wshiper_tunisian_adapter_v1"

_adapter_dir = pathlib.Path(ADAPTER_PATH)
if not _adapter_dir.is_dir():
    raise FileNotFoundError(
        f"Adapter folder not found: {ADAPTER_PATH}\n"
        "Add the Kaggle dataset `aymendhieb/whisper-tunisian-adapter` to this notebook."
    )
_need = ("adapter_config.json", "adapter_model.safetensors")
_missing = [n for n in _need if not (_adapter_dir / n).is_file()]
if _missing:
    raise FileNotFoundError(f"Missing in adapter folder: {_missing} (path={ADAPTER_PATH})")

print("✅ ADAPTER_PATH =", ADAPTER_PATH)

✅ ADAPTER_PATH = /kaggle/input/datasets/aymendhieb/whisper-tunisian-adapter/wshiper_tunisian_adapter_v1


In [8]:
import os, re, gc, time, pathlib, warnings, random, json, shutil
os.environ["CUDA_VISIBLE_DEVICES"]              = "0"
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]           = "180"
for v in ["MASTER_ADDR","MASTER_PORT","WORLD_SIZE","RANK","LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from transformers.generation.configuration_utils import GenerationConfig
from peft import PeftModel

# ── Single GPU accelerate config ──────────────────────────────────────────────
cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME   = "openai/whisper-small"
# ADAPTER_PATH: run the cell above, or set default (Tunisian v1 on Kaggle)
_DEFAULT_TN_ADAPTER = "/kaggle/input/datasets/aymendhieb/whisper-tunisian-adapter/wshiper_tunisian_adapter_v1"
if "ADAPTER_PATH" not in globals():
    ADAPTER_PATH = _DEFAULT_TN_ADAPTER
OUTPUT_DIR        = "/kaggle/working/whisper_tunisian_final"
DATASET_NAME      = "linagora/linto-dataset-audio-ar-tn"
TARGET_SR         = 16_000
MAX_TRAIN_SAMPLES = 1500   # keep small so preprocessing finishes in ~15 min
MAX_EVAL_SAMPLES  = 150
SEED              = 42
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"✅ Device       : {DEVICE} ({_p.name}, sm_{_p.major}{_p.minor})")
    print(f"✅ torch        : {torch.__version__} (CUDA {torch.version.cuda})")
    try:
        _t = torch.randn(256, 256, device="cuda", dtype=torch.float32)
        float((_t @ _t).sum().item())
        torch.cuda.synchronize()
    except Exception as _e:
        raise RuntimeError(
            "CUDA failed a simple matmul: PyTorch build likely has no kernels for this GPU "
            "(cudaErrorNoKernelImageForDevice). Common on Kaggle P100 + default torch 2.10+cu128.\n\n"
            "Fix A — stay on P100: run the notebook cell that installs torch+cu118 (2.2.2), then "
            "Runtime → Restart session, then run training again.\n"
            "Fix B — switch Kaggle Accelerator to GPU T4 (works with default cu12 torch).\n"
        ) from _e
else:
    print(f"✅ Device       : {DEVICE}")

print(f"✅ GPU count    : {torch.cuda.device_count()}")
print(f"✅ Base model   : {BASE_MODEL_NAME}")
print(f"✅ Adapter      : {ADAPTER_PATH}")
print(f"✅ Train samples: {MAX_TRAIN_SAMPLES}")


def whisper_safe_adapter_dir(adapter_src: str, work_parent: str) -> str:
    """Whisper needs `input_features`, not `input_ids`. If the adapter was saved with
    `task_type` (e.g. SEQ_2_SEQ_LM), PEFT wraps it in a class that forwards `input_ids`
    and training crashes. Copy the adapter, drop `task_type`, load from the copy."""
    src, parent = pathlib.Path(adapter_src), pathlib.Path(work_parent)
    parent.mkdir(parents=True, exist_ok=True)
    out = parent / "_peft_adapter_whisper_safe"
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    cfg_path = out / "adapter_config.json"
    if cfg_path.is_file():
        cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        removed = cfg.pop("task_type", None)
        if removed is not None:
            cfg_path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
            print(f"   ⚙️  adapter_config: removed task_type={removed!r} (Whisper + PEFT fix)")
    return str(out)


ADAPTER_LOAD_PATH = whisper_safe_adapter_dir(
    ADAPTER_PATH, str(pathlib.Path(OUTPUT_DIR).parent)
)
print(f"✅ Adapter load : {ADAPTER_LOAD_PATH}")

# ── Text helpers ──────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (text
        .replace("أ","ا").replace("إ","ا").replace("آ","ا")
        .replace("ى","ي").replace("ة","ه")
        .replace("ـ","")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )

# ── Dataset + Processor + True streaming preprocessing ───────────────────────
print("\n⬇️  Loading processor first...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)
print("✅ Processor ready")

print("\n⬇️  Streaming dataset...")

def load_with_retry(name, retries=4, wait=20):
    for attempt in range(retries):
        try:
            ds = load_dataset(name, split="train", streaming=True)
            return ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
        except Exception:
            if attempt < retries - 1:
                print(f"   ⚠️  Attempt {attempt+1} failed — retrying in {wait}s")
                time.sleep(wait)
            else:
                raise

stream_ds = load_with_retry(DATASET_NAME)
first = next(iter(stream_ds))
print(f"   Columns: {list(first.keys())}")

# Single-sample preprocessing directly from stream (no Dataset.from_list / no .map)
def process_sample(ex):
    try:
        arr = np.array(ex["audio"]["array"], dtype=np.float32)
        sr  = int(ex["audio"]["sampling_rate"])

        if arr.ndim != 1 or arr.size == 0:
            return None
        if arr.shape[0] < sr * 0.1:       # < 0.5s
            return None
        if np.max(np.abs(arr)) < 1e-6:    # silent
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:                    # too long
            return None

        text = normalize_ar(ex.get("transcript", ""))
        if not transcript_ok(text):
            return None

        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]
        ids = processor.tokenizer(
            text,
            max_length=448,
            truncation=True,
        ).input_ids
        if len(ids) < 2:
            return None

        return {"input_features": feats, "labels": ids}
    except Exception:
        return None

# Build eval then train in one streaming pass with watchdog-style progress
needed_eval = MAX_EVAL_SAMPLES
needed_train = MAX_TRAIN_SAMPLES
max_seen = (needed_eval + needed_train) * 12  # hard stop to avoid infinite waiting

eval_features, eval_labels = [], []
train_features, train_labels = [], []

seen = 0
accepted = 0
skipped = 0
t0 = time.time()

print(f"   Target good samples: eval={needed_eval}, train={needed_train}")
for ex in stream_ds:
    seen += 1
    item = process_sample(ex)

    if item is None:
        skipped += 1
    else:
        accepted += 1
        if len(eval_features) < needed_eval:
            eval_features.append(item["input_features"])
            eval_labels.append(item["labels"])
        elif len(train_features) < needed_train:
            train_features.append(item["input_features"])
            train_labels.append(item["labels"])

    if seen % 100 == 0:
        elapsed = time.time() - t0
        rate_good = accepted / max(elapsed, 1e-6)
        rem_good = (needed_eval - len(eval_features)) + (needed_train - len(train_features))
        eta = rem_good / max(rate_good, 1e-6)
        print(
            f"   seen={seen:,} | good={accepted:,} | skipped={skipped:,} "
            f"| eval={len(eval_features):,}/{needed_eval:,} "
            f"| train={len(train_features):,}/{needed_train:,} "
            f"| ETA~{eta/60:.1f} min"
        )

    if len(eval_features) >= needed_eval and len(train_features) >= needed_train:
        break
    if seen >= max_seen:
        print("   ⚠️ Reached max_seen guard; proceeding with what was collected.")
        break

elapsed = time.time() - t0

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels,
})
train_ds = Dataset.from_dict({
    "input_features": train_features,
    "labels": train_labels,
})

print(
    f"\n✅ Preprocessing done in {elapsed/60:.1f} min — "
    f"Train: {len(train_ds):,} | Eval: {len(eval_ds):,}"
)

if len(train_ds) < max(200, int(0.3 * MAX_TRAIN_SAMPLES)):
    raise RuntimeError(
        "Too few train samples were collected. Increase max_seen or relax filters."
    )



✅ Device       : cuda (Tesla P100-PCIE-16GB, sm_60)
✅ torch        : 2.2.2+cu118 (CUDA 11.8)
✅ GPU count    : 1
✅ Base model   : openai/whisper-small
✅ Adapter      : /kaggle/input/datasets/aymendhieb/whisper-tunisian-adapter/wshiper_tunisian_adapter_v1
✅ Train samples: 1500
✅ Adapter load : /kaggle/working/_peft_adapter_whisper_safe

⬇️  Loading processor first...
✅ Processor ready

⬇️  Streaming dataset...
   Columns: ['audio_id', 'audio', 'segments', 'transcript']
   Target good samples: eval=150, train=1500
   seen=100 | good=100 | skipped=0 | eval=100/150 | train=0/1,500 | ETA~0.4 min
   seen=200 | good=200 | skipped=0 | eval=150/150 | train=50/1,500 | ETA~0.5 min
   seen=300 | good=300 | skipped=0 | eval=150/150 | train=150/1,500 | ETA~0.4 min
   seen=400 | good=400 | skipped=0 | eval=150/150 | train=250/1,500 | ETA~0.6 min
   seen=500 | good=498 | skipped=2 | eval=150/150 | train=348/1,500 | ETA~0.5 min
   seen=600 | good=598 | skipped=2 | eval=150/150 | train=448/1,500 | ETA~0.

In [9]:
# ── Load model ────────────────────────────────────────────────────────────────
print("\n⬇️  Loading base Whisper...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

print("📦 Loading LoRA adapter (trainable — no merge)...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_LOAD_PATH,
    is_trainable=True,
)

for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad_(True)
for name, param in model.named_parameters():
    if "model.encoder" in name and "lora_" not in name:
        param.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)
model.generation_config.suppress_tokens = []

# Batched eval: Whisper must not auto-detect mixed languages per sample (ValueError).
# Force Arabic transcription for all eval/generate calls.
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

model.to(DEVICE)
print(f"✅ Model on {DEVICE}")

# PEFT + gradient checkpointing: without this, backward often sees no graph
# ("element 0 of tensors does not require grad and does not have a grad_fn").
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels":         labels,
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)




⬇️  Loading base Whisper...
📦 Loading LoRA adapter (trainable — no merge)...
   Trainable: 1,769,472 / 243,504,384 (0.73%)
✅ Model on cuda


In [10]:
# ── Load model ────────────────────────────────────────────────────────────────
print("\n⬇️  Loading base Whisper...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

print("📦 Loading LoRA adapter (trainable — no merge)...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_LOAD_PATH,
    is_trainable=True,
)

for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad_(True)
for name, param in model.named_parameters():
    if "model.encoder" in name and "lora_" not in name:
        param.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)
model.generation_config.suppress_tokens = []

# Batched eval: Whisper must not auto-detect mixed languages per sample (ValueError).
# Force Arabic transcription for all eval/generate calls.
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

model.to(DEVICE)
print(f"✅ Model on {DEVICE}")

# PEFT + gradient checkpointing: without this, backward often sees no graph
# ("element 0 of tensors does not require grad and does not have a grad_fn").
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels":         labels,
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)




⬇️  Loading base Whisper...
📦 Loading LoRA adapter (trainable — no merge)...
   Trainable: 1,769,472 / 243,504,384 (0.73%)
✅ Model on cuda


In [11]:
# ── Training args ─────────────────────────────────────────────────────────────
# Trainer that always passes Whisper generate kwargs (fixes multi-language batch error on eval)
class WhisperSeq2SeqTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer that forces language/task for Whisper batched generation during evaluate()."""

    def __init__(self, *args, whisper_generate_kwargs=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.whisper_generate_kwargs = whisper_generate_kwargs or {}

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**self.whisper_generate_kwargs, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

    def predict(self, test_dataset, ignore_keys=None, metric_key_prefix="test", **gen_kwargs):
        gen_kwargs = {**self.whisper_generate_kwargs, **gen_kwargs}
        return super().predict(test_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)


training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperSeq2SeqTrainer(
    args=training_args,
    model=model,
    whisper_generate_kwargs={"language": "arabic", "task": "transcribe"},
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Force single GPU
trainer.args._n_gpu = 1

total_steps = (len(train_ds) // (16 * 2)) * 4
print(f"\n🚀 Training Whisper (medical → Tunisian Arabic)")
print(f"   Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")
print(f"   Epochs: 4 | Effective batch: 32 | Steps: ~{total_steps:,}")
print(f"   Est. time on P100: ~{total_steps * 2.2 / 3600:.1f}h\n")

train_result = trainer.train()
print(f"\n✅ Done — steps: {train_result.global_step}")

metrics = trainer.evaluate()
print(f"📊 Final WER : {metrics.get('eval_wer', float('nan')):.2f}%")
print(f"📊 Final CER : {metrics.get('eval_cer', float('nan')):.2f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n💾 Adapter saved → {OUTPUT_DIR}")

gc.collect()
torch.cuda.empty_cache()


🚀 Training Whisper (medical → Tunisian Arabic)
   Train: 1,500 | Eval: 150
   Epochs: 4 | Effective batch: 32 | Steps: ~184
   Est. time on P100: ~0.1h

{'loss': 2.6234, 'grad_norm': 4.214249610900879, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.2127659574468085}
{'loss': 2.6168, 'grad_norm': 4.972164154052734, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.425531914893617}
{'loss': 2.5473, 'grad_norm': 3.777862310409546, 'learning_rate': 3e-06, 'epoch': 0.6382978723404256}
{'loss': 2.4212, 'grad_norm': 4.428297996520996, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.851063829787234}
{'eval_loss': 1.9199565649032593, 'eval_wer': 81.26482213438734, 'eval_cer': 55.97365945437441, 'eval_runtime': 77.2111, 'eval_samples_per_second': 1.943, 'eval_steps_per_second': 0.246, 'epoch': 1.0}
{'loss': 2.4213, 'grad_norm': 3.8414852619171143, 'learning_rate': 5e-06, 'epoch': 1.0638297872340425}
{'loss': 2.2489, 'grad_norm': 3.0119102001190186, 'learning_rate': 4.935497706683698e-

---
## LinTO — full fine-tune Whisper-small (~100k samples, streaming)

**Run order:** this cell only (needs GPU + HF dataset access).

- Reserves **eval** examples first (by `audio_id`), then streams **train** up to **100k** (no full RAM load of 100k audios).
- **20h budget:** defaults target **~12.5k optimizer steps** (2× passes over 100k with batch 8×2). Lower `NUM_EPOCHS` or `MAX_TRAIN_SAMPLES` if needed.
- **Disk:** HF cache + `OUTPUT_DIR` checkpoints; Kaggle `/kaggle/working` may need space.


In [ ]:
# LinTO Tunisian Arabic — FULL fine-tune (no LoRA), streaming up to 100k train samples
import os, re, gc, time, math, pathlib, warnings, random

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset
from datasets.utils.logging import disable_progress_bar

disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
from transformers.generation.configuration_utils import GenerationConfig

# ── Config ───────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
OUTPUT_DIR = "/kaggle/working/whisper_small_linto_100k_ft"
TARGET_SR = 16_000

MAX_TRAIN_SAMPLES = 100_000   # cap (streaming)
MAX_EVAL_SAMPLES = 1_500      # held-out eval (memory friendly)
STREAM_SHUFFLE_BUFFER = 50_000

# Training / P100-friendly
PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM = 2
NUM_EPOCHS = 2                # logical passes over the 100k stream; use max_steps below
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.05

# Rough P100 timing for planning (adjust after first 50 steps if you log wall time)
SEC_PER_STEP_GUESS = 2.7

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"   GPU: {p.name} (sm_{p.major}{p.minor}) | torch {torch.__version__}")

# ── Text ─────────────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )


import hashlib  # ← move to top of file with other imports

def ex_id(ex) -> str:
    if ex.get("audio_id") is not None:
        return str(ex["audio_id"])
    key = normalize_ar(ex.get("transcript", ""))[:120] + "|" + ex.get("audio", {}).get("path", "")
    return hashlib.md5(key.encode()).hexdigest()


# ── Stream loader ────────────────────────────────────────────────────────────
def load_stream(name: str, seed: int):
    ds = load_dataset(name, split="train", streaming=True)
    try:
        ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    except Exception:
        pass
    try:
        ds = ds.shuffle(seed=seed, buffer_size=STREAM_SHUFFLE_BUFFER)
    except Exception:
        pass
    return ds


def process_sample(ex, processor: WhisperProcessor):
    try:
        arr = np.array(ex["audio"]["array"], dtype=np.float32)
        sr = int(ex["audio"]["sampling_rate"])
        if arr.ndim != 1 or arr.size == 0:
            return None
        if arr.shape[0] < sr * 0.1:
            return None
        if np.max(np.abs(arr)) < 1e-6:
            return None
        dur = arr.shape[0] / sr
        if dur > 20.0:
            return None
        text = normalize_ar(ex.get("transcript", ""))
        if not transcript_ok(text):
            return None
        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]
        ids = processor.tokenizer(text, max_length=448, truncation=True).input_ids
        if len(ids) < 2:
            return None
        return {"input_features": feats, "labels": ids}
    except Exception:
        return None


# ── Phase 1: build eval set + eval_id set ─────────────────────────────────────
print("\n⬇️  Phase 1 — streaming eval split (reserved ids)...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)
stream_ev = load_stream(DATASET_NAME, seed=SEED)

eval_features: List[np.ndarray] = []
eval_labels: List[List[int]] = []
eval_ids: set = set()
seen = 0
t0 = time.time()

for ex in stream_ev:
    seen += 1
    if not transcript_ok(ex.get("transcript", "")):
        continue
    eid = ex_id(ex)
    if eid in eval_ids:
        continue
    item = process_sample(ex, processor)
    if item is None:
        continue
    eval_ids.add(eid)
    eval_features.append(item["input_features"])
    eval_labels.append(item["labels"])
    if len(eval_features) >= MAX_EVAL_SAMPLES:
        break
    if seen % 2000 == 0:
        print(f"   scanned={seen:,} | eval_ok={len(eval_features):,}/{MAX_EVAL_SAMPLES}")

if len(eval_features) < max(100, MAX_EVAL_SAMPLES // 3):
    raise RuntimeError(f"Too few eval samples collected: {len(eval_features)}")

eval_ds = Dataset.from_dict({"input_features": eval_features, "labels": eval_labels})
del eval_features, eval_labels
gc.collect()
print(f"   ✅ Eval built: {len(eval_ds):,} | unique ids: {len(eval_ids):,} | {time.time()-t0:.1f}s")

# ── Phase 2: train generator (skips eval ids) ───────────────────────────────
def train_example_gen():
    stream_tr = load_stream(DATASET_NAME, seed=SEED + 7)
    n = 0
    skipped_eval = 0
    for ex in stream_tr:
        if not transcript_ok(ex.get("transcript", "")):
            continue
        eid = ex_id(ex)
        if eid in eval_ids:
            skipped_eval += 1
            continue
        item = process_sample(ex, processor)
        if item is None:
            continue
        yield item
        n += 1
        if n >= MAX_TRAIN_SAMPLES:
            break
        if n % 5000 == 0:
            print(f"   train_yield={n:,} / {MAX_TRAIN_SAMPLES:,} | skipped_eval_hits={skipped_eval:,}")


from datasets import IterableDataset
train_ds = IterableDataset.from_generator(train_example_gen)

# Steps (IterableDataset length unknown → use max_steps)
world = 1
samples_per_step = PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM * world
steps_per_epoch = max(1, math.ceil(MAX_TRAIN_SAMPLES / samples_per_step))
max_steps = steps_per_epoch * NUM_EPOCHS

# Time estimate
est_hours = max_steps * SEC_PER_STEP_GUESS / 3600.0
print("\n📌 Plan")
print(f"   Train cap: {MAX_TRAIN_SAMPLES:,} | Eval: {len(eval_ds):,}")
print(f"   Batch {PER_DEVICE_TRAIN_BATCH} × grad_accum {GRAD_ACCUM} → {samples_per_step} samples/step")
print(f"   Steps/epoch ≈ {steps_per_epoch:,} | max_steps = {max_steps:,} ({NUM_EPOCHS} logical epochs)")
print(f"   ⏱️  Rough GPU time ≈ {est_hours:.1f} h (guess {SEC_PER_STEP_GUESS}s/step on P100-class); 20h budget: {'OK' if est_hours < 19 else 'tight — lower NUM_EPOCHS or MAX_TRAIN_SAMPLES'}")

# ── Model (full fine-tune — all params trainable) ───────────────────────────
print("\n⬇️  Loading Whisper-small (full weights)...")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME, low_cpu_mem_usage=True)
model.config.use_cache = False
model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.to(DEVICE)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable:,} (full fine-tune)")

# ── Collator ────────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]
        return {"input_features": batch["input_features"], "labels": labels}


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}


class WhisperEvalTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**{"language": "arabic", "task": "transcribe"}, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)


warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_steps=max_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=torch.cuda.is_available(),
    evaluation_strategy="steps",
    eval_steps=steps_per_epoch,
    save_strategy="steps",
    save_steps=steps_per_epoch,
    logging_steps=100,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.args._n_gpu = 1

_wall = time.time()
print("\n🚀 Starting training...")
train_result = trainer.train()
print(f"\n✅ Training finished in {(time.time()-_wall)/3600:.2f} h — steps: {train_result.global_step}")

metrics = trainer.evaluate()
print(f"📊 Best checkpoint eval WER: {metrics.get('eval_wer', float('nan')):.2f}%")

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"💾 Saved full model + processor → {OUTPUT_DIR}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n⏱️  Total wall time (build + train + eval + save): {(time.time()-t0)/3600:.2f} h")


✅ Device: cuda
   GPU: Tesla P100-PCIE-16GB (sm_60) | torch 2.2.2+cu118

⬇️  Phase 1 — streaming eval split (reserved ids)...
   ✅ Eval built: 1,500 | unique ids: 1,500 | 314.7s
   train_yield=5,000 / 100,000 | skipped_eval_hits=531
   train_yield=10,000 / 100,000 | skipped_eval_hits=1,091

📌 Plan
   Train cap: 100,000 | Eval: 1,500
   Batch 8 × grad_accum 2 → 16 samples/step
   Steps/epoch ≈ 6,250 | max_steps = 12,500 (2 logical epochs)
   ⏱️  Rough GPU time ≈ 9.4 h (guess 2.7s/step on P100-class); 20h budget: OK

⬇️  Loading Whisper-small (full weights)...
   Trainable parameters: 240,582,912 (full fine-tune)

🚀 Starting training...
{'loss': 3.8993, 'grad_norm': 16.063180923461914, 'learning_rate': 1.5520000000000001e-06, 'epoch': 0.11730205278592376}
{'loss': 1.9389, 'grad_norm': 18.940574645996094, 'learning_rate': 3.152e-06, 'epoch': 0.23460410557184752}
{'loss': 1.526, 'grad_norm': 17.421457290649414, 'learning_rate': 4.752e-06, 'epoch': 0.3519061583577713}
{'loss': 1.267, 'grad_

In [ ]:
import os
os.system(
    "pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2 "
    "--index-url https://download.pytorch.org/whl/cu118 --quiet"
)
os.kill(os.getpid(), 9)  # force kernel restart

In [1]:
import torch
print(torch.__version__)          # should be 2.2.2+cu118
x = torch.randn(4, 4, device="cuda")
print((x @ x).sum())             # should print a number, not crash

2.2.2+cu118
tensor(-5.4852, device='cuda:0')


In [1]:
import os
os.system("pip uninstall -y torchcodec")

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128


0

In [ ]:
import os, subprocess

# Step 1: Fix torch FIRST (P100 needs cu118, not cu128)
subprocess.run(
    "pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2 "
    "--index-url https://download.pytorch.org/whl/cu118 -q",
    shell=True
)

# Step 2: Fix datasets + audio backend
subprocess.run(
    "pip install datasets==2.20.0 soundfile librosa -q",
    shell=True
)

# Step 3: Remove torchcodec entirely
subprocess.run("pip uninstall -y torchcodec", shell=True)

print("✅ Done — restarting kernel...")
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 80.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 46.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 112.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 34.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.5.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.


In [2]:
!pip uninstall -y numpy
!pip install -q "numpy<2"
!pip install -q --upgrade "torch==2.2.2" "torchaudio==2.2.2"

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is

In [4]:
!pip uninstall -y numpy torch torchaudio torchvision scipy transformers datasets accelerate evaluate librosa soundfile
!pip install -q "numpy==1.26.4"
!pip install -q "scipy==1.11.4" "soundfile" "librosa==0.10.2"
!pip install -q "torch==2.2.2" "torchaudio==2.2.2" --index-url https://download.pytorch.org/whl/cu118
!pip install -q "transformers==4.38.2" "datasets==2.18.0" "accelerate==0.27.2" "evaluate==0.4.1"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: torch 2.2.2+cu118
Uninstalling torch-2.2.2+cu118:
  Successfully uninstalled torch-2.2.2+cu118
Found existing installation: torchaudio 2.2.2+cu118
Uninstalling torchaudio-2.2.2+cu118:
  Successfully uninstalled torchaudio-2.2.2+cu118
Found existing installation: torchvision 0.17.2+cu118
Uninstalling torchvision-0.17.2+cu118:
  Successfully uninstalled torchvision-0.17.2+cu118
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: datasets 2.20.0
Uninstalling datasets-2.20.0:
  Successfully uninstalled datasets-2.20.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import io, math, re
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from datasets import load_dataset
from transformers import WhisperProcessor

BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
TARGET_SR = 16000

def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return len(t) >= 5 and len(t.split()) >= 2 and len(t) < 400

def _read_audio_from_example(ex):
    audio = ex["audio"]

    if audio.get("array") is not None and audio.get("sampling_rate") is not None:
        arr = np.asarray(audio["array"], dtype=np.float32)
        sr = int(audio["sampling_rate"])
    elif audio.get("bytes") is not None:
        arr, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
    elif audio.get("path") is not None:
        arr, sr = sf.read(audio["path"], dtype="float32")
    else:
        raise ValueError(f"Unsupported audio keys: {list(audio.keys())}")

    if arr.ndim == 2:
        arr = arr.mean(axis=1)

    if sr != TARGET_SR:
        g = math.gcd(int(sr), int(TARGET_SR))
        arr = resample_poly(arr, TARGET_SR // g, sr // g).astype(np.float32)
        sr = TARGET_SR

    return arr, sr

processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)
ds = load_dataset(DATASET_NAME, split="train", streaming=True)

for i, ex in enumerate(ds):
    print(f"\n--- SAMPLE {i} ---")
    print("text:", ex["transcript"][:100])

    arr, sr = _read_audio_from_example(ex)
    print("audio shape:", arr.shape, "sr:", sr)

    feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]
    ids = processor.tokenizer(normalize_ar(ex["transcript"])).input_ids

    print("features shape:", np.array(feats).shape)
    print("label length:", len(ids))
    print("SUCCESS")
    if i == 2:
        break

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]


--- SAMPLE 0 ---
text: أسبقية قبل أنا ما وصلت خممت فيه كيما باش نحكيو من بعد إلا ما أنا كإنطريبرنور كباعث مشروع صارولي برشا
audio shape: (225808,) sr: 16000


2026-04-10 15:08:56.300128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775833736.481749     386 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775833736.533916     386 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775833736.945954     386 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775833736.945989     386 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775833736.945992     386 computation_placer.cc:177] computation placer alr

features shape: (80, 3000)
label length: 109
SUCCESS

--- SAMPLE 1 ---
text: و الصلاة و السلام على رسول الله
audio shape: (55946,) sr: 16000
features shape: (80, 3000)
label length: 15
SUCCESS

--- SAMPLE 2 ---
text: و على آله وصحابته و من والاه
audio shape: (69249,) sr: 16000
features shape: (80, 3000)
label length: 16
SUCCESS


In [ ]:
# LinTO Tunisian Arabic — FULL fine-tune (no LoRA), streaming up to 100k train samples
import os, re, gc, time, math, io, warnings, random

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import evaluate
import soundfile as sf
from scipy.signal import resample_poly
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from transformers.generation.configuration_utils import GenerationConfig

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
OUTPUT_DIR = "/kaggle/working/whisper_small_linto_100k_ft"
TARGET_SR = 16_000

MAX_TRAIN_SAMPLES = 100_000
MAX_EVAL_SAMPLES = 1_500
STREAM_SHUFFLE_BUFFER = 50_000

PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM = 2
NUM_EPOCHS = 2
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.05

MAX_TRAIN_HOURS = 2.0
SEC_PER_STEP_GUESS = 2.7
EVAL_EVERY_STEPS = 100

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"   GPU: {p.name} (sm_{p.major}{p.minor}) | torch {torch.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
# TEXT HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_text(ex) -> str:
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )

def ex_id(ex) -> str:
    if ex.get("audio_id") is not None:
        return str(ex["audio_id"])

    text = get_text(ex)
    audio = ex.get("audio", {})
    audio_path = audio.get("path", "") if isinstance(audio, dict) else ""
    return str(hash((normalize_ar(text)[:120], audio_path)))

# ──────────────────────────────────────────────────────────────────────────────
# STREAM LOADER
# ──────────────────────────────────────────────────────────────────────────────
def load_stream(name: str, seed: int):
    ds = load_dataset(name, split="train", streaming=True)
    try:
        ds = ds.shuffle(seed=seed, buffer_size=STREAM_SHUFFLE_BUFFER)
    except Exception as e:
        print("⚠️ Shuffle skipped:", e)
    return ds

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO READER
# ──────────────────────────────────────────────────────────────────────────────
def _read_audio_from_example(ex):
    audio = ex.get("audio", None)
    if audio is None:
        raise ValueError(f"No 'audio' field. Example keys: {list(ex.keys())}")

    # Common case: decoded audio dict
    if isinstance(audio, dict):
        if audio.get("array") is not None and audio.get("sampling_rate") is not None:
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = int(audio["sampling_rate"])

        elif audio.get("bytes") is not None:
            arr, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")

        elif audio.get("path") is not None:
            arr, sr = sf.read(audio["path"], dtype="float32")

        else:
            raise ValueError(f"Unsupported audio dict keys: {list(audio.keys())}")
    else:
        raise ValueError(f"Unsupported audio type: {type(audio)}")

    arr = np.asarray(arr, dtype=np.float32)

    if arr.ndim == 2:
        arr = arr.mean(axis=1)

    if arr.ndim != 1:
        raise ValueError(f"Audio must be 1D after mono conversion, got shape={arr.shape}")

    if sr != TARGET_SR:
        g = math.gcd(int(sr), int(TARGET_SR))
        arr = resample_poly(arr, TARGET_SR // g, sr // g).astype(np.float32, copy=False)
        sr = TARGET_SR

    return arr, sr

# ──────────────────────────────────────────────────────────────────────────────
# SAMPLE PROCESSOR
# ──────────────────────────────────────────────────────────────────────────────
def process_sample(ex, processor: WhisperProcessor, debug=False):
    try:
        arr, sr = _read_audio_from_example(ex)

        if arr.size == 0:
            if debug: print("Rejected: empty audio")
            return None

        if arr.shape[0] < sr * 0.1:
            if debug: print("Rejected: too short")
            return None

        if np.max(np.abs(arr)) < 1e-6:
            if debug: print("Rejected: silent audio")
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:
            if debug: print(f"Rejected: too long ({dur:.2f}s)")
            return None

        raw_text = get_text(ex)
        text = normalize_ar(raw_text)

        if not transcript_ok(text):
            if debug: print("Rejected: bad transcript ->", repr(raw_text[:120]))
            return None

        feats = processor.feature_extractor(
            arr, sampling_rate=sr
        ).input_features[0]

        ids = processor.tokenizer(
            text,
            max_length=448,
            truncation=True
        ).input_ids

        if len(ids) < 2:
            if debug: print("Rejected: tokenized text too short")
            return None

        return {"input_features": feats, "labels": ids}

    except Exception as e:
        if debug:
            print("process_sample ERROR:", type(e).__name__, str(e))
        return None

# ──────────────────────────────────────────────────────────────────────────────
# LOAD PROCESSOR
# ──────────────────────────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
# ──────────────────────────────────────────────────────────────────────────────
# DEBUG: PRINT FIRST FEW SAMPLES
# ──────────────────────────────────────────────────────────────────────────────
print("\n🔍 Debugging first 5 raw samples...")
dbg_stream = load_stream(DATASET_NAME, seed=SEED)

for i, raw_ex in enumerate(dbg_stream):
    print(f"\n--- SAMPLE {i} ---")
    print("Keys:", list(raw_ex.keys()))
    txt = get_text(raw_ex)
    print("Text preview:", repr(txt[:120]))
    audio = raw_ex.get("audio", None)
    print("Audio type:", type(audio))
    if isinstance(audio, dict):
        print("Audio keys:", list(audio.keys()))
    result = process_sample(raw_ex, processor, debug=True)
    print("Accepted:", result is not None)
    if i >= 4:
        break

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD EVAL SET
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Phase 1 — streaming eval split (reserved ids)...")
stream_ev = load_stream(DATASET_NAME, seed=SEED)

eval_features: List[np.ndarray] = []
eval_labels: List[List[int]] = []
eval_ids: set = set()

seen = 0
t0 = time.time()

for ex in stream_ev:
    seen += 1

    text = get_text(ex)
    if not transcript_ok(text):
        continue

    eid = ex_id(ex)
    if eid in eval_ids:
        continue

    item = process_sample(ex, processor, debug=False)
    if item is None:
        continue

    eval_ids.add(eid)
    eval_features.append(item["input_features"])
    eval_labels.append(item["labels"])

    if len(eval_features) >= MAX_EVAL_SAMPLES:
        break

    if seen % 2000 == 0:
        print(f"   scanned={seen:,} | eval_ok={len(eval_features):,}/{MAX_EVAL_SAMPLES}")

print(f"\n   Final scanned={seen:,} | eval_ok={len(eval_features):,}")

if len(eval_features) < max(100, MAX_EVAL_SAMPLES // 3):
    raise RuntimeError(
        f"Too few eval samples collected: {len(eval_features)}. "
        f"Dataset text/audio format is still not matching the code."
    )

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels
})
del eval_features, eval_labels
gc.collect()

print(f"   ✅ Eval built: {len(eval_ds):,} | unique ids: {len(eval_ids):,} | {time.time()-t0:.1f}s")

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 2: TRAIN GENERATOR
# ──────────────────────────────────────────────────────────────────────────────
def train_example_gen():
    stream_tr = load_stream(DATASET_NAME, seed=SEED + 7)
    n = 0
    skipped_eval = 0

    for ex in stream_tr:
        text = get_text(ex)
        if not transcript_ok(text):
            continue

        eid = ex_id(ex)
        if eid in eval_ids:
            skipped_eval += 1
            continue

        item = process_sample(ex, processor, debug=False)
        if item is None:
            continue

        yield item
        n += 1

        if n >= MAX_TRAIN_SAMPLES:
            break

        if n % 5000 == 0:
            print(f"   train_yield={n:,}/{MAX_TRAIN_SAMPLES:,} | skipped_eval_hits={skipped_eval:,}")

train_ds = Dataset.from_generator(train_example_gen)

# ──────────────────────────────────────────────────────────────────────────────
# TRAINING STEPS
# ──────────────────────────────────────────────────────────────────────────────
world = 1
samples_per_step = PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM * world
steps_per_epoch = max(1, math.ceil(MAX_TRAIN_SAMPLES / samples_per_step))
planned_steps = steps_per_epoch * NUM_EPOCHS
max_steps_for_budget = max(1, int((MAX_TRAIN_HOURS * 3600) / SEC_PER_STEP_GUESS))
max_steps = min(planned_steps, max_steps_for_budget)

est_hours = max_steps * SEC_PER_STEP_GUESS / 3600.0

print("\n📌 Plan")
print(f"   Train cap: {MAX_TRAIN_SAMPLES:,} | Eval: {len(eval_ds):,}")
print(f"   Batch {PER_DEVICE_TRAIN_BATCH} × grad_accum {GRAD_ACCUM} → {samples_per_step} samples/step")
print(f"   Steps/epoch ≈ {steps_per_epoch:,} | planned_steps = {planned_steps:,}")
print(f"   Runtime cap: {MAX_TRAIN_HOURS:.1f}h → max_steps_for_budget = {max_steps_for_budget:,}")
print(f"   Using max_steps = {max_steps:,}")
print(f"   ⏱️ Rough GPU time ≈ {est_hours:.2f} h")

# ──────────────────────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Loading Whisper-small (full weights)...")
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
)

model.config.use_cache = False
model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_NAME)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.to(DEVICE)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable:,} (full fine-tune)")

# ──────────────────────────────────────────────────────────────────────────────
# DATA COLLATOR
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

class WhisperEvalTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**{"language": "arabic", "task": "transcribe"}, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_steps=max_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True if torch.cuda.is_available() else False, 
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_STEPS,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.args._n_gpu = 1

# ──────────────────────────────────────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────────────────────────────────────
_wall = time.time()
print("\n🚀 Starting training...")

train_result = None
try:
    train_result = trainer.train()
    print(f"\n✅ Training finished in {(time.time()-_wall)/3600:.2f} h — steps: {train_result.global_step}")
except KeyboardInterrupt:
    print("\n⚠️ Interrupted. Saving emergency snapshot...")
    emergency_dir = os.path.join(OUTPUT_DIR, "interrupted")
    os.makedirs(emergency_dir, exist_ok=True)
    trainer.save_model(emergency_dir)
    processor.save_pretrained(emergency_dir)
    print(f"💾 Emergency snapshot saved → {emergency_dir}")
    raise
finally:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"💾 Saved latest full model + processor → {OUTPUT_DIR}")

metrics = trainer.evaluate()
print(f"📊 Best checkpoint eval WER: {metrics.get('eval_wer', float('nan')):.2f}%")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n⏱️ Total wall time (build + train + eval + save): {(time.time()-t0)/3600:.2f} h")

✅ Device: cuda
   GPU: Tesla P100-PCIE-16GB (sm_60) | torch 2.2.2+cu118

🔍 Debugging first 5 raw samples...


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 0b7547ff-652f-466d-adbe-388f3cdd0e0c)')' thrown while requesting GET https://huggingface.co/datasets/linagora/linto-dataset-audio-ar-tn/resolve/99796bfd5e8b48febaac60633d1ec97a658fa511/data/TunSwitchCS/train/train-00004-of-00005.parquet
Retrying in 1s [Retry 1/5].



--- SAMPLE 0 ---
Keys: ['audio_id', 'audio', 'segments', 'transcript']
Text preview: 'ما نقولش لا تو نشريها و مباعد نعمللها recyclage ولا تو نستعملها فالدار ولا تو نخليها'
Audio type: <class 'dict'>
Audio keys: ['path', 'array', 'sampling_rate']
Accepted: True

--- SAMPLE 1 ---
Keys: ['audio_id', 'audio', 'segments', 'transcript']
Text preview: 'ماكنتش نعرف كيفاش نقرا كنت ضايعة عاللخر و ماكنتش نقرا étude'
Audio type: <class 'dict'>
Audio keys: ['path', 'array', 'sampling_rate']
Accepted: True

--- SAMPLE 2 ---
Keys: ['audio_id', 'audio', 'segments', 'transcript']
Text preview: 'قلت لك قلت لك'
Audio type: <class 'dict'>
Audio keys: ['path', 'array', 'sampling_rate']
Accepted: True

--- SAMPLE 3 ---
Keys: ['audio_id', 'audio', 'segments', 'transcript']
Text preview: 'في الكامور'
Audio type: <class 'dict'>
Audio keys: ['path', 'array', 'sampling_rate']
Accepted: True

--- SAMPLE 4 ---
Keys: ['audio_id', 'audio', 'segments', 'transcript']
Text preview: 'يشوف روحو'
Audio type: <class 'dict

In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.__version__)

Tesla T4
2.10.0+cu128


In [6]:

CACHE_DIR = "/kaggle/working/hf_cache"
RESUME_CHECKPOINT = "/kaggle/input/datasets/aymendhieb/whisper-tunisian-100k/whisper_tunisian_100k/checkpoint2100"

# LinTO Tunisian Arabic — FULL fine-tune (no LoRA), streaming up to 100k train samples
import os, re, gc, time, math, io, warnings, random

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import evaluate
import soundfile as sf
from scipy.signal import resample_poly
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from transformers.generation.configuration_utils import GenerationConfig

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
OUTPUT_DIR = "/kaggle/working/whisper_small_linto_100k_ft"
TARGET_SR = 16_000

MAX_TRAIN_SAMPLES = 100_000
MAX_EVAL_SAMPLES = 1_500
STREAM_SHUFFLE_BUFFER = 50_000

PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM = 2
NUM_EPOCHS = 2
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.05

MAX_TRAIN_HOURS = 2.0
SEC_PER_STEP_GUESS = 2.7
EVAL_EVERY_STEPS = 100

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"   GPU: {p.name} (sm_{p.major}{p.minor}) | torch {torch.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
# TEXT HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_text(ex) -> str:
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )

def ex_id(ex) -> str:
    if ex.get("audio_id") is not None:
        return str(ex["audio_id"])

    text = get_text(ex)
    audio = ex.get("audio", {})
    audio_path = audio.get("path", "") if isinstance(audio, dict) else ""
    return str(hash((normalize_ar(text)[:120], audio_path)))

# ──────────────────────────────────────────────────────────────────────────────
# STREAM LOADER
# ──────────────────────────────────────────────────────────────────────────────
def load_stream(name: str, seed: int):
    ds = load_dataset(name, split="train", streaming=True)
    try:
        ds = ds.shuffle(seed=seed, buffer_size=STREAM_SHUFFLE_BUFFER)
    except Exception as e:
        print("⚠️ Shuffle skipped:", e)
    return ds

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO READER
# ──────────────────────────────────────────────────────────────────────────────
def _read_audio_from_example(ex):
    audio = ex.get("audio", None)
    if audio is None:
        raise ValueError(f"No 'audio' field. Example keys: {list(ex.keys())}")

    if isinstance(audio, dict):
        if audio.get("array") is not None and audio.get("sampling_rate") is not None:
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = int(audio["sampling_rate"])
        elif audio.get("bytes") is not None:
            arr, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
        elif audio.get("path") is not None:
            arr, sr = sf.read(audio["path"], dtype="float32")
        else:
            raise ValueError(f"Unsupported audio dict keys: {list(audio.keys())}")
    else:
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            arr = samples.data.cpu().numpy() if hasattr(samples.data, "cpu") else np.asarray(samples.data)
            sr = int(samples.sample_rate)
        elif hasattr(audio, "decode"):
            decoded = audio.decode()
            if isinstance(decoded, dict) and "array" in decoded and "sampling_rate" in decoded:
                arr = np.asarray(decoded["array"], dtype=np.float32)
                sr = int(decoded["sampling_rate"])
            else:
                raise ValueError(f"Unsupported decoded audio output type: {type(decoded)}")
        else:
            raise ValueError(f"Unsupported audio type: {type(audio)}")

    arr = np.asarray(arr, dtype=np.float32)

    if arr.ndim == 2:
        if arr.shape[0] <= 4 and arr.shape[1] > arr.shape[0]:
            arr = arr.mean(axis=0)
        elif arr.shape[1] <= 4 and arr.shape[0] > arr.shape[1]:
            arr = arr.mean(axis=1)
        else:
            arr = arr.mean(axis=0 if arr.shape[0] < arr.shape[1] else 1)

    if arr.ndim != 1:
        raise ValueError(f"Audio must be 1D after mono conversion, got shape={arr.shape}")

    if sr != TARGET_SR:
        g = math.gcd(int(sr), int(TARGET_SR))
        arr = resample_poly(arr, TARGET_SR // g, sr // g).astype(np.float32, copy=False)
        sr = TARGET_SR

    return arr, sr


def process_sample(ex, processor: WhisperProcessor, debug=False):
    try:
        arr, sr = _read_audio_from_example(ex)  # ← was missing

        if debug:
            print("Waveform shape:", arr.shape, "| sr:", sr, "| duration:", arr.shape[0] / sr)

        if arr.size == 0:
            if debug: print("Rejected: empty audio")
            return None

        if arr.shape[0] < sr * 0.1:
            if debug: print("Rejected: too short")
            return None

        if np.max(np.abs(arr)) < 1e-6:
            if debug: print("Rejected: silent audio")
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:
            if debug: print(f"Rejected: too long ({dur:.2f}s)")
            return None

        raw_text = get_text(ex)
        text = normalize_ar(raw_text)

        if not transcript_ok(text):
            if debug: print("Rejected: bad transcript ->", repr(raw_text[:120]))
            return None

        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]

        ids = processor.tokenizer(text, max_length=448, truncation=True).input_ids

        if len(ids) < 2:
            if debug: print("Rejected: tokenized text too short")
            return None

        return {"input_features": feats, "labels": ids}

    except Exception as e:
        if debug:
            print("process_sample ERROR:", type(e).__name__, str(e))
        return None

# ──────────────────────────────────────────────────────────────────────────────
# LOAD PROCESSOR
# ──────────────────────────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
# ──────────────────────────────────────────────────────────────────────────────
# DEBUG: PRINT FIRST FEW SAMPLES
# ──────────────────────────────────────────────────────────────────────────────
print("\n🔍 Debugging first 5 raw samples...")
dbg_stream = load_stream(DATASET_NAME, seed=SEED)

for i, raw_ex in enumerate(dbg_stream):
    print(f"\n--- SAMPLE {i} ---")
    print("Keys:", list(raw_ex.keys()))
    txt = get_text(raw_ex)
    print("Text preview:", repr(txt[:120]))
    audio = raw_ex.get("audio", None)
    print("Audio type:", type(audio))
    if isinstance(audio, dict):
        print("Audio keys:", list(audio.keys()))
    result = process_sample(raw_ex, processor, debug=True)
    print("Accepted:", result is not None)
    if i >= 4:
        break

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD EVAL SET
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Phase 1 — streaming eval split (reserved ids)...")
stream_ev = load_stream(DATASET_NAME, seed=SEED)

eval_features: List[np.ndarray] = []
eval_labels: List[List[int]] = []
eval_ids: set = set()

seen = 0
t0 = time.time()

for ex in stream_ev:
    seen += 1

    text = get_text(ex)
    if not transcript_ok(text):
        continue

    eid = ex_id(ex)
    if eid in eval_ids:
        continue

    item = process_sample(ex, processor, debug=False)
    if item is None:
        continue

    eval_ids.add(eid)
    eval_features.append(item["input_features"])
    eval_labels.append(item["labels"])

    if len(eval_features) >= MAX_EVAL_SAMPLES:
        break

    if seen % 2000 == 0:
        print(f"   scanned={seen:,} | eval_ok={len(eval_features):,}/{MAX_EVAL_SAMPLES}")

print(f"\n   Final scanned={seen:,} | eval_ok={len(eval_features):,}")

if len(eval_features) < max(100, MAX_EVAL_SAMPLES // 3):
    raise RuntimeError(
        f"Too few eval samples collected: {len(eval_features)}. "
        f"Dataset text/audio format is still not matching the code."
    )

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels
})
del eval_features, eval_labels
gc.collect()

print(f"   ✅ Eval built: {len(eval_ds):,} | unique ids: {len(eval_ids):,} | {time.time()-t0:.1f}s")

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 2: TRAIN GENERATOR
# ──────────────────────────────────────────────────────────────────────────────
def train_example_gen():
    stream_tr = load_stream(DATASET_NAME, seed=SEED + 7)
    n = 0
    skipped_eval = 0

    for ex in stream_tr:
        text = get_text(ex)
        if not transcript_ok(text):
            continue

        eid = ex_id(ex)
        if eid in eval_ids:
            skipped_eval += 1
            continue

        item = process_sample(ex, processor, debug=False)
        if item is None:
            continue

        yield item
        n += 1

        if n >= MAX_TRAIN_SAMPLES:
            break

        if n % 5000 == 0:
            print(f"   train_yield={n:,}/{MAX_TRAIN_SAMPLES:,} | skipped_eval_hits={skipped_eval:,}")

train_ds = Dataset.from_generator(train_example_gen)

# ──────────────────────────────────────────────────────────────────────────────
# TRAINING STEPS
# ──────────────────────────────────────────────────────────────────────────────
world = 1
samples_per_step = PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM * world
steps_per_epoch = max(1, math.ceil(MAX_TRAIN_SAMPLES / samples_per_step))
planned_steps = steps_per_epoch * NUM_EPOCHS
max_steps_for_budget = max(1, int((MAX_TRAIN_HOURS * 3600) / SEC_PER_STEP_GUESS))
max_steps = min(planned_steps, max_steps_for_budget)

est_hours = max_steps * SEC_PER_STEP_GUESS / 3600.0

print("\n📌 Plan")
print(f"   Train cap: {MAX_TRAIN_SAMPLES:,} | Eval: {len(eval_ds):,}")
print(f"   Batch {PER_DEVICE_TRAIN_BATCH} × grad_accum {GRAD_ACCUM} → {samples_per_step} samples/step")
print(f"   Steps/epoch ≈ {steps_per_epoch:,} | planned_steps = {planned_steps:,}")
print(f"   Runtime cap: {MAX_TRAIN_HOURS:.1f}h → max_steps_for_budget = {max_steps_for_budget:,}")
print(f"   Using max_steps = {max_steps:,}")
print(f"   ⏱️ Rough GPU time ≈ {est_hours:.2f} h")

# ──────────────────────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Loading Whisper-small (full weights)...")
model = WhisperForConditionalGeneration.from_pretrained(
    RESUME_CHECKPOINT,
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
)

model.config.use_cache = False
model.generation_config = GenerationConfig.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.to(DEVICE)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable:,} (full fine-tune)")

# ──────────────────────────────────────────────────────────────────────────────
# DATA COLLATOR
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

class WhisperEvalTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**{"language": "arabic", "task": "transcribe"}, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_steps=max_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True if torch.cuda.is_available() else False, 
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_STEPS,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.args._n_gpu = 1

# ──────────────────────────────────────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────────────────────────────────────
_wall = time.time()
print("\n🚀 Starting training...")


print("Checkpoint exists:", os.path.exists(RESUME_CHECKPOINT))
print("Checkpoint is dir:", os.path.isdir(RESUME_CHECKPOINT))
print("Checkpoint files:", os.listdir(RESUME_CHECKPOINT))

train_result = None
try:
    train_result = trainer.train()
    print(f"\n✅ Training finished in {(time.time()-_wall)/3600:.2f} h — steps: {train_result.global_step}")
except KeyboardInterrupt:
    print("\n⚠️ Interrupted. Saving emergency snapshot...")
    emergency_dir = os.path.join(OUTPUT_DIR, "interrupted")
    os.makedirs(emergency_dir, exist_ok=True)
    trainer.save_model(emergency_dir)
    processor.save_pretrained(emergency_dir)
    print(f"💾 Emergency snapshot saved → {emergency_dir}")
    raise
finally:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"💾 Saved latest full model + processor → {OUTPUT_DIR}")

metrics = trainer.evaluate()
print(f"📊 Best checkpoint eval WER: {metrics.get('eval_wer', float('nan')):.2f}%")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n⏱️ Total wall time (build + train + eval + save): {(time.time()-t0)/3600:.2f} h")

✅ Device: cuda
   GPU: Tesla T4 (sm_75) | torch 2.10.0+cu128


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Emergency snapshot saved → /kaggle/working/whisper_small_linto_100k_ft/interrupted


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Saved latest full model + processor → /kaggle/working/whisper_small_linto_100k_ft


KeyboardInterrupt: 

In [5]:

CACHE_DIR = "/kaggle/working/hf_cache"
RESUME_CHECKPOINT = "/kaggle/input/datasets/aymendhieb/checkpoint100k/check2point2300"

# LinTO Tunisian Arabic — FULL fine-tune (no LoRA), streaming up to 200k train samples
import os, re, gc, time, math, io, warnings, random

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import evaluate
import soundfile as sf
from scipy.signal import resample_poly
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from transformers.generation.configuration_utils import GenerationConfig

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
OUTPUT_DIR = "/kaggle/working/whisper_small_linto_200k_ft"
TARGET_SR = 16_000

MAX_TRAIN_SAMPLES = 200_000
MAX_EVAL_SAMPLES = 1_000
STREAM_SHUFFLE_BUFFER = 50_000

PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM = 2
NUM_EPOCHS = 1
LEARNING_RATE = 5e-6
WARMUP_RATIO = 0.03

MAX_TRAIN_HOURS = 10.0
SEC_PER_STEP_GUESS = 2.7
EVAL_EVERY_STEPS = 5000

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"   GPU: {p.name} (sm_{p.major}{p.minor}) | torch {torch.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
# TEXT HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_text(ex) -> str:
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 5
        and len(t.split()) >= 2
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 400
    )

def ex_id(ex) -> str:
    if ex.get("audio_id") is not None:
        return str(ex["audio_id"])

    text = get_text(ex)
    audio = ex.get("audio", {})
    audio_path = audio.get("path", "") if isinstance(audio, dict) else ""
    return str(hash((normalize_ar(text)[:120], audio_path)))

# ──────────────────────────────────────────────────────────────────────────────
# STREAM LOADER
# ──────────────────────────────────────────────────────────────────────────────
def load_stream(name: str, seed: int):
    ds = load_dataset(name, split="train", streaming=True)
    try:
        ds = ds.shuffle(seed=seed, buffer_size=STREAM_SHUFFLE_BUFFER)
    except Exception as e:
        print("⚠️ Shuffle skipped:", e)
    return ds

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO READER
# ──────────────────────────────────────────────────────────────────────────────
def _read_audio_from_example(ex):
    audio = ex.get("audio", None)
    if audio is None:
        raise ValueError(f"No 'audio' field. Example keys: {list(ex.keys())}")

    if isinstance(audio, dict):
        if audio.get("array") is not None and audio.get("sampling_rate") is not None:
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = int(audio["sampling_rate"])
        elif audio.get("bytes") is not None:
            arr, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
        elif audio.get("path") is not None:
            arr, sr = sf.read(audio["path"], dtype="float32")
        else:
            raise ValueError(f"Unsupported audio dict keys: {list(audio.keys())}")
    else:
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            arr = samples.data.cpu().numpy() if hasattr(samples.data, "cpu") else np.asarray(samples.data)
            sr = int(samples.sample_rate)
        elif hasattr(audio, "decode"):
            decoded = audio.decode()
            if isinstance(decoded, dict) and "array" in decoded and "sampling_rate" in decoded:
                arr = np.asarray(decoded["array"], dtype=np.float32)
                sr = int(decoded["sampling_rate"])
            else:
                raise ValueError(f"Unsupported decoded audio output type: {type(decoded)}")
        else:
            raise ValueError(f"Unsupported audio type: {type(audio)}")

    arr = np.asarray(arr, dtype=np.float32)

    if arr.ndim == 2:
        if arr.shape[0] <= 4 and arr.shape[1] > arr.shape[0]:
            arr = arr.mean(axis=0)
        elif arr.shape[1] <= 4 and arr.shape[0] > arr.shape[1]:
            arr = arr.mean(axis=1)
        else:
            arr = arr.mean(axis=0 if arr.shape[0] < arr.shape[1] else 1)

    if arr.ndim != 1:
        raise ValueError(f"Audio must be 1D after mono conversion, got shape={arr.shape}")

    if sr != TARGET_SR:
        g = math.gcd(int(sr), int(TARGET_SR))
        arr = resample_poly(arr, TARGET_SR // g, sr // g).astype(np.float32, copy=False)
        sr = TARGET_SR

    return arr, sr


def process_sample(ex, processor: WhisperProcessor, debug=False):
    try:
        arr, sr = _read_audio_from_example(ex)  # ← was missing

        if debug:
            print("Waveform shape:", arr.shape, "| sr:", sr, "| duration:", arr.shape[0] / sr)

        if arr.size == 0:
            if debug: print("Rejected: empty audio")
            return None

        if arr.shape[0] < sr * 0.1:
            if debug: print("Rejected: too short")
            return None

        if np.max(np.abs(arr)) < 1e-6:
            if debug: print("Rejected: silent audio")
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:
            if debug: print(f"Rejected: too long ({dur:.2f}s)")
            return None

        raw_text = get_text(ex)
        text = normalize_ar(raw_text)

        if not transcript_ok(text):
            if debug: print("Rejected: bad transcript ->", repr(raw_text[:120]))
            return None

        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]

        ids = processor.tokenizer(text, max_length=448, truncation=True).input_ids

        if len(ids) < 2:
            if debug: print("Rejected: tokenized text too short")
            return None

        return {"input_features": feats, "labels": ids}

    except Exception as e:
        if debug:
            print("process_sample ERROR:", type(e).__name__, str(e))
        return None

# ──────────────────────────────────────────────────────────────────────────────
# LOAD PROCESSOR
# ──────────────────────────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
# ──────────────────────────────────────────────────────────────────────────────
# DEBUG: PRINT FIRST FEW SAMPLES
# ──────────────────────────────────────────────────────────────────────────────
print("\n🔍 Debugging first 5 raw samples...")
dbg_stream = load_stream(DATASET_NAME, seed=SEED)

for i, raw_ex in enumerate(dbg_stream):
    print(f"\n--- SAMPLE {i} ---")
    print("Keys:", list(raw_ex.keys()))
    txt = get_text(raw_ex)
    print("Text preview:", repr(txt[:120]))
    audio = raw_ex.get("audio", None)
    print("Audio type:", type(audio))
    if isinstance(audio, dict):
        print("Audio keys:", list(audio.keys()))
    result = process_sample(raw_ex, processor, debug=True)
    print("Accepted:", result is not None)
    if i >= 4:
        break

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD EVAL SET
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Phase 1 — streaming eval split (reserved ids)...")
stream_ev = load_stream(DATASET_NAME, seed=SEED)

eval_features: List[np.ndarray] = []
eval_labels: List[List[int]] = []
eval_ids: set = set()

seen = 0
t0 = time.time()

for ex in stream_ev:
    seen += 1

    text = get_text(ex)
    if not transcript_ok(text):
        continue

    eid = ex_id(ex)
    if eid in eval_ids:
        continue

    item = process_sample(ex, processor, debug=False)
    if item is None:
        continue

    eval_ids.add(eid)
    eval_features.append(item["input_features"])
    eval_labels.append(item["labels"])

    if len(eval_features) >= MAX_EVAL_SAMPLES:
        break

    if seen % 2000 == 0:
        print(f"   scanned={seen:,} | eval_ok={len(eval_features):,}/{MAX_EVAL_SAMPLES}")

print(f"\n   Final scanned={seen:,} | eval_ok={len(eval_features):,}")

if len(eval_features) < max(100, MAX_EVAL_SAMPLES // 3):
    raise RuntimeError(
        f"Too few eval samples collected: {len(eval_features)}. "
        f"Dataset text/audio format is still not matching the code."
    )

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels
})
del eval_features, eval_labels
gc.collect()

print(f"   ✅ Eval built: {len(eval_ds):,} | unique ids: {len(eval_ids):,} | {time.time()-t0:.1f}s")

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 2: TRAIN GENERATOR
# ──────────────────────────────────────────────────────────────────────────────
def train_example_gen():
    stream_tr = load_stream(DATASET_NAME, seed=SEED + 7)
    n = 0
    skipped_eval = 0

    for ex in stream_tr:
        text = get_text(ex)
        if not transcript_ok(text):
            continue

        eid = ex_id(ex)
        if eid in eval_ids:
            skipped_eval += 1
            continue

        item = process_sample(ex, processor, debug=False)
        if item is None:
            continue

        yield item
        n += 1

        if n >= MAX_TRAIN_SAMPLES:
            break

        if n % 5000 == 0:
            print(f"   train_yield={n:,}/{MAX_TRAIN_SAMPLES:,} | skipped_eval_hits={skipped_eval:,}")

train_ds = Dataset.from_generator(train_example_gen)

# ──────────────────────────────────────────────────────────────────────────────
# TRAINING STEPS
# ──────────────────────────────────────────────────────────────────────────────
world = 1
samples_per_step = PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM * world
steps_per_epoch = max(1, math.ceil(MAX_TRAIN_SAMPLES / samples_per_step))
planned_steps = steps_per_epoch * NUM_EPOCHS
max_steps_for_budget = max(1, int((MAX_TRAIN_HOURS * 3600) / SEC_PER_STEP_GUESS))
max_steps = min(planned_steps, max_steps_for_budget, 12500)

est_hours = max_steps * SEC_PER_STEP_GUESS / 3600.0

print("\n📌 Plan")
print(f"   Train cap: {MAX_TRAIN_SAMPLES:,} | Eval: {len(eval_ds):,}")
print(f"   Batch {PER_DEVICE_TRAIN_BATCH} × grad_accum {GRAD_ACCUM} → {samples_per_step} samples/step")
print(f"   Steps/epoch ≈ {steps_per_epoch:,} | planned_steps = {planned_steps:,}")
print(f"   Runtime cap: {MAX_TRAIN_HOURS:.1f}h → max_steps_for_budget = {max_steps_for_budget:,}")
print(f"   Using max_steps = {max_steps:,}")
print(f"   ⏱️ Rough GPU time ≈ {est_hours:.2f} h")

# ──────────────────────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Loading Whisper-small (full weights)...")
model = WhisperForConditionalGeneration.from_pretrained(
    RESUME_CHECKPOINT,
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
)

model.config.use_cache = False
model.generation_config = GenerationConfig.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.to(DEVICE)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable:,} (full fine-tune)")

# ──────────────────────────────────────────────────────────────────────────────
# DATA COLLATOR
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

class WhisperEvalTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**{"language": "arabic", "task": "transcribe"}, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_steps=max_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True if torch.cuda.is_available() else False, 
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_STEPS,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.args._n_gpu = 1

# ──────────────────────────────────────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────────────────────────────────────
_wall = time.time()
print("\n🚀 Starting training...")


print("Checkpoint exists:", os.path.exists(RESUME_CHECKPOINT))
print("Checkpoint is dir:", os.path.isdir(RESUME_CHECKPOINT))
print("Checkpoint files:", os.listdir(RESUME_CHECKPOINT))

train_result = None
try:
    train_result = trainer.train()
    print(f"\n✅ Training finished in {(time.time()-_wall)/3600:.2f} h — steps: {train_result.global_step}")
except KeyboardInterrupt:
    print("\n⚠️ Interrupted. Saving emergency snapshot...")
    emergency_dir = os.path.join(OUTPUT_DIR, "interrupted")
    os.makedirs(emergency_dir, exist_ok=True)
    trainer.save_model(emergency_dir)
    processor.save_pretrained(emergency_dir)
    print(f"💾 Emergency snapshot saved → {emergency_dir}")
    raise
finally:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"💾 Saved latest full model + processor → {OUTPUT_DIR}")

metrics = trainer.evaluate()
print(f"📊 Best checkpoint eval WER: {metrics.get('eval_wer', float('nan')):.2f}%")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n⏱️ Total wall time (build + train + eval + save): {(time.time()-t0)/3600:.2f} h")

✅ Device: cuda
   GPU: Tesla T4 (sm_75) | torch 2.10.0+cu128


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Emergency snapshot saved → /kaggle/working/whisper_small_linto_200k_ft/interrupted


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Saved latest full model + processor → /kaggle/working/whisper_small_linto_200k_ft


KeyboardInterrupt: 

In [12]:

CACHE_DIR = "/kaggle/working/hf_cache"
RESUME_CHECKPOINT = "/kaggle/input/datasets/aymendhieb/tunisian-200k-checkpoint5000/checkpoint-5000"
# LinTO Tunisian Arabic — FULL fine-tune (no LoRA), streaming up to 150k train samples from checkpoint-5000
import os, re, gc, time, math, io, warnings, random

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for v in ["MASTER_ADDR", "MASTER_PORT", "WORLD_SIZE", "RANK", "LOCAL_RANK"]:
    os.environ.pop(v, None)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import evaluate
import soundfile as sf
from scipy.signal import resample_poly
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Dataset
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from transformers.generation.configuration_utils import GenerationConfig

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "linagora/linto-dataset-audio-ar-tn"
OUTPUT_DIR = "/kaggle/working/whisper_small_linto_200k_ft"
TARGET_SR = 16_000

MAX_TRAIN_SAMPLES = 150_000
MAX_EVAL_SAMPLES = 800
STREAM_SHUFFLE_BUFFER = 50_000

PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM = 2
NUM_EPOCHS = 1
LEARNING_RATE = 3e-6
WARMUP_RATIO = 0.02

MAX_TRAIN_HOURS = 8.5
SEC_PER_STEP_GUESS = 2.7
EVAL_EVERY_STEPS = 2000

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"   GPU: {p.name} (sm_{p.major}{p.minor}) | torch {torch.__version__}")

# ──────────────────────────────────────────────────────────────────────────────
# TEXT HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_text(ex) -> str:
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

def transcript_ok(text: str) -> bool:
    t = normalize_ar(text)
    return (
        len(t) >= 8
        and len(t.split()) >= 3
        and not re.search(r"(.)\1{4,}", t)
        and len(t) < 300
    )

def ex_id(ex) -> str:
    if ex.get("audio_id") is not None:
        return str(ex["audio_id"])

    text = get_text(ex)
    audio = ex.get("audio", {})
    audio_path = audio.get("path", "") if isinstance(audio, dict) else ""
    return str(hash((normalize_ar(text)[:120], audio_path)))

# ──────────────────────────────────────────────────────────────────────────────
# STREAM LOADER
# ──────────────────────────────────────────────────────────────────────────────
def load_stream(name: str, seed: int):
    ds = load_dataset(name, split="train", streaming=True)
    try:
        ds = ds.shuffle(seed=seed, buffer_size=STREAM_SHUFFLE_BUFFER)
    except Exception as e:
        print("⚠️ Shuffle skipped:", e)
    return ds

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO READER
# ──────────────────────────────────────────────────────────────────────────────
def _read_audio_from_example(ex):
    audio = ex.get("audio", None)
    if audio is None:
        raise ValueError(f"No 'audio' field. Example keys: {list(ex.keys())}")

    if isinstance(audio, dict):
        if audio.get("array") is not None and audio.get("sampling_rate") is not None:
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = int(audio["sampling_rate"])
        elif audio.get("bytes") is not None:
            arr, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
        elif audio.get("path") is not None:
            arr, sr = sf.read(audio["path"], dtype="float32")
        else:
            raise ValueError(f"Unsupported audio dict keys: {list(audio.keys())}")
    else:
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            arr = samples.data.cpu().numpy() if hasattr(samples.data, "cpu") else np.asarray(samples.data)
            sr = int(samples.sample_rate)
        elif hasattr(audio, "decode"):
            decoded = audio.decode()
            if isinstance(decoded, dict) and "array" in decoded and "sampling_rate" in decoded:
                arr = np.asarray(decoded["array"], dtype=np.float32)
                sr = int(decoded["sampling_rate"])
            else:
                raise ValueError(f"Unsupported decoded audio output type: {type(decoded)}")
        else:
            raise ValueError(f"Unsupported audio type: {type(audio)}")

    arr = np.asarray(arr, dtype=np.float32)

    if arr.ndim == 2:
        if arr.shape[0] <= 4 and arr.shape[1] > arr.shape[0]:
            arr = arr.mean(axis=0)
        elif arr.shape[1] <= 4 and arr.shape[0] > arr.shape[1]:
            arr = arr.mean(axis=1)
        else:
            arr = arr.mean(axis=0 if arr.shape[0] < arr.shape[1] else 1)

    if arr.ndim != 1:
        raise ValueError(f"Audio must be 1D after mono conversion, got shape={arr.shape}")

    if sr != TARGET_SR:
        g = math.gcd(int(sr), int(TARGET_SR))
        arr = resample_poly(arr, TARGET_SR // g, sr // g).astype(np.float32, copy=False)
        sr = TARGET_SR

    return arr, sr


def process_sample(ex, processor: WhisperProcessor, debug=False):
    try:
        arr, sr = _read_audio_from_example(ex)  # ← was missing

        if debug:
            print("Waveform shape:", arr.shape, "| sr:", sr, "| duration:", arr.shape[0] / sr)

        if arr.size == 0:
            if debug: print("Rejected: empty audio")
            return None

        if arr.shape[0] < sr * 0.1:
            if debug: print("Rejected: too short")
            return None

        if np.max(np.abs(arr)) < 1e-6:
            if debug: print("Rejected: silent audio")
            return None

        dur = arr.shape[0] / sr
        if dur > 20.0:
            if debug: print(f"Rejected: too long ({dur:.2f}s)")
            return None

        raw_text = get_text(ex)
        text = normalize_ar(raw_text)

        if not transcript_ok(text):
            if debug: print("Rejected: bad transcript ->", repr(raw_text[:120]))
            return None

        feats = processor.feature_extractor(arr, sampling_rate=sr).input_features[0]

        ids = processor.tokenizer(text, max_length=448, truncation=True).input_ids

        if len(ids) < 2:
            if debug: print("Rejected: tokenized text too short")
            return None

        return {"input_features": feats, "labels": ids}

    except Exception as e:
        if debug:
            print("process_sample ERROR:", type(e).__name__, str(e))
        return None

# ──────────────────────────────────────────────────────────────────────────────
# LOAD PROCESSOR
# ──────────────────────────────────────────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
# ──────────────────────────────────────────────────────────────────────────────
# DEBUG: PRINT FIRST FEW SAMPLES
# ──────────────────────────────────────────────────────────────────────────────
print("\n🔍 Debugging first 5 raw samples...")
dbg_stream = load_stream(DATASET_NAME, seed=SEED)

for i, raw_ex in enumerate(dbg_stream):
    print(f"\n--- SAMPLE {i} ---")
    print("Keys:", list(raw_ex.keys()))
    txt = get_text(raw_ex)
    print("Text preview:", repr(txt[:120]))
    audio = raw_ex.get("audio", None)
    print("Audio type:", type(audio))
    if isinstance(audio, dict):
        print("Audio keys:", list(audio.keys()))
    result = process_sample(raw_ex, processor, debug=True)
    print("Accepted:", result is not None)
    if i >= 4:
        break

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD EVAL SET
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Phase 1 — streaming eval split (reserved ids)...")
stream_ev = load_stream(DATASET_NAME, seed=SEED)

eval_features: List[np.ndarray] = []
eval_labels: List[List[int]] = []
eval_ids: set = set()

seen = 0
t0 = time.time()

for ex in stream_ev:
    seen += 1

    text = get_text(ex)
    if not transcript_ok(text):
        continue

    eid = ex_id(ex)
    if eid in eval_ids:
        continue

    item = process_sample(ex, processor, debug=False)
    if item is None:
        continue

    eval_ids.add(eid)
    eval_features.append(item["input_features"])
    eval_labels.append(item["labels"])

    if len(eval_features) >= MAX_EVAL_SAMPLES:
        break

    if seen % 2000 == 0:
        print(f"   scanned={seen:,} | eval_ok={len(eval_features):,}/{MAX_EVAL_SAMPLES}")

print(f"\n   Final scanned={seen:,} | eval_ok={len(eval_features):,}")

if len(eval_features) < max(100, MAX_EVAL_SAMPLES // 3):
    raise RuntimeError(
        f"Too few eval samples collected: {len(eval_features)}. "
        f"Dataset text/audio format is still not matching the code."
    )

eval_ds = Dataset.from_dict({
    "input_features": eval_features,
    "labels": eval_labels
})
del eval_features, eval_labels
gc.collect()

print(f"   ✅ Eval built: {len(eval_ds):,} | unique ids: {len(eval_ids):,} | {time.time()-t0:.1f}s")

# ──────────────────────────────────────────────────────────────────────────────
# PHASE 2: TRAIN GENERATOR
# ──────────────────────────────────────────────────────────────────────────────
def train_example_gen():
    stream_tr = load_stream(DATASET_NAME, seed=SEED + 7)
    n = 0
    skipped_eval = 0

    for ex in stream_tr:
        text = get_text(ex)
        if not transcript_ok(text):
            continue

        eid = ex_id(ex)
        if eid in eval_ids:
            skipped_eval += 1
            continue

        item = process_sample(ex, processor, debug=False)
        if item is None:
            continue

        yield item
        n += 1

        if n >= MAX_TRAIN_SAMPLES:
            break

        if n % 5000 == 0:
            print(f"   train_yield={n:,}/{MAX_TRAIN_SAMPLES:,} | skipped_eval_hits={skipped_eval:,}")

train_ds = Dataset.from_generator(train_example_gen)

# ──────────────────────────────────────────────────────────────────────────────
# TRAINING STEPS
# ──────────────────────────────────────────────────────────────────────────────
world = 1
samples_per_step = PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM * world
steps_per_epoch = max(1, math.ceil(MAX_TRAIN_SAMPLES / samples_per_step))
planned_steps = steps_per_epoch * NUM_EPOCHS
max_steps_for_budget = max(1, int((MAX_TRAIN_HOURS * 3600) / SEC_PER_STEP_GUESS))
max_steps = min(planned_steps, max_steps_for_budget, 7000)

est_hours = max_steps * SEC_PER_STEP_GUESS / 3600.0

print("\n📌 Plan")
print(f"   Train cap: {MAX_TRAIN_SAMPLES:,} | Eval: {len(eval_ds):,}")
print(f"   Batch {PER_DEVICE_TRAIN_BATCH} × grad_accum {GRAD_ACCUM} → {samples_per_step} samples/step")
print(f"   Steps/epoch ≈ {steps_per_epoch:,} | planned_steps = {planned_steps:,}")
print(f"   Runtime cap: {MAX_TRAIN_HOURS:.1f}h → max_steps_for_budget = {max_steps_for_budget:,}")
print(f"   Using max_steps = {max_steps:,}")
print(f"   ⏱️ Rough GPU time ≈ {est_hours:.2f} h")

# ──────────────────────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────────────────────
print("\n⬇️ Loading Whisper-small (full weights)...")
model = WhisperForConditionalGeneration.from_pretrained(
    RESUME_CHECKPOINT,
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
)

model.config.use_cache = False
model.generation_config = GenerationConfig.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=CACHE_DIR
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.to(DEVICE)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable:,} (full fine-tune)")

# ──────────────────────────────────────────────────────────────────────────────
# DATA COLLATOR
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        bos = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos is not None and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            "input_features": batch["input_features"],
            "labels": labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_ar(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

class WhisperEvalTrainer(Seq2SeqTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **gen_kwargs):
        gen_kwargs = {**{"language": "arabic", "task": "transcribe"}, **gen_kwargs}
        return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix, **gen_kwargs)

warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_steps=max_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True if torch.cuda.is_available() else False, 
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_STEPS,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    seed=SEED,
)

trainer = WhisperEvalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.args._n_gpu = 1

# ──────────────────────────────────────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────────────────────────────────────
_wall = time.time()
print("\n🚀 Starting training...")


print("Checkpoint exists:", os.path.exists(RESUME_CHECKPOINT))
print("Checkpoint is dir:", os.path.isdir(RESUME_CHECKPOINT))
print("Checkpoint files:", os.listdir(RESUME_CHECKPOINT))

train_result = None
try:
    train_result = trainer.train()
    print(f"\n✅ Training finished in {(time.time()-_wall)/3600:.2f} h — steps: {train_result.global_step}")
except KeyboardInterrupt:
    print("\n⚠️ Interrupted. Saving emergency snapshot...")
    emergency_dir = os.path.join(OUTPUT_DIR, "interrupted")
    os.makedirs(emergency_dir, exist_ok=True)
    trainer.save_model(emergency_dir)
    processor.save_pretrained(emergency_dir)
    print(f"💾 Emergency snapshot saved → {emergency_dir}")
    raise
finally:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"💾 Saved latest full model + processor → {OUTPUT_DIR}")

print("✅ Training complete. Best model already loaded.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n⏱️ Total wall time (build + train + eval + save): {(time.time()-t0)/3600:.2f} h")

✅ Device: cuda
   GPU: Tesla T4 (sm_75) | torch 2.10.0+cu128


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

{'loss': '0.006772', 'grad_norm': '0.1626', 'learning_rate': '1.4e-07', 'epoch': '7.553'}
{'loss': '0.007771', 'grad_norm': '1.007', 'learning_rate': '1.259e-07', 'epoch': '7.615'}
{'loss': '0.005497', 'grad_norm': '1.062', 'learning_rate': '1.125e-07', 'epoch': '7.678'}
{'loss': '0.006628', 'grad_norm': '0.1112', 'learning_rate': '9.979e-08', 'epoch': '7.74'}
{'loss': '0.006313', 'grad_norm': '0.07328', 'learning_rate': '8.784e-08', 'epoch': '7.803'}
{'loss': '0.00627', 'grad_norm': '0.09254', 'learning_rate': '7.663e-08', 'epoch': '7.865'}
{'loss': '0.004459', 'grad_norm': '0.1664', 'learning_rate': '6.617e-08', 'epoch': '7.928'}
{'loss': '0.00794', 'grad_norm': '0.05679', 'learning_rate': '5.646e-08', 'epoch': '7.99'}
{'loss': '0.006382', 'grad_norm': '1.649', 'learning_rate': '4.75e-08', 'epoch': '8.052'}
{'loss': '0.006245', 'grad_norm': '1.82', 'learning_rate': '3.931e-08', 'epoch': '8.115'}
{'loss': '0.006822', 'grad_norm': '0.9052', 'learning_rate': '3.188e-08', 'epoch': '8.177

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2.811e+04', 'train_samples_per_second': '3.984', 'train_steps_per_second': '0.249', 'train_loss': '0.01446', 'epoch': '8.739'}

✅ Training finished in 7.81 h — steps: 7000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Saved latest full model + processor → /kaggle/working/whisper_small_linto_200k_ft
✅ Training complete. Best model already loaded.

⏱️ Total wall time (build + train + eval + save): 8.03 h


In [1]:
# =========================================================
# 0) Install if needed
# =========================================================
# !pip install -q transformers datasets evaluate jiwer peft accelerate soundfile librosa

# =========================================================
# 1) Imports
# =========================================================
import os
import re
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import load_dataset, Audio, Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoProcessor,
    AutoModelForCTC,
)
from peft import PeftModel

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16_000
EVAL_LIMIT = 50   # increase later if you want more stable metrics
print("Device:", DEVICE)

# =========================================================
# 2) Text helpers
# =========================================================
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_text(ex) -> str:
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

def valid_eval_example(ex) -> bool:
    txt = get_text(ex)
    if not txt.strip():
        return False
    audio = ex.get("audio", None)
    if audio is None:
        return False
    return True

# =========================================================
# 3) Load datasets
# =========================================================
# Medical Speech dataset
medical_ds = load_dataset(
    "Sabrinek8/tedxtn-train",
    split="train"
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

# LinTO Tunisian dataset
linto_ds = load_dataset(
    "linagora/linto-dataset-audio-ar-tn",
    split="train"
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Keep only valid examples and make small fair subsets
medical_eval = medical_ds.filter(valid_eval_example).select(range(min(EVAL_LIMIT, len(medical_ds))))
linto_eval = linto_ds.filter(valid_eval_example).select(range(min(EVAL_LIMIT, len(linto_ds))))

print("Medical eval size:", len(medical_eval))
print("LinTO eval size:", len(linto_eval))

# =========================================================
# 4) Metrics
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_wer_cer(preds, refs):
    preds = [normalize_ar(x) for x in preds]
    refs  = [normalize_ar(x) for x in refs]
    wer = 100 * wer_metric.compute(predictions=preds, references=refs)
    cer = 100 * cer_metric.compute(predictions=preds, references=refs)
    return round(wer, 2), round(cer, 2)

# =========================================================
# 5) Path helper
#    Put your exact paths here once and reuse them
# =========================================================
# IMPORTANT:
# Adjust these to your actual Kaggle input paths if needed.

MODEL_PATHS = {
    # ---------- Model comparison / medical ----------
    "whisper_lora_medical": "/kaggle/input/datasets/aymendhieb/whisper-adapter-final-v1/whisper_adapter_final",   # change if needed
    "whisper_dora_medical": "/kaggle/input/datasets/aymendhieb1/whisper-dora-adapter-v1/whisper_dora_adapter",                           # change if needed
    "wav2vec2_medical": "/kaggle/input/datasets/aymendhieb1/wav2vec2-saved-v1/wav2vec2-saved",                                     # change if needed

    # ---------- Tunisian progression / LinTO ----------
    "whisper_tunisian_adapter": "/kaggle/input/datasets/aymendhieb/whisper-tunisian-adapter/wshiper_tunisian_adapter_v1",                      # TEDxTN adapter
    "whisper_tunisian_100k": "/kaggle/input/datasets/aymendhieb/whisper-tunisian-100k/whisper_tunisian_100k",                            # full FT
    "checkpoint_2300_100k": "/kaggle/input/datasets/aymendhieb/checkpoint100k/check2point2300",                    # full FT checkpoint
    "checkpoint_5000_200k": "/kaggle/input/datasets/aymendhieb/tunisian-200k-checkpoint5000/checkpoint-5000",      # full FT checkpoint
    "checkpoint_7000_150k": "/kaggle/input/datasets/aymendhieb/checkpoint-7000/checkpoint2__7000",                   # final checkpoint
}

# =========================================================
# 6) Evaluation functions
# =========================================================
def eval_whisper_full(model_path, dataset, model_label):
    print(f"\nEvaluating Whisper full model: {model_label}")
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained(model_path).to(DEVICE)

    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="ar",
        task="transcribe"
    )
    model.generation_config.suppress_tokens = []

    preds, refs = [], []

    for ex in dataset:
        audio = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        ref = get_text(ex)

        inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
        input_features = inputs["input_features"].to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(input_features, max_new_tokens=80)

        pred = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
        preds.append(pred)
        refs.append(ref)

    wer, cer = compute_wer_cer(preds, refs)
    return wer, cer

def eval_whisper_adapter(adapter_path, dataset, model_label, base_model="openai/whisper-small"):
    print(f"\nEvaluating Whisper adapter: {model_label}")
    processor = WhisperProcessor.from_pretrained(base_model)
    base = WhisperForConditionalGeneration.from_pretrained(base_model).to(DEVICE)
    model = PeftModel.from_pretrained(base, adapter_path).to(DEVICE)

    # Merge for cleaner inference if possible
    try:
        model = model.merge_and_unload()
    except Exception:
        pass

    if hasattr(model, "generation_config"):
        model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language="ar",
            task="transcribe"
        )
        model.generation_config.suppress_tokens = []

    preds, refs = [], []

    for ex in dataset:
        audio = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        ref = get_text(ex)

        inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
        input_features = inputs["input_features"].to(DEVICE)

        with torch.no_grad():
            pred_ids = model.generate(input_features, max_new_tokens=80)

        pred = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
        preds.append(pred)
        refs.append(ref)

    wer, cer = compute_wer_cer(preds, refs)
    return wer, cer

def eval_wav2vec2(model_path, dataset, model_label):
    print(f"\nEvaluating Wav2Vec2: {model_label}")
    processor = AutoProcessor.from_pretrained(model_path)
    model = AutoModelForCTC.from_pretrained(model_path).to(DEVICE)

    preds, refs = [], []

    for ex in dataset:
        audio = ex["audio"]["array"]
        sr = ex["audio"]["sampling_rate"]
        ref = get_text(ex)

        inputs = processor(audio, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs["input_values"].to(DEVICE)

        with torch.no_grad():
            logits = model(input_values).logits

        pred_ids = torch.argmax(logits, dim=-1)
        pred = processor.batch_decode(pred_ids)[0]

        preds.append(pred)
        refs.append(ref)

    wer, cer = compute_wer_cer(preds, refs)
    return wer, cer

# =========================================================
# 7) TABLE 1 — Model comparison on Medical Speech
# =========================================================
medical_models = [
    {
        "Model": "Whisper LoRA",
        "Method": "LoRA",
        "Trainable Params": "~2.3M (~1%)",
        "kind": "whisper_adapter",
        "path": MODEL_PATHS["whisper_lora_medical"],
    },
    {
        "Model": "Whisper DoRA",
        "Method": "DoRA",
        "Trainable Params": "~2.3M (~1%)",
        "kind": "whisper_adapter",
        "path": MODEL_PATHS["whisper_dora_medical"],
    },
    {
        "Model": "Wav2Vec2-base",
        "Method": "Full fine-tuning",
        "Trainable Params": "~90M",
        "kind": "wav2vec2",
        "path": MODEL_PATHS["wav2vec2_medical"],
    },
]

medical_results = []

for m in medical_models:
    try:
        if m["kind"] == "whisper_adapter":
            wer, cer = eval_whisper_adapter(m["path"], medical_eval, m["Model"])
        elif m["kind"] == "wav2vec2":
            wer, cer = eval_wav2vec2(m["path"], medical_eval, m["Model"])
        else:
            raise ValueError("Unsupported kind")
    except Exception as e:
        print(f"Error while evaluating {m['Model']}: {e}")
        wer, cer = None, None

    medical_results.append({
        "Model": m["Model"],
        "Method": m["Method"],
        "Trainable Params": m["Trainable Params"],
        "WER (%)": wer,
        "CER (%)": cer,
    })

medical_results_df = pd.DataFrame(medical_results)
print("\n🏆 FINAL RESULTS — Medical Speech")
print(medical_results_df)

# =========================================================
# 8) TABLE 2 — Hyperparameters summary
#    Fill these from your actual runs
# =========================================================
hyperparams = [
    {
        "Model": "Whisper LoRA",
        "Dataset": "Medical Speech",
        "Method": "LoRA",
        "LR": "fill_me",
        "Batch": "fill_me",
        "Steps": "fill_me",
        "Epochs": "fill_me",
        "Notes": "Medical-domain LoRA",
    },
    {
        "Model": "Whisper DoRA",
        "Dataset": "Medical Speech",
        "Method": "DoRA",
        "LR": "fill_me",
        "Batch": "fill_me",
        "Steps": "fill_me",
        "Epochs": "fill_me",
        "Notes": "Medical-domain DoRA",
    },
    {
        "Model": "Wav2Vec2-base",
        "Dataset": "Medical Speech",
        "Method": "Full FT",
        "LR": "fill_me",
        "Batch": "fill_me",
        "Steps": "fill_me",
        "Epochs": "fill_me",
        "Notes": "CNN frozen / unfrozen as used",
    },
    {
        "Model": "whisper_tunisian_adapter",
        "Dataset": "TEDxTN / LinTO follow-up",
        "Method": "Adapter / PEFT",
        "LR": "fill_me",
        "Batch": "fill_me",
        "Steps": "fill_me",
        "Epochs": "fill_me",
        "Notes": "Original Tunisian adapter",
    },
    {
        "Model": "whisper_tunisian_100k",
        "Dataset": "LinTO",
        "Method": "Full FT",
        "LR": "1e-5",
        "Batch": "8 x grad_accum 2",
        "Steps": "100k run",
        "Epochs": "2 planned",
        "Notes": "Full fine-tuning on 100k",
    },
    {
        "Model": "checkpoint2300",
        "Dataset": "LinTO",
        "Method": "Full FT checkpoint",
        "LR": "1e-5",
        "Batch": "8 x grad_accum 2",
        "Steps": "2300",
        "Epochs": "partial",
        "Notes": "100k run checkpoint",
    },
    {
        "Model": "checkpoint5000",
        "Dataset": "LinTO",
        "Method": "Full FT checkpoint",
        "LR": "5e-6 or 3e-6 depending run",
        "Batch": "8 x grad_accum 2",
        "Steps": "5000",
        "Epochs": "partial",
        "Notes": "200k continuation",
    },
    {
        "Model": "checkpoint7000",
        "Dataset": "LinTO",
        "Method": "Full FT final",
        "LR": "3e-6",
        "Batch": "8 x grad_accum 2",
        "Steps": "7000",
        "Epochs": "1 capped",
        "Notes": "150k final continuation",
    },
]

hyperparams_df = pd.DataFrame(hyperparams)
print("\n⚙️ TRAINING CONFIGURATION")
print(hyperparams_df)

# =========================================================
# 9) TABLE 3 — Tunisian progression on LinTO
# =========================================================
tunisian_models = [
    {
        "Step": 1,
        "Model / Checkpoint": "whisper_tunisian_adapter",
        "Data Size": "TEDxTN / adapter",
        "Method": "Adapter / PEFT",
        "kind": "whisper_adapter",
        "path": MODEL_PATHS["whisper_tunisian_adapter"],
    },
    {
        "Step": 2,
        "Model / Checkpoint": "whisper_tunisian_100k",
        "Data Size": "100k",
        "Method": "Full FT",
        "kind": "whisper_full",
        "path": MODEL_PATHS["whisper_tunisian_100k"],
    },
    {
        "Step": 3,
        "Model / Checkpoint": "checkpoint2300",
        "Data Size": "100k",
        "Method": "FT checkpoint",
        "kind": "whisper_full",
        "path": MODEL_PATHS["checkpoint_2300_100k"],
    },
    {
        "Step": 4,
        "Model / Checkpoint": "checkpoint5000",
        "Data Size": "200k",
        "Method": "FT checkpoint",
        "kind": "whisper_full",
        "path": MODEL_PATHS["checkpoint_5000_200k"],
    },
    {
        "Step": 5,
        "Model / Checkpoint": "checkpoint7000",
        "Data Size": "150k",
        "Method": "FT final",
        "kind": "whisper_full",
        "path": MODEL_PATHS["checkpoint_7000_150k"],
    },
]

tunisian_results = []

for m in tunisian_models:
    try:
        if m["kind"] == "whisper_adapter":
            wer, cer = eval_whisper_adapter(m["path"], linto_eval, m["Model / Checkpoint"])
        elif m["kind"] == "whisper_full":
            wer, cer = eval_whisper_full(m["path"], linto_eval, m["Model / Checkpoint"])
        else:
            raise ValueError("Unsupported kind")
    except Exception as e:
        print(f"Error while evaluating {m['Model / Checkpoint']}: {e}")
        wer, cer = None, None

    tunisian_results.append({
        "Step": m["Step"],
        "Model / Checkpoint": m["Model / Checkpoint"],
        "Data Size": m["Data Size"],
        "Method": m["Method"],
        "WER (%)": wer,
        "CER (%)": cer,
    })

tunisian_results_df = pd.DataFrame(tunisian_results)
print("\n📈 TUNISIAN ASR — Training Progression on LinTO")
print(tunisian_results_df)

# =========================================================
# 10) Optional: export tables
# =========================================================
medical_results_df.to_csv("medical_model_comparison.csv", index=False)
hyperparams_df.to_csv("hyperparams_summary.csv", index=False)
tunisian_results_df.to_csv("tunisian_progression.csv", index=False)

print("\nSaved:")
print("- medical_model_comparison.csv")
print("- hyperparams_summary.csv")
print("- tunisian_progression.csv")

ModuleNotFoundError: No module named 'evaluate'

In [6]:
# =========================================================
# CELL 1: Install if needed
# =========================================================
!pip install -q jiwer transformers datasets soundfile librosa

# =========================================================
# CELL 2: Imports
# =========================================================
import os
import re
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16000

print("Device:", DEVICE)

# =========================================================
# CELL 3: Path to checkpoint_7000
# =========================================================
MODEL_PATH = "/kaggle/input/datasets/aymendhieb/checkpoint-7000/checkpoint2__7000"

print("Exists:", os.path.exists(MODEL_PATH))
print("Is dir :", os.path.isdir(MODEL_PATH))
print("Files  :", os.listdir(MODEL_PATH))

# =========================================================
# CELL 4: Text helpers
# =========================================================
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_ref_text(ex):
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

# =========================================================
# CELL 5: Load dataset
# Choose ONE dataset depending on what you want to evaluate
# =========================================================

# --- Option A: LinTO Tunisian ---
ds = load_dataset(
    "linagora/linto-dataset-audio-ar-tn",
    split="train[:100]"
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

# --- Option B: TEDxTN ---
# ds = load_dataset(
#     "Sabrinek8/tedxtn-train",
#     split="train[:100]"
# ).cast_column("audio", Audio(sampling_rate=TARGET_SR))

print("Dataset size:", len(ds))
print("Columns:", ds.column_names)

# =========================================================
# CELL 6: Load checkpoint_7000
# =========================================================
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.eval()

print("Model loaded.")

# =========================================================
# CELL 7: Inference function
# =========================================================
def transcribe_audio(audio_array, sampling_rate=16000):
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )
    input_features = inputs["input_features"].to(DEVICE)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=120
        )

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()

# =========================================================
# CELL 8: Run evaluation and collect results
# =========================================================
results = []

for i, ex in enumerate(ds):
    ref = get_ref_text(ex)
    if not ref.strip():
        continue

    audio = ex["audio"]["array"]
    sr = ex["audio"]["sampling_rate"]

    pred = transcribe_audio(audio, sampling_rate=sr)

    results.append({
        "sample_id": i,
        "reference": normalize_ar(ref),
        "prediction": normalize_ar(pred),
    })

print("Collected results:", len(results))

# =========================================================
# CELL 9: Overall WER / CER
# =========================================================
refs = [row["reference"] for row in results]
preds = [row["prediction"] for row in results]

overall_wer = wer(refs, preds)
overall_cer = cer(refs, preds)

print(f"WER: {overall_wer * 100:.2f}%")
print(f"CER: {overall_cer * 100:.2f}%")

# =========================================================
# CELL 10: Per-sample WER
# =========================================================
for row in results[:10]:   # print first 10 only
    sample_wer = wer([row["reference"]], [row["prediction"]])
    print(f"Sample {row['sample_id']} WER: {sample_wer * 100:.2f}%")

# =========================================================
# CELL 11: Save results to CSV
# =========================================================
import pandas as pd

df_results = pd.DataFrame(results)
df_results["sample_wer"] = [
    wer([row["reference"]], [row["prediction"]]) * 100
    for row in results
]

df_results.to_csv("checkpoint7000_eval_results.csv", index=False)
print("Saved: checkpoint7000_eval_results.csv")

Device: cpu
Exists: True
Is dir : True
Files  : ['config.json', 'trainer_state.json', 'training_args.bin', 'scaler.pt', 'scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json']


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

data/AmenyKH/train/train-00000-of-00002.(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=120) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans

Collected results: 100
WER: 8.28%
CER: 2.85%
Sample 0 WER: 13.16%
Sample 1 WER: 42.86%
Sample 2 WER: 0.00%
Sample 3 WER: 0.00%
Sample 4 WER: 0.00%
Sample 5 WER: 0.00%
Sample 6 WER: 12.50%
Sample 7 WER: 0.00%
Sample 8 WER: 0.00%
Sample 9 WER: 0.00%
Saved: checkpoint7000_eval_results.csv


In [8]:
# =========================================================
# CELL 1: Install if needed
# =========================================================
!pip install -q jiwer transformers datasets soundfile librosa sentencepiece accelerate

# =========================================================
# CELL 2: Imports
# =========================================================
import os
import re
import torch
import pandas as pd
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
)
from jiwer import wer, cer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
TARGET_SR = 16000

print("Device:", DEVICE)
print("Dtype :", DTYPE)

# =========================================================
# CELL 3: Paths
# =========================================================
MODEL_PATH = "/kaggle/input/datasets/aymendhieb/checkpoint-7000/checkpoint2__7000"
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("ASR exists:", os.path.exists(MODEL_PATH))
print("ASR is dir:", os.path.isdir(MODEL_PATH))
print("ASR files :", os.listdir(MODEL_PATH))

# =========================================================
# CELL 4: Text helpers
# =========================================================
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_ref_text(ex):
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

# =========================================================
# CELL 5: Load dataset
# Choose ONE dataset
# =========================================================

# --- Option A: LinTO Tunisian ---
ds = load_dataset(
    "linagora/linto-dataset-audio-ar-tn",
    split="train[:50]"   # use 50 first for speed
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

# --- Option B: TEDxTN ---
# ds = load_dataset(
#     "Sabrinek8/tedxtn-train",
#     split="train[:50]"
# ).cast_column("audio", Audio(sampling_rate=TARGET_SR))

print("Dataset size:", len(ds))
print("Columns:", ds.column_names)

# =========================================================
# CELL 6: Load ASR model
# =========================================================
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.eval()

print("ASR model loaded.")

# =========================================================
# CELL 7: Load Qwen
# =========================================================
print("Loading Qwen tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

print("Loading Qwen model...")
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Qwen loaded.")

# =========================================================
# CELL 8: ASR transcription
# =========================================================
def transcribe_audio(audio_array, sampling_rate=16000):
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )
    input_features = inputs["input_features"].to(DEVICE)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=80   # faster than 120
        )

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()

# =========================================================
# CELL 9: Qwen post-correction
# =========================================================
def llm_darija_fix(raw_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You correct ASR text written in Tunisian Darija. "
                "Keep exactly the same meaning. "
                "Keep the Tunisian dialect. "
                "Do not translate. "
                "Do not summarize. "
                "Do not add information. "
                "Only fix obvious ASR mistakes. "
                "Return only the corrected text."
            ),
        },
        {
            "role": "user",
            "content": raw_text,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    fixed = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return fixed

# =========================================================
# CELL 10: Evaluate raw + clean
# =========================================================
results = []

for i, ex in enumerate(ds):
    ref = get_ref_text(ex)
    if not ref.strip():
        continue

    audio = ex["audio"]["array"]
    sr = ex["audio"]["sampling_rate"]

    raw_pred = transcribe_audio(audio, sampling_rate=sr)
    clean_pred = llm_darija_fix(raw_pred)

    results.append({
        "sample_id": i,
        "reference": normalize_ar(ref),
        "raw_prediction": normalize_ar(raw_pred),
        "clean_prediction": normalize_ar(clean_pred),
    })

print("Collected results:", len(results))

# =========================================================
# CELL 11: Overall WER / CER
# =========================================================
refs = [row["reference"] for row in results]
raw_preds = [row["raw_prediction"] for row in results]
clean_preds = [row["clean_prediction"] for row in results]

raw_wer = wer(refs, raw_preds) * 100
raw_cer = cer(refs, raw_preds) * 100

clean_wer = wer(refs, clean_preds) * 100
clean_cer = cer(refs, clean_preds) * 100

print("\n=== OVERALL RESULTS ===")
print(f"RAW   -> WER: {raw_wer:.2f}% | CER: {raw_cer:.2f}%")
print(f"CLEAN -> WER: {clean_wer:.2f}% | CER: {clean_cer:.2f}%")

# =========================================================
# CELL 12: Per-sample WER
# =========================================================
for row in results[:10]:
    sample_wer_raw = wer([row["reference"]], [row["raw_prediction"]]) * 100
    sample_wer_clean = wer([row["reference"]], [row["clean_prediction"]]) * 100
    print(
        f"Sample {row['sample_id']} | "
        f"RAW WER: {sample_wer_raw:.2f}% | "
        f"CLEAN WER: {sample_wer_clean:.2f}%"
    )

# =========================================================
# CELL 13: Save CSVs
# =========================================================
df_results = pd.DataFrame(results)

df_results["sample_wer_raw"] = [
    wer([row["reference"]], [row["raw_prediction"]]) * 100
    for row in results
]
df_results["sample_wer_clean"] = [
    wer([row["reference"]], [row["clean_prediction"]]) * 100
    for row in results
]

summary_df = pd.DataFrame([
    {"System": "Checkpoint7000 raw", "WER (%)": round(raw_wer, 2), "CER (%)": round(raw_cer, 2)},
    {"System": "Checkpoint7000 + Qwen", "WER (%)": round(clean_wer, 2), "CER (%)": round(clean_cer, 2)},
])

df_results.to_csv("checkpoint7000_raw_vs_clean_details.csv", index=False)
summary_df.to_csv("checkpoint7000_raw_vs_clean_summary.csv", index=False)

print("\nSaved:")
print("- checkpoint7000_raw_vs_clean_details.csv")
print("- checkpoint7000_raw_vs_clean_summary.csv")
print("\nSummary table:")
print(summary_df)

Device: cpu
Dtype : torch.float32
ASR exists: True
ASR is dir: True
ASR files : ['config.json', 'trainer_state.json', 'training_args.bin', 'scaler.pt', 'scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json']


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Dataset size: 50
Columns: ['audio_id', 'audio', 'segments', 'transcript']


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded.


Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take 

Collected results: 50

=== OVERALL RESULTS ===
RAW   -> WER: 8.73% | CER: 4.64%
CLEAN -> WER: 30.12% | CER: 15.21%
Sample 0 | RAW WER: 34.21% | CLEAN WER: 42.11%
Sample 1 | RAW WER: 42.86% | CLEAN WER: 42.86%
Sample 2 | RAW WER: 0.00% | CLEAN WER: 28.57%
Sample 3 | RAW WER: 0.00% | CLEAN WER: 0.00%
Sample 4 | RAW WER: 0.00% | CLEAN WER: 0.00%
Sample 5 | RAW WER: 0.00% | CLEAN WER: 100.00%
Sample 6 | RAW WER: 12.50% | CLEAN WER: 25.00%
Sample 7 | RAW WER: 0.00% | CLEAN WER: 20.00%
Sample 8 | RAW WER: 0.00% | CLEAN WER: 0.00%
Sample 9 | RAW WER: 0.00% | CLEAN WER: 16.67%

Saved:
- checkpoint7000_raw_vs_clean_details.csv
- checkpoint7000_raw_vs_clean_summary.csv

Summary table:
                  System  WER (%)  CER (%)
0     Checkpoint7000 raw     8.73     4.64
1  Checkpoint7000 + Qwen    30.12    15.21


In [9]:
!pip install -q jiwer transformers datasets soundfile librosa

import os
import re
import torch
import pandas as pd
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16000

MODEL_PATH = "/kaggle/input/datasets/aymendhieb/checkpoint-7000/checkpoint2__7000"

print("Device:", DEVICE)
print("Exists:", os.path.exists(MODEL_PATH))
print("Is dir :", os.path.isdir(MODEL_PATH))
print("Files  :", os.listdir(MODEL_PATH))

def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = (
        text.replace("أ", "ا")
        .replace("إ", "ا")
        .replace("آ", "ا")
        .replace("ى", "ي")
        .replace("ة", "ه")
        .replace("ـ", "")
    )
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_ref_text(ex):
    for key in ["transcript", "transcription", "text", "sentence", "normalized_text"]:
        val = ex.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

# ---------------------------------------------------------
# NEW DATASET: TEDxTN-style external evaluation
# ---------------------------------------------------------
ds = load_dataset(
    "Sabrinek8/tedxtn-train",
    split="train[:100]"
).cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Better: randomize so you do not always evaluate the first 100
ds = ds.shuffle(seed=42)

print("Dataset size:", len(ds))
print("Columns:", ds.column_names)

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar",
    task="transcribe"
)
model.generation_config.suppress_tokens = []
model.eval()

def transcribe_audio(audio_array, sampling_rate=16000):
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )
    input_features = inputs["input_features"].to(DEVICE)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=80
        )

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()

results = []

for i, ex in enumerate(ds):
    ref = get_ref_text(ex)
    if not ref.strip():
        continue

    audio = ex["audio"]["array"]
    sr = ex["audio"]["sampling_rate"]

    pred = transcribe_audio(audio, sampling_rate=sr)

    results.append({
        "sample_id": i,
        "reference": normalize_ar(ref),
        "prediction": normalize_ar(pred),
    })

print("Collected results:", len(results))

refs = [row["reference"] for row in results]
preds = [row["prediction"] for row in results]

overall_wer = wer(refs, preds) * 100
overall_cer = cer(refs, preds) * 100

print(f"TEDxTN-style WER: {overall_wer:.2f}%")
print(f"TEDxTN-style CER: {overall_cer:.2f}%")

df_results = pd.DataFrame(results)
df_results["sample_wer"] = [
    wer([row["reference"]], [row["prediction"]]) * 100
    for row in results
]

df_results.to_csv("checkpoint7000_tedxtn_eval_results.csv", index=False)
print("Saved: checkpoint7000_tedxtn_eval_results.csv")

Device: cpu
Exists: True
Is dir : True
Files  : ['config.json', 'trainer_state.json', 'training_args.bin', 'scaler.pt', 'scheduler.pt', 'model.safetensors', 'rng_state.pth', 'generation_config.json']


README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/15552 [00:00<?, ? examples/s]

Dataset size: 100
Columns: ['audio', 'text']


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Collected results: 100
TEDxTN-style WER: 53.80%
TEDxTN-style CER: 21.07%
Saved: checkpoint7000_tedxtn_eval_results.csv
